# BadTime PoC: ETTh1 + TimesNet, 96 → 96 (Colab / Kaggle)

Notebook này chứa toàn bộ mã của `badtime_repro`. Nó tự ghi mã ra đĩa, tải ETTh1 (kiểm tra sha256), chạy kiểm tra tính đúng, huấn luyện **Benign (clean)** và **BadTime** với nhiều seed, chạy đối chứng 4 điều kiện, rồi lập báo cáo so với Bảng V của bài báo v2.

**Trước khi chạy**
- Colab: `Runtime → Change runtime type → T4 GPU`.
- Kaggle: `Settings → Accelerator → GPU T4/P100` và **bật Internet** (cần để tải ETTh1). Nếu không có Internet, thêm ETTh1.csv làm Kaggle Dataset; ô dữ liệu sẽ tự tìm trong `/kaggle/input`.
- Chạy lần lượt từ trên xuống (`Run all` cũng được). Ước lượng trên T4: kiểm tra chậm ~5–10 phút (cố ý chạy trên CPU), mỗi lần train ~3–8 phút.

**Lưu ý khi đọc kết quả.** Đây là bản tái lập độc lập, còn nhiều giả định (A01–A21, in ở cuối notebook). Một con số khác bài báo chưa đủ để kết luận bài báo sai; xem `docs/06-validation-plan.md`.

## 1. Tham số của notebook

In [ ]:
# Seed cho mỗi nhánh. Đề xuất 3 lần lặp độc lập (docs/04).
SEEDS = [2021, 2022, 2023]

# Kiểm tra huấn luyện thật (CPU, mô hình nhỏ): đường xử lý sạch == BadTime đã tắt, tính lại chỉ số, đối chứng.
# Nên để True ở lần chạy đầu tiên; đây là cổng tính đúng trước khi xem số.
RUN_SLOW_TESTS = True

# Ghi đè cấu hình cho MỌI lệnh train/report, ví dụ ["attack.pattern.amplitude=2.0"].
# Đổi giả định thì nên đổi luôn runs_root để không trộn với lần chạy mặc định, ví dụ "runs_root=runs_amp2".
EXTRA_SET = []

# Mục 9 (tùy chọn): khảo sát độ nhạy với vài giả định chưa xác định.
RUN_SENSITIVITY = False

## 2. Môi trường

In [ ]:
import os, sys, platform, subprocess

PACKAGE_DIRS = ['badtime_repro', 'badtime_repro/attack', 'badtime_repro/common', 'badtime_repro/config', 'badtime_repro/data', 'badtime_repro/evaluation', 'badtime_repro/models', 'badtime_repro/models/tslib', 'badtime_repro/runs', 'badtime_repro/training', 'badtime_repro/training/methods', 'configs', 'dataset/ETTh1', 'tests']

if os.path.isdir("/kaggle/working"):
    PLATFORM = "kaggle"
    PROJECT = "/kaggle/working/Badtime"
elif "google.colab" in sys.modules or os.path.isdir("/content"):
    PLATFORM = "colab"
    PROJECT = "/content/Badtime"
else:
    PLATFORM = "local"
    PROJECT = os.path.abspath("Badtime")

for d in PACKAGE_DIRS:
    os.makedirs(os.path.join(PROJECT, d), exist_ok=True)

os.chdir(PROJECT)
print("platform:", PLATFORM, "| project:", PROJECT)

import torch, numpy, pandas
print("python", platform.python_version(), "| torch", torch.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    print("CẢNH BÁO: không có GPU, huấn luyện sẽ rất chậm.")

try:
    import pytest  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)

SET_ARGS = " ".join(f"--set {s}" for s in EXTRA_SET)
SEEDS_ARG = " ".join(str(s) for s in SEEDS)

## 3. Ghi mã nguồn ra đĩa

Mỗi ô dưới đây là một file của dự án, giống hệt bản trong folder `Badtime` của bạn. TimesNet được vendored từ Time-Series-Library@`4e938a17` (MIT). Các ô đã thu gọn; mở ra để xem.

In [ ]:
%%writefile badtime_repro/__init__.py
"""Tái lập độc lập BadTime (arXiv 2508.04189v2), slot PoC ETTh1 + TimesNet, 96 -> 96.

Các tầng (một module chỉ import cùng tầng hoặc tầng thấp hơn; tests/test_architecture.py kiểm tra tự động):

    9  cli, __main__                         giao diện dòng lệnh
    8  evaluation (controls, report, ...)    đối chứng 4 điều kiện, tính lại chỉ số, báo cáo
    7  experiment                            mặt tiền: kiểm tra -> dữ liệu -> huấn luyện
    6  training (Trainer, TrainingMethod)    vòng huấn luyện chung + các phương pháp          [torch]
    5  models                                mô hình dự báo (TimesNet vendored)               [torch]
    4  metrics, runs, support                hợp đồng phép đo, hiện vật lần chạy, kiểm tra hỗ trợ
    3  attack (plan, ... | trigger, gat)     khối thiết kế BadTime (numpy) | phần torch
    2  data                                  nguồn, phân chia, chuẩn hóa, cửa sổ
    1  catalog, config                       danh mục, schema cấu hình
    0  common, assumptions                   lỗi, registry, làm tròn, JSON; sổ giả định
"""

__version__ = "0.2.0"

In [ ]:
%%writefile badtime_repro/__main__.py
import sys

from .cli import main

if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile badtime_repro/assumptions.py
"""Sổ giả định đang áp dụng trong mã. Được ghi vào hồ sơ (profile.json) của MỌI lần chạy.

status: "source" = có trong bài báo v2 (2508.04189v2.pdf); "proposal" = đề xuất của dự án;
"unknown" = bài báo không xác định, lựa chọn hiện tại ảnh hưởng trực tiếp con số.
`config` trỏ tới trường cấu hình điều khiển giả định đó (nếu có).
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Dict, Optional, Tuple

STATUSES = ("source", "proposal", "unknown")


@dataclass(frozen=True)
class Assumption:
    id: str
    topic: str
    choice: str
    status: str
    config: Optional[str] = None
    note: Optional[str] = None

    def __post_init__(self) -> None:
        if self.status not in STATUSES:
            raise ValueError(f"{self.id}: status không hợp lệ {self.status}")

    def to_dict(self) -> Dict[str, Optional[str]]:
        return {k: v for k, v in asdict(self).items() if v is not None}


_RAW = [
    {
        "id": "A01",
        "topic": "Phiên bản thư viện nền",
        "choice": (
            "TimesNet vendored từ thuml/Time-Series-Library@4e938a17 (18/04/2026); siêu tham số theo "
            "scripts/long_term_forecast/ETT_script/TimesNet_ETTh1.sh"
        ),
        "status": "proposal",
        "config": "timesnet",
        "note": "v2 mục V-A chỉ nói dùng kiến trúc và siêu tham số huấn luyện của TSLib, không nêu commit.",
    },
    {
        "id": "A02",
        "topic": "Phân chia dữ liệu",
        "choice": (
            "6/2/2 trên toàn chuỗi 17420 hàng: train [0, 10452), val [10452, 13936), test [13936, 17420). Ranh giới "
            "12/4/4 tháng của TSLib là cấu hình đối chiếu riêng."
        ),
        "status": "source",
        "config": "data.split_protocol",
        "note": (
            "v2 Bảng XI: ETTh1 17,420 mốc, tỉ lệ 6/2/2. Ranh giới cụ thể là do dự án tính (int(0.6N), int(0.2N))."
        ),
    },
    {
        "id": "A03",
        "topic": "Lịch sử từ phân vùng trước",
        "choice": (
            "Cửa sổ val/test được dùng tối đa seq_len bước lịch sử thuộc phân vùng trước; nhãn luôn nằm trọn trong "
            "phân vùng."
        ),
        "status": "proposal",
        "config": "data.allow_history_from_previous_split",
    },
    {
        "id": "A04",
        "topic": "Chuẩn hóa",
        "choice": (
            "Chuẩn hóa từng kênh bằng mean/std (ddof=0) của phần huấn luyện; báo cáo cả thang chuẩn hóa và đơn vị "
            "gốc."
        ),
        "status": "proposal",
    },
    {
        "id": "A05",
        "topic": "Kênh đích",
        "choice": "OT. alpha_S = 0.1 x 7 kênh = 0.7, làm tròn thành 1 kênh. Tên kênh được nêu tường minh.",
        "status": "unknown",
        "config": "attack.target_channels",
        "note": "v2 không nêu tên kênh đích của ETTh1.",
    },
    {
        "id": "A06",
        "topic": "Công thức số của Hump",
        "choice": (
            "Nội suy tuyến tính 7 điểm 'cone' của BackTime [0.1278 ... 0.3853 ... 0.1278] lên 53 điểm; giá trị tuyệt "
            "đối ở thang chuẩn hóa, amplitude 1, baseline 0."
        ),
        "status": "unknown",
        "config": "attack.pattern",
        "note": (
            "v2 Bảng X dùng Hump/Upward/Up-and-down, tương ứng cone/up_trend/up_and_down của BackTime, nhưng không "
            "cho số. Có thể chọn 'hump_sine' làm dạng thay thế. Biên độ ảnh hưởng trực tiếp MAE_pa."
        ),
    },
    {
        "id": "A07",
        "topic": "Làm tròn số mẫu và số kênh",
        "choice": (
            "k_sim = floor(gamma alpha_T |D|) = 307, k_diff = floor((1-gamma) alpha_T |D|) = 718; số kênh bị sửa "
            "floor(0.3 x 7) = 2."
        ),
        "status": "source",
        "config": "attack.poison_ratio / similar_ratio / poison_channel_ratio",
        "note": (
            "Floor theo Thuật toán 1 dòng 7-8. TopK alpha_P|V| (Thuật toán 2) dùng floor là lựa chọn của dự án."
        ),
    },
    {
        "id": "A08",
        "topic": "Chia ngân sách trigger",
        "choice": (
            "keep_total: 29 = 15 + 14, đoạn dài hơn cho kênh có ảnh hưởng cao hơn. Cấu hình thay thế: equal_segments "
            "14 + 14 = 28, ceil_segments 15 + 15 = 30."
        ),
        "status": "unknown",
        "config": "attack.budget_protocol",
        "note": "v2 viết độ dài đoạn t_TGR/|V_p| = 14.5, không nguyên.",
    },
    {
        "id": "A09",
        "topic": "Huấn luyện GAT",
        "choice": (
            "Nguồn: 1 tầng, 4 head, self-loop, z-score theo phần huấn luyện, dự báo bước kế tiếp trên V_t, attention "
            "trung bình theo head. Đề xuất: đặc trưng nút là cửa sổ 96 bước; hidden 64; Adam lr 1e-3; 20 epoch; batch"
            " 256; LeakyReLU 0.2."
        ),
        "status": "proposal",
        "config": "attack.gat",
    },
    {
        "id": "A10",
        "topic": "Kênh bị sửa có thể trùng kênh đích",
        "choice": "Cho phép (không loại kênh đích khỏi ứng viên).",
        "status": "source",
        "config": "attack.exclude_target_from_poison_channels",
        "note": "v2 Observation 2: V_p thường gồm cả kênh đích.",
    },
    {
        "id": "A11",
        "topic": "Tính độ trễ",
        "choice": (
            "Nguồn (Thuật toán 3): delta trong [ceil(t_TGR/|V_p|), t_IN] = [15, 96]; cộng tương quan Pearson qua mọi "
            "cửa sổ và mọi kênh đích. Đề xuất: cửa sổ là mọi đầu vào nằm trong phần huấn luyện; bỏ delta có t_IN - "
            "delta < 8; bỏ cửa sổ có phương sai ~0."
        ),
        "status": "proposal",
        "config": "attack.lag.min_corr_len",
    },
    {
        "id": "A12",
        "topic": "Khoảng cách mục tiêu khi chọn mẫu và khởi tạo",
        "choice": (
            "Euclid thường giữa nhãn [0, t_pin) của kênh đích và p, ở thang chuẩn hóa. t* lấy trong các nhãn của cửa "
            "sổ huấn luyện."
        ),
        "status": "proposal",
        "note": "Euclid theo Thuật toán 1 và Công thức 8; thang đo là lựa chọn của dự án.",
    },
    {
        "id": "A13",
        "topic": "Phép tổng hợp mất mát và trọng số",
        "choice": (
            "Tổng trên phần tử của mỗi mẫu rồi chia số mẫu (đúng cách viết Công thức 16-18). alpha = (1,1,1), beta = "
            "(1,1,1,1)."
        ),
        "status": "unknown",
        "config": "attack.loss_reduction / model_loss_weights / trigger_loss_weights",
        "note": "v2: các trọng số đặt theo kinh nghiệm và có tăng trọng số L_p^{V_t}, nhưng không cho giá trị.",
    },
    {
        "id": "A14",
        "topic": "Phạm vi mất mát kênh đích",
        "choice": (
            "L_p^{V_t} tính trên toàn khoảng dự báo: [0, t_pin) so với p, [t_pin, P) so với nhãn gốc (Công thức 14, "
            "17)."
        ),
        "status": "source",
    },
    {
        "id": "A15",
        "topic": "Lịch huấn luyện",
        "choice": (
            "Adam lr 1e-4, giảm 1/2 mỗi epoch (TSLib type1), batch 32, 10 epoch. Clean dừng sớm với patience 3 theo "
            "MSE val; BadTime chạy đủ 10 epoch."
        ),
        "status": "proposal",
        "config": "train",
    },
    {
        "id": "A16",
        "topic": "Cập nhật trigger",
        "choice": (
            "Nguồn: mỗi M = 2 epoch, cố định mô hình, duyệt các mẫu bị chọn, eps làm trơn 0.8. Đề xuất: sau các epoch"
            " có (epoch+1) % 2 == 0; 1 lượt; Adam lr 0.01 giữ trạng thái; batch 32; mô hình ở chế độ eval."
        ),
        "status": "proposal",
        "config": "attack.trigger_*",
    },
    {
        "id": "A17",
        "topic": "Chọn cặp trạng thái",
        "choice": (
            "Cặp (mô hình, trigger) cuối mỗi epoch. Ứng viên: MAE_c(val) <= 1.05 x min MAE_c(val); trong đó chọn "
            "MAE_pa(val) nhỏ nhất."
        ),
        "status": "proposal",
        "config": "attack.state_selection.clean_tolerance",
    },
    {
        "id": "A18",
        "topic": "Loss của nhánh sạch",
        "choice": "MSE như TSLib (mặc định). Dùng mse_mae để kiểm tra đường xử lý so với BadTime đã vô hiệu hóa.",
        "status": "proposal",
        "config": "train.clean_loss",
    },
    {
        "id": "A19",
        "topic": "Đánh giá có trigger",
        "choice": "Gắn trigger vào MỌI cửa sổ hợp lệ của tập đánh giá; báo cả val và test.",
        "status": "proposal",
        "note": (
            "v2 không nói rõ tập dùng cho Bảng IV/V; với phát hiện bất thường thì nói 'poisoned validation set'."
        ),
    },
    {
        "id": "A20",
        "topic": "Thang đo của số bài báo",
        "choice": "Giả định số trong bài báo ở thang chuẩn hóa (như TSLib); chưa xác nhận.",
        "status": "unknown",
    },
    {
        "id": "A21",
        "topic": "Dự báo TimesNet phụ thuộc lô",
        "choice": (
            "FFT_for_Period lấy trung bình biên độ trên cả lô, nên dự báo phụ thuộc thành phần lô. Đánh giá theo thứ "
            "tự thời gian, lô 32, không xáo; mọi nhánh dùng cùng quy tắc."
        ),
        "status": "proposal",
        "config": "train.eval_batch_size",
    },
]

ASSUMPTIONS: Tuple[Assumption, ...] = tuple(Assumption(**d) for d in _RAW)

In [ ]:
%%writefile badtime_repro/catalog.py
"""Danh mục lựa chọn ở mức thiết kế và ma trận hỗ trợ (docs/07-configuration-support.md).

Danh mục chỉ ghi lựa chọn nào tồn tại và đang ở trạng thái nào. Phần cài đặt nằm ở các registry riêng
(`data.sources.DATA_SOURCES`, `models.MODEL_FACTORIES`, `training.methods.TRAINING_METHODS`). `tests/test_catalog.py`
buộc mọi mục có thể chạy phải có cài đặt tương ứng.

Thêm một tên vào danh mục KHÔNG làm lựa chọn đó chạy được.

Trạng thái phản ánh bằng chứng hiện có, không phải mong muốn: một thành phần chỉ lên SUPPORTED khi các kiểm tra
tương ứng đã thực sự chạy đạt. Phần torch (TimesNet, clean, badtime) chưa được chạy ở môi trường viết mã, nên còn
IMPLEMENTED; nâng lên SUPPORTED sau khi tests/test_torch_units.py và tests/test_training.py chạy đạt.
"""

from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from typing import Dict, FrozenSet, Mapping, Optional, Tuple, TypeVar


class SupportStatus(str, Enum):
    DESCRIBED = "described_not_implemented"     # Có mô tả, chưa triển khai
    IMPLEMENTED = "implemented_not_verified"    # Đã triển khai, chưa kiểm chứng
    SUPPORTED = "supported_in_scope"            # Đã hỗ trợ trong phạm vi

    @property
    def runnable(self) -> bool:
        return self is not SupportStatus.DESCRIBED


@dataclass(frozen=True)
class DatasetSpec:
    name: str
    folder: str                       # tương đối với gốc dự án, dùng "/"
    status: SupportStatus
    source: str = ""                  # khóa trong data.sources.DATA_SOURCES
    file_name: str = ""
    expected_columns: Tuple[str, ...] = ()
    freq: str = ""
    note: str = ""


@dataclass(frozen=True)
class ModelSpec:
    name: str
    status: SupportStatus
    source: str = ""
    note: str = ""


@dataclass(frozen=True)
class MethodSpec:
    name: str
    status: SupportStatus
    note: str = ""


ETT_COLUMNS = ("date", "HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT")

_D, _I, _S = SupportStatus.DESCRIBED, SupportStatus.IMPLEMENTED, SupportStatus.SUPPORTED

T = TypeVar("T")


@dataclass(frozen=True)
class Catalog:
    datasets: Mapping[str, DatasetSpec]
    models: Mapping[str, ModelSpec]
    methods: Mapping[str, MethodSpec]
    seq_lens: Mapping[int, SupportStatus]
    pred_lens: Mapping[int, SupportStatus]
    supported_slots: FrozenSet[Tuple[str, str]]

    @staticmethod
    def canonical(name: object, table: Mapping[str, T]) -> Optional[str]:
        """Nhận diện tên không phân biệt hoa/thường. Trả về tên chuẩn hoặc None."""
        if not isinstance(name, str):
            return None

        lowered = name.strip().lower()
        for key in table:
            if key.lower() == lowered:
                return key

        return None

    def slot_supported(self, dataset: str, model: str) -> bool:
        return (dataset, model) in self.supported_slots


def _datasets() -> Dict[str, DatasetSpec]:
    return {
        "ETTh1": DatasetSpec("ETTh1", "dataset/ETTh1", _S, "ett_csv", "ETTh1.csv", ETT_COLUMNS, "h", "Slot PoC"),
        "ETTh2": DatasetSpec("ETTh2", "dataset/ETTh2", _D, "ett_csv", "ETTh2.csv", ETT_COLUMNS, "h"),
        "ETTm1": DatasetSpec("ETTm1", "dataset/ETTm1", _D, "ett_csv", "ETTm1.csv", ETT_COLUMNS, "t"),
        "ETTm2": DatasetSpec("ETTm2", "dataset/ETTm2", _D, "ett_csv", "ETTm2.csv", ETT_COLUMNS, "t"),
        "Electricity": DatasetSpec("Electricity", "dataset/Electricity", _D),
        "Weather": DatasetSpec("Weather", "dataset/Weather", _D),
        "Exchange": DatasetSpec("Exchange", "dataset/Exchange", _D),
    }


def _models() -> Dict[str, ModelSpec]:
    return {
        "Autoformer": ModelSpec("Autoformer", _D),
        "Mamba": ModelSpec("Mamba", _D),
        "TimesNet": ModelSpec(
            "TimesNet", _I, source="thuml/Time-Series-Library@4e938a1767106324dd753b2a44832bf870a0252e",
            note="vendored, chưa chạy kiểm tra torch",
        ),
        "iTransformer": ModelSpec("iTransformer", _D),
        "TSMixer": ModelSpec("TSMixer", _D),
        "TimeMixer": ModelSpec("TimeMixer", _D),
    }


def _methods() -> Dict[str, MethodSpec]:
    return {
        "clean": MethodSpec("clean", _I, "Huấn luyện sạch (Benign), chưa chạy kiểm tra torch"),
        "badtime": MethodSpec("badtime", _I, "BadTime tái lập độc lập, còn giả định của dự án"),
        "backtime": MethodSpec(
            "backtime", _D, "Đối chứng BackTime: chưa chuyển mã xiaolin-cs/BackTime sang giao thức này"
        ),
    }


CATALOG = Catalog(
    datasets=_datasets(),
    models=_models(),
    methods=_methods(),
    seq_lens={96: _S},
    pred_lens={96: _S, 192: _D, 336: _D, 720: _D},
    # Ô duy nhất của ma trận dataset x mô hình thuộc phạm vi PoC.
    supported_slots=frozenset({("ETTh1", "TimesNet")}),
)

In [ ]:
%%writefile badtime_repro/cli.py
"""Giao diện dòng lệnh. Chạy từ gốc dự án:

    python -m badtime_repro check   [--config FILE] [--set KHOA=GIATRI ...] [--method M]
    python -m badtime_repro train   --method {clean,badtime} [--seeds S ...] [--config FILE] [--set ...]
    python -m badtime_repro controls --clean-run DIR --attack-run DIR
    python -m badtime_repro report  [--config FILE] [--set ...]
    python -m badtime_repro recompute --run DIR

Mã thoát: 0 thành công, 1 lỗi khi chạy, 2 cấu hình không hợp lệ, 3 chưa hỗ trợ, 4 dữ liệu thiếu/không hợp lệ.
"""

from __future__ import annotations

import argparse
import logging
import os
import sys
from abc import ABC, abstractmethod
from typing import Dict, List, Optional, Sequence

from .common.errors import BadTimeError, ConfigInvalidError
from .config.loader import load_config
from .config.schema import ExperimentConfig
from .evaluation.recompute import MetricRecomputer
from .evaluation.report import MarkdownReportRenderer, ReportBuilder
from .experiment import Experiment
from .support import SupportChecker, SupportVerdict

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DEFAULT_CONFIG = os.path.join(PROJECT_ROOT, "configs", "etth1_timesnet_96.json")


class Command(ABC):
    name: str = ""
    help: str = ""
    uses_config: bool = True

    def __init__(self, project_root: str):
        self.project_root = project_root

    def configure(self, parser: argparse.ArgumentParser) -> None:
        if self.uses_config:
            parser.add_argument("--config", default=DEFAULT_CONFIG)
            parser.add_argument("--set", action="append", default=[], metavar="KHOA=GIATRI")

    def load(self, args: argparse.Namespace) -> ExperimentConfig:
        return load_config(args.config, args.set)

    @abstractmethod
    def run(self, args: argparse.Namespace) -> int:
        ...


class CheckCommand(Command):
    name = "check"
    help = "Kiểm tra tổ hợp + dữ liệu, không huấn luyện"

    def configure(self, parser: argparse.ArgumentParser) -> None:
        super().configure(parser)
        parser.add_argument("--method", default=None)

    def run(self, args: argparse.Namespace) -> int:
        cfg = self.load(args)
        if args.method:
            cfg = cfg.replace(method=args.method)

        cfg, dataset = Experiment(self.project_root).resolve(cfg)
        r = dataset.report
        print(f"Tổ hợp hợp lệ: {cfg.dataset} + {cfg.model} + {cfg.method}, {cfg.seq_len} -> {cfg.pred_len}")
        print(f"Dữ liệu: {r.path}")
        print(f"  sha256 {r.sha256}")
        print(f"  {r.n_rows} hàng, {r.start} -> {r.end}, kênh {r.channels}")
        print(f"  thiếu {r.n_missing}, trùng mốc {r.n_duplicate_timestamps}, bước không đều {r.n_irregular_steps}")
        for name, (s, e) in dataset.bounds.items():
            st = dataset.starts[name]
            print(f"  {name:5s} hàng [{s}, {e})  {len(st)} cửa sổ, s in [{st[0]}, {st[-1]}]")

        return 0


class TrainCommand(Command):
    name = "train"
    help = "Huấn luyện một nhánh với một hoặc nhiều seed"

    def configure(self, parser: argparse.ArgumentParser) -> None:
        super().configure(parser)
        parser.add_argument("--method", required=True)
        parser.add_argument("--seeds", type=int, nargs="+", default=[2021])

    def run(self, args: argparse.Namespace) -> int:
        cfg = self.load(args)
        experiment = Experiment(self.project_root)
        for seed in args.seeds:
            experiment.train(cfg, args.method, seed)

        return 0


class ControlsCommand(Command):
    name = "controls"
    help = "Đối chứng bốn điều kiện mô hình x đầu vào"
    uses_config = False

    def configure(self, parser: argparse.ArgumentParser) -> None:
        parser.add_argument("--clean-run", required=True)
        parser.add_argument("--attack-run", required=True)

    def run(self, args: argparse.Namespace) -> int:
        from .evaluation.controls import ControlExperiment

        ControlExperiment(self.project_root).run(args.clean_run, args.attack_run)
        return 0


class ReportCommand(Command):
    name = "report"
    help = "Tổng hợp các lần chạy so với bài báo"

    def run(self, args: argparse.Namespace) -> int:
        check = SupportChecker().check(self.load(args))
        if check.verdict is SupportVerdict.INVALID:
            raise ConfigInvalidError(check.messages)

        cfg = check.config
        builder = ReportBuilder(os.path.join(self.project_root, cfg.runs_root, cfg.slot), cfg.slot)
        report = builder.build()
        builder.write(report)
        print(MarkdownReportRenderer().render(report))
        return 0


class RecomputeCommand(Command):
    name = "recompute"
    help = "Tính lại chỉ số từ dự báo đã lưu"
    uses_config = False

    def configure(self, parser: argparse.ArgumentParser) -> None:
        parser.add_argument("--run", required=True)

    def run(self, args: argparse.Namespace) -> int:
        result = MetricRecomputer().run(args.run)
        print(result)
        return 0 if all(v["ok"] for v in result.values()) else 1


COMMANDS = (CheckCommand, TrainCommand, ControlsCommand, ReportCommand, RecomputeCommand)


class Application:
    def __init__(self, project_root: str = PROJECT_ROOT):
        self.commands: Dict[str, Command] = {cls.name: cls(project_root) for cls in COMMANDS}
        self.parser = argparse.ArgumentParser(prog="badtime_repro")
        sub = self.parser.add_subparsers(dest="cmd", required=True)
        for name, command in self.commands.items():
            command.configure(sub.add_parser(name, help=command.help))

    def main(self, argv: Optional[Sequence[str]] = None) -> int:
        _configure_io()
        args = self.parser.parse_args(argv)
        try:
            return self.commands[args.cmd].run(args)
        except BadTimeError as err:
            for message in err.messages:
                print(message, file=sys.stderr)

            return err.exit_code


def _configure_io() -> None:
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8", errors="replace")
        except (AttributeError, ValueError):
            pass

    root = logging.getLogger("badtime_repro")
    if not root.handlers:
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(logging.Formatter("%(message)s"))
        root.addHandler(handler)
        root.setLevel(logging.INFO)
        root.propagate = False


def main(argv: Optional[List[str]] = None) -> int:
    return Application().main(argv)

In [ ]:
%%writefile badtime_repro/experiment.py
"""Mặt tiền (facade) của ứng dụng: kiểm tra tổ hợp -> chuẩn bị dữ liệu -> huấn luyện một lần chạy.

`resolve` không cần torch (dùng cho lệnh `check`); `train` mới import tầng huấn luyện.
"""

from __future__ import annotations

import logging
import os
from dataclasses import dataclass, field
from typing import Tuple

from .config.schema import ExperimentConfig
from .data.dataset import PreparedDataset
from .data.pipeline import DataPipeline
from .runs.store import RunStore
from .support import SupportChecker

log = logging.getLogger(__name__)


@dataclass
class Experiment:
    project_root: str
    checker: SupportChecker = field(default_factory=SupportChecker)

    def resolve(self, cfg: ExperimentConfig) -> Tuple[ExperimentConfig, PreparedDataset]:
        """Ném ConfigInvalidError / UnsupportedSelectionError / DataUnavailableError nếu không chạy được."""
        report = self.checker.check(cfg)
        resolved = report.raise_if_blocked()
        for message in report.messages:
            log.info(message)

        return resolved, DataPipeline(self.project_root).prepare(resolved)

    def train(self, cfg: ExperimentConfig, method: str, seed: int) -> RunStore:
        import torch

        from .training.context import RunContext, resolve_device
        from .training.methods import TRAINING_METHODS
        from .training.trainer import Trainer

        resolved, dataset = self.resolve(cfg.replace(method=method))
        if resolved.train.deterministic:
            os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
            torch.use_deterministic_algorithms(True)

        store = RunStore.create(self.project_root, resolved, resolved.method, seed)
        store.mark_running(resolved.method, seed)
        log.info("Thư mục lần chạy: %s", store.path)
        try:
            ctx = RunContext(resolved, dataset, resolve_device(resolved.train.device), seed, store)
            method_impl = TRAINING_METHODS.resolve(resolved.method)(ctx)
            outcome = Trainer(ctx, method_impl).run()
        except BaseException as exc:
            store.mark_failed(exc)
            raise

        store.mark_completed(outcome.selected_epoch)
        return store

In [ ]:
%%writefile badtime_repro/metrics.py
"""Hợp đồng phép đo (docs/06). Thuần numpy để tính lại được từ dự báo đã lưu.

Mọi sai số là trung bình theo từng giá trị hợp lệ (không lấy trung bình không trọng số giữa các lô).
- mae_c / mse_c        : dự báo từ đầu vào sạch vs nhãn gốc, mọi kênh, toàn khoảng dự báo
- mae_pa / mse_pa      : dự báo từ đầu vào có trigger vs mục tiêu, kênh đích, đoạn [0, t_pin)
- mae_pn / mse_pn      : dự báo từ đầu vào có trigger vs nhãn gốc, kênh ngoài đích, toàn khoảng dự báo
- mae_rest / mse_rest  : dự báo từ đầu vào có trigger vs nhãn gốc, kênh đích, đoạn [t_pin, pred_len)
Mỗi phép đo có hai thang: chuẩn hóa (*_norm) và đơn vị gốc (*_orig).
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np

from .attack.plan import AttackPlan
from .data.scaling import StandardScaler

Metrics = Dict[str, float]


@dataclass(frozen=True)
class AttackTarget:
    """Phần của kế hoạch tác động mà phép đo cần."""

    target_idx: Tuple[int, ...]
    pattern: np.ndarray            # [t_pin, n_target], thang chuẩn hóa

    @classmethod
    def of(cls, plan: AttackPlan) -> "AttackTarget":
        return cls(tuple(plan.target_idx), plan.pattern)


class MetricCalculator:
    def __init__(self, scaler: StandardScaler):
        self.scaler = scaler

    # ----------------------------------------------------------------------------- dự báo

    def forecast(self, pred_clean: np.ndarray, y: np.ndarray,
                 pred_trig: Optional[np.ndarray] = None, target: Optional[AttackTarget] = None) -> Metrics:
        """pred_*, y: [n, P, C] thang chuẩn hóa."""
        all_ch = list(range(y.shape[-1]))
        out: Metrics = {}
        self._pair(out, "c", pred_clean, y, all_ch, all_ch)
        if pred_trig is None:
            return out

        if target is None:
            raise ValueError("Cần AttackTarget khi có dự báo từ đầu vào có trigger.")

        tgt = list(target.target_idx)
        other = [c for c in all_ch if c not in tgt]
        t_pin = target.pattern.shape[0]
        pat = np.broadcast_to(target.pattern[None, :, :], (len(y), t_pin, len(tgt)))

        self._pair(out, "pa", pred_trig[:, :t_pin][:, :, tgt], pat, tgt, tgt)
        if other:
            self._pair(out, "pn", pred_trig[:, :, other], y[:, :, other], other, other)

        if t_pin < y.shape[1]:
            self._pair(out, "rest", pred_trig[:, t_pin:][:, :, tgt], y[:, t_pin:][:, :, tgt], tgt, tgt)

        return out

    def _pair(self, out: Metrics, tag: str, pred: np.ndarray, ref: np.ndarray,
              pred_ch: List[int], ref_ch: List[int]) -> None:
        mae, mse, cnt = _mae_mse(pred, ref)
        out[f"mae_{tag}_norm"], out[f"mse_{tag}_norm"] = mae, mse
        mae, mse, _ = _mae_mse(self.scaler.inverse(pred, pred_ch), self.scaler.inverse(ref, ref_ch))
        out[f"mae_{tag}_orig"], out[f"mse_{tag}_orig"] = mae, mse
        out[f"n_values_{tag}"] = cnt

    # ----------------------------------------------------------------------------- mức sửa đầu vào

    @staticmethod
    def input_change(x_clean: np.ndarray, x_trig: np.ndarray, plan: AttackPlan) -> Dict[str, float]:
        """Số ô thay đổi, độ lớn thay đổi, bước nhảy tại biên nối đoạn (thang chuẩn hóa)."""
        diff = np.abs(np.asarray(x_trig) - np.asarray(x_clean))
        changed = diff > 0
        jumps = []
        for (t_s, t_e), ch, off in plan.segments():
            length = t_e - t_s
            if off > 0:
                jumps.append(np.abs(x_trig[:, off, ch] - x_trig[:, off - 1, ch]))

            if off + length < x_trig.shape[1]:
                jumps.append(np.abs(x_trig[:, off + length, ch] - x_trig[:, off + length - 1, ch]))

        jumps_all = np.concatenate(jumps) if jumps else np.zeros(1)
        return {
            "changed_cells_per_window_mean": float(changed.sum(axis=(1, 2)).mean()),
            "registered_cells_per_window": int(plan.trigger_len),
            "abs_change_mean_over_registered_cells": float(diff.sum() / (len(x_clean) * plan.trigger_len)),
            "abs_change_max": float(diff.max()),
            "boundary_jump_mean": float(jumps_all.mean()),
            "boundary_jump_max": float(jumps_all.max()),
        }


def _mae_mse(a: np.ndarray, b: np.ndarray) -> Tuple[float, float, int]:
    d = np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)
    return float(np.abs(d).mean()), float((d * d).mean()), int(d.size)

In [ ]:
%%writefile badtime_repro/support.py
"""Kiểm tra TOÀN BỘ tổ hợp trước khi chạy (docs/07). Không bao giờ tự chuyển sang slot PoC.

Nằm ở tầng ứng dụng (trên config, data, attack) vì cần biết tên các giao thức mà các tầng đó cài đặt.

Ba kết luận tách biệt:
- INVALID      : tên không có trong danh mục hoặc giá trị mâu thuẫn
- UNSUPPORTED  : hợp lệ về thiết kế nhưng chưa triển khai -> dừng trước huấn luyện
- OK           : được phép chạy (vẫn còn bước kiểm tra dữ liệu)
"""

from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Iterable, List, Optional, Tuple

from .attack.budget import BUDGET_PROTOCOLS
from .attack.patterns import PATTERNS
from .catalog import CATALOG, Catalog, SupportStatus
from .common.errors import ConfigInvalidError, UnsupportedSelectionError
from .config.schema import ExperimentConfig
from .data.splits import SPLIT_PROTOCOLS

# Tên hợp lệ của các lựa chọn mà phần cài đặt nằm ở tầng torch (không import được ở đây).
# tests/test_config.py đối chiếu các bộ tên này với phần cài đặt tương ứng.
CLEAN_LOSSES = ("mse", "mse_mae")
LOSS_REDUCTIONS = ("mean", "sum")
LR_SCHEDULES = ("type1", "constant")
TIMESNET_EMBEDS = ("timeF",)
PATTERN_SCALES = ("normalized",)


class SupportVerdict(str, Enum):
    OK = "ok"
    UNSUPPORTED = "unsupported"
    INVALID = "invalid"


@dataclass(frozen=True)
class SupportReport:
    verdict: SupportVerdict
    messages: Tuple[str, ...] = ()
    config: Optional[ExperimentConfig] = None     # cấu hình với tên đã chuẩn hóa (None nếu INVALID ở bước tên)

    @property
    def ok(self) -> bool:
        return self.verdict is SupportVerdict.OK

    def raise_if_blocked(self) -> ExperimentConfig:
        if self.verdict is SupportVerdict.INVALID:
            raise ConfigInvalidError(self.messages)

        if self.verdict is SupportVerdict.UNSUPPORTED:
            raise UnsupportedSelectionError(self.messages)

        return self.config


@dataclass
class SupportChecker:
    catalog: Catalog = field(default_factory=lambda: CATALOG)

    def check(self, cfg: ExperimentConfig) -> SupportReport:
        cat = self.catalog
        dataset = cat.canonical(cfg.dataset, cat.datasets)
        model = cat.canonical(cfg.model, cat.models)
        method = cat.canonical(cfg.method, cat.methods)

        invalid = self._name_errors(cfg, dataset, model, method)
        if invalid:
            return SupportReport(SupportVerdict.INVALID, tuple(["Cấu hình không hợp lệ:"] + invalid))

        resolved = cfg.replace(dataset=dataset, model=model, method=method)
        blocked = self._support_errors(resolved)
        if blocked:
            return SupportReport(SupportVerdict.UNSUPPORTED, tuple(blocked), resolved)

        contradictions = self.consistency_errors(resolved)
        if contradictions:
            return SupportReport(SupportVerdict.INVALID, tuple(["Cấu hình không hợp lệ:"] + contradictions), resolved)

        notes = []
        for kind, spec in (("dataset", cat.datasets[dataset]), ("mô hình", cat.models[model]),
                           ("phương pháp", cat.methods[method])):
            if spec.status is SupportStatus.IMPLEMENTED:
                detail = f" ({spec.note})" if spec.note else ""
                notes.append(f"Lưu ý: {kind} {spec.name} ở trạng thái 'đã triển khai, chưa kiểm chứng'{detail}.")

        return SupportReport(SupportVerdict.OK, tuple(notes), resolved)

    # -----------------------------------------------------------------------------------------

    def _name_errors(self, cfg: ExperimentConfig, dataset: Optional[str], model: Optional[str],
                     method: Optional[str]) -> List[str]:
        cat = self.catalog
        errors = []
        if dataset is None:
            errors.append(f"Dataset '{cfg.dataset}' không có trong danh mục. Hợp lệ: {', '.join(cat.datasets)}.")

        if model is None:
            errors.append(f"Mô hình '{cfg.model}' không có trong danh mục. Hợp lệ: {', '.join(cat.models)}.")

        if method is None:
            errors.append(f"Phương pháp '{cfg.method}' không có trong danh mục. Hợp lệ: {', '.join(cat.methods)}.")

        if cfg.seq_len not in cat.seq_lens:
            errors.append(f"Độ dài lịch sử {cfg.seq_len} không có trong thiết kế. Hợp lệ: {list(cat.seq_lens)}.")

        if cfg.pred_len not in cat.pred_lens:
            errors.append(f"Khoảng dự báo {cfg.pred_len} không có trong thiết kế. Hợp lệ: {list(cat.pred_lens)}.")

        return errors

    def _support_errors(self, cfg: ExperimentConfig) -> List[str]:
        cat = self.catalog
        ds = cat.datasets[cfg.dataset]
        md = cat.models[cfg.model]
        mt = cat.methods[cfg.method]
        messages = []

        if not ds.status.runnable:
            messages.append(
                f"{ds.name} có trong thiết kế, nhưng chưa được hỗ trợ ở PoC. Thư mục dự kiến: {ds.folder}/. "
                "Phạm vi PoC là ETTh1 + TimesNet, dự báo 96 bước."
            )

        if not md.status.runnable:
            messages.append(f"{md.name} chưa được tích hợp cho PoC. Hiện chưa thể chạy tổ hợp {ds.name} + {md.name}.")

        if not messages and not cat.slot_supported(ds.name, md.name):
            messages.append(f"Tổ hợp {ds.name} + {md.name} chưa được kiểm tra; chưa hỗ trợ.")

        if not cat.seq_lens[cfg.seq_len].runnable:
            messages.append(f"Độ dài lịch sử {cfg.seq_len} chưa được hỗ trợ ở PoC.")

        if not cat.pred_lens[cfg.pred_len].runnable:
            messages.append(f"Khoảng dự báo {cfg.pred_len} nằm trong thiết kế mở rộng, chưa được hỗ trợ ở PoC.")

        if not mt.status.runnable:
            messages.append(f"Phương pháp {mt.name}: {mt.note}. Chưa thể chạy.")

        return messages

    @staticmethod
    def consistency_errors(cfg: ExperimentConfig) -> List[str]:
        """Kiểm tra ngữ nghĩa (schema đã đảm bảo đúng kiểu). Gom mọi lỗi, không dừng ở lỗi đầu tiên."""
        errors: List[str] = []
        tr, tn, att = cfg.train, cfg.timesnet, cfg.attack
        pat, gat = att.pattern, att.gat

        def one_of(path: str, value: str, allowed: Iterable[str]) -> None:
            allowed = list(allowed)
            if value not in allowed:
                errors.append(f"{path} = '{value}' không hợp lệ; phải thuộc {allowed}.")

        def at_least(path: str, value: float, low: float) -> None:
            if value < low:
                errors.append(f"{path} phải >= {low}, nhận {value}.")

        def within(path: str, value: float, low: float, high: float) -> None:
            if not (low <= value <= high):
                errors.append(f"{path} phải nằm trong [{low}, {high}], nhận {value}.")

        # Danh mục giá trị rời rạc
        one_of("data.split_protocol", cfg.data.split_protocol, SPLIT_PROTOCOLS)
        one_of("train.clean_loss", tr.clean_loss, CLEAN_LOSSES)
        one_of("train.lradj", tr.lradj, LR_SCHEDULES)
        one_of("timesnet.embed", tn.embed, TIMESNET_EMBEDS)
        one_of("attack.pattern.shape", pat.shape, PATTERNS)
        one_of("attack.pattern.scale", pat.scale, PATTERN_SCALES)
        one_of("attack.budget_protocol", att.budget_protocol, BUDGET_PROTOCOLS)
        one_of("attack.loss_reduction", att.loss_reduction, LOSS_REDUCTIONS)
        if not (tr.device in ("auto", "cpu") or tr.device == "cuda" or tr.device.startswith("cuda:")):
            errors.append(f"train.device = '{tr.device}' không hợp lệ; dùng 'auto', 'cpu', 'cuda' hoặc 'cuda:<i>'.")

        # Số nguyên dương
        for path, value in (
            ("train.epochs", tr.epochs), ("train.batch_size", tr.batch_size),
            ("train.eval_batch_size", tr.eval_batch_size),
            ("train.early_stopping_patience", tr.early_stopping_patience),
            ("timesnet.d_model", tn.d_model), ("timesnet.d_ff", tn.d_ff), ("timesnet.e_layers", tn.e_layers),
            ("timesnet.top_k", tn.top_k), ("timesnet.num_kernels", tn.num_kernels),
            ("attack.gat.hidden_dim", gat.hidden_dim), ("attack.gat.heads", gat.heads),
            ("attack.gat.epochs", gat.epochs), ("attack.gat.batch_size", gat.batch_size),
            ("attack.trigger_total_len", att.trigger_total_len),
            ("attack.trigger_update_every", att.trigger_update_every),
            ("attack.trigger_passes_per_update", att.trigger_passes_per_update),
            ("attack.trigger_batch_size", att.trigger_batch_size),
        ):
            at_least(path, value, 1)

        at_least("timesnet.label_len", tn.label_len, 0)
        at_least("attack.lag.min_corr_len", att.lag.min_corr_len, 2)      # tương quan cần ít nhất 2 điểm

        # Số thực
        for path, value in (("train.learning_rate", tr.learning_rate), ("attack.gat.learning_rate", gat.learning_rate),
                            ("attack.trigger_learning_rate", att.trigger_learning_rate)):
            if value <= 0:
                errors.append(f"{path} phải > 0, nhận {value}.")

        within("timesnet.dropout", tn.dropout, 0.0, 1.0)
        for key in ("poison_ratio", "similar_ratio", "poison_channel_ratio", "target_channel_ratio", "interp_lambda"):
            within(f"attack.{key}", getattr(att, key), 0.0, 1.0)

        at_least("attack.smooth_eps", att.smooth_eps, 0.0)
        at_least("attack.gat.leaky_slope", gat.leaky_slope, 0.0)
        at_least("attack.state_selection.clean_tolerance", att.state_selection.clean_tolerance, 0.0)
        for name, weights in (("model_loss_weights", att.model_loss_weights),
                              ("trigger_loss_weights", att.trigger_loss_weights)):
            if any(w < 0 for w in weights):
                errors.append(f"attack.{name} không được có trọng số âm.")

        # Quan hệ giữa các trường
        if not (1 <= pat.length <= cfg.pred_len):
            errors.append(f"attack.pattern.length = {pat.length} phải nằm trong [1, {cfg.pred_len}].")

        if att.trigger_total_len > cfg.seq_len:
            errors.append(f"attack.trigger_total_len = {att.trigger_total_len} vượt độ dài lịch sử {cfg.seq_len}.")

        if not att.target_channels:
            errors.append("attack.target_channels phải nêu rõ tên kênh đích; không suy từ thứ tự cột.")
        elif len(set(att.target_channels)) != len(att.target_channels):
            errors.append("attack.target_channels có tên kênh bị lặp.")

        return errors

In [ ]:
%%writefile badtime_repro/attack/__init__.py
"""Tầng tấn công.

Các module numpy (patterns, budget, channels, sample_selection, lag, plan, injection) không phụ thuộc torch.
Các module torch (trigger, objectives, gat) chỉ được import khi huấn luyện.
"""

In [ ]:
%%writefile badtime_repro/attack/budget.py
"""Chia tổng độ dài trigger t_TGR cho |V_p| kênh.

v2 ghi độ dài đoạn t_TGR/|V_p| (= 14.5, không nguyên): giả định A08.
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Dict, List

from ..common.errors import PlanError
from ..common.numeric import ceil_div


class BudgetProtocol(ABC):
    name: str = ""

    def split(self, total_len: int, n_channels: int) -> List[int]:
        if n_channels < 1:
            raise PlanError("Cần ít nhất 1 kênh bị sửa.")

        if n_channels > total_len:
            raise PlanError(f"Ngân sách trigger {total_len} nhỏ hơn số kênh bị sửa {n_channels}.")

        return self._split(total_len, n_channels)

    @abstractmethod
    def _split(self, total_len: int, n_channels: int) -> List[int]:
        ...


class KeepTotal(BudgetProtocol):
    """Giữ đúng tổng; các đoạn chênh nhau tối đa 1 bước, đoạn dài hơn đứng trước (29 -> 15 + 14)."""

    name = "keep_total"

    def _split(self, total_len: int, n_channels: int) -> List[int]:
        base = total_len // n_channels
        extra = total_len - base * n_channels
        return [base + 1 if i < extra else base for i in range(n_channels)]


class EqualSegments(BudgetProtocol):
    """Các đoạn bằng nhau floor(t_TGR / |V_p|); tổng thực tế có thể nhỏ hơn (29 -> 14 + 14)."""

    name = "equal_segments"

    def _split(self, total_len: int, n_channels: int) -> List[int]:
        return [total_len // n_channels] * n_channels


class CeilSegments(BudgetProtocol):
    """Các đoạn bằng nhau ceil(t_TGR / |V_p|) = L của Thuật toán 3; tổng thực tế có thể lớn hơn (29 -> 15 + 15)."""

    name = "ceil_segments"

    def _split(self, total_len: int, n_channels: int) -> List[int]:
        return [ceil_div(total_len, n_channels)] * n_channels


BUDGET_PROTOCOLS: Dict[str, BudgetProtocol] = {p.name: p for p in (KeepTotal(), EqualSegments(), CeilSegments())}


def get_budget_protocol(name: str) -> BudgetProtocol:
    if name not in BUDGET_PROTOCOLS:
        raise PlanError(f"budget_protocol không hợp lệ: {name}")

    return BUDGET_PROTOCOLS[name]

In [ ]:
%%writefile badtime_repro/attack/channels.py
"""Kênh đích V_t và kênh bị sửa V_p (Thuật toán 2).

Điểm ảnh hưởng do một `InfluenceScorer` cung cấp. BadTime dùng GAT (`attack.gat.GatInfluenceScorer`, cần torch);
`FixedInfluenceScorer` dùng cho kiểm thử và cho các biến thể bỏ thành phần (vd. w/o-GAT chọn ngẫu nhiên).
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np

from ..common.errors import PlanError
from ..common.numeric import floor_count, round_half_up


@dataclass(frozen=True)
class InfluenceResult:
    scores: np.ndarray                                   # [C], Influence(i) của từng kênh
    history: List[Dict[str, Any]] = field(default_factory=list)


class InfluenceScorer(ABC):
    @abstractmethod
    def score(self, train_norm: np.ndarray, seq_len: int, target_idx: Sequence[int], seed: int) -> InfluenceResult:
        """train_norm: [n_rows, C] phần huấn luyện đã chuẩn hóa."""


class FixedInfluenceScorer(InfluenceScorer):
    def __init__(self, scores: Sequence[float]):
        self.scores = np.asarray(scores, dtype=np.float64)

    def score(self, train_norm: np.ndarray, seq_len: int, target_idx: Sequence[int], seed: int) -> InfluenceResult:
        if len(self.scores) != train_norm.shape[1]:
            raise PlanError(f"Cần {train_norm.shape[1]} điểm ảnh hưởng, nhận {len(self.scores)}.")

        return InfluenceResult(self.scores.copy())


class TargetChannelResolver:
    """Kênh đích phải được nêu tên. Tỉ lệ alpha_S chỉ dùng để kiểm tra nhất quán với Bảng III."""

    def __init__(self, names: Sequence[str], ratio: float):
        self.names = list(names)
        self.ratio = ratio

    def resolve(self, channels: Sequence[str]) -> Tuple[List[int], Optional[str]]:
        idx = []
        for name in self.names:
            if name not in channels:
                raise PlanError(f"Kênh đích '{name}' không có trong dữ liệu {list(channels)}.")

            idx.append(list(channels).index(name))

        expected = max(1, round_half_up(self.ratio * len(channels)))
        note = None
        if expected != len(idx):
            note = f"Số kênh đích {len(idx)} khác tỷ lệ cấu hình ({self.ratio} x {len(channels)} -> {expected})."

        return idx, note


class PoisonChannelSelector:
    """V_p = TopK(Influence, k = floor(alpha_P * |V|)), tối thiểu 1; hòa điểm thì theo chỉ số kênh."""

    def __init__(self, ratio: float, exclude_targets: bool):
        self.ratio = ratio
        self.exclude_targets = exclude_targets

    def count(self, n_channels: int) -> int:
        return max(1, floor_count(self.ratio * n_channels))

    def select(self, influence: Sequence[float], target_idx: Sequence[int]) -> List[int]:
        n_poison = self.count(len(influence))
        exclude = set(target_idx) if self.exclude_targets else set()
        order = sorted(range(len(influence)), key=lambda j: (-influence[j], j))
        order = [j for j in order if j not in exclude]
        if len(order) < n_poison:
            raise PlanError(f"Không đủ kênh ứng viên ({len(order)}) để chọn {n_poison} kênh bị sửa.")

        return order[:n_poison]

In [ ]:
%%writefile badtime_repro/attack/gat.py
"""Chấm điểm ảnh hưởng kênh bằng GAT (Thuật toán 2, Công thức 4-5 của v2).

Nguồn:
- Đồ thị có hướng đầy đủ trên |V| kênh, láng giềng vào của j gồm cả chính j.
- Một tầng GAT, n_head = 4, chuỗi được z-score trên phần huấn luyện.
- Nhiệm vụ phụ trợ: dự báo bước kế tiếp trên các kênh đích V_t.
- alpha_ij lấy trung bình theo head; Influence(i) = sum_{j in V_t} mean alpha_ij.

ĐỀ XUẤT của dự án (A09), vì bài báo không nêu:
- Đặc trưng nút là cửa sổ seq_len bước của kênh (thay cho cả chuỗi H[i, :]), và nhãn là giá trị bước kế tiếp.
- Hidden 64 mỗi head; nối các head, ELU, Linear(heads * hidden -> 1); MSE; Adam lr 1e-3; 20 epoch; batch 256.
- Attention được lấy trung bình trên mọi cửa sổ huấn luyện.
"""

from __future__ import annotations

from typing import Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from ..config.schema import GatConfig
from .channels import InfluenceResult, InfluenceScorer


class GraphAttentionNextStep(nn.Module):
    def __init__(self, in_len: int, hidden_dim: int, heads: int, leaky_slope: float, self_loops: bool):
        super().__init__()
        self.heads = heads
        self.hidden_dim = hidden_dim
        self.w = nn.Linear(in_len, heads * hidden_dim, bias=False)
        self.a_src = nn.Parameter(torch.empty(heads, hidden_dim))
        self.a_dst = nn.Parameter(torch.empty(heads, hidden_dim))
        self.head = nn.Linear(heads * hidden_dim, 1)
        self.leaky_slope = leaky_slope
        self.self_loops = self_loops
        nn.init.xavier_uniform_(self.w.weight)
        nn.init.xavier_uniform_(self.a_src)
        nn.init.xavier_uniform_(self.a_dst)

    def attention(self, h: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """h: [B, N, L]. Trả về alpha [B, heads, N_dst(j), N_src(i)] (tổng theo i bằng 1) và W h [B, heads, N, H]."""
        b, n, _ = h.shape
        wh = self.w(h).view(b, n, self.heads, self.hidden_dim).permute(0, 2, 1, 3)
        src = (wh * self.a_src[None, :, None, :]).sum(-1)  # a^T phần W h_i (nguồn)
        dst = (wh * self.a_dst[None, :, None, :]).sum(-1)  # a^T phần W h_j (đích)
        e = F.leaky_relu(dst[:, :, :, None] + src[:, :, None, :], self.leaky_slope)
        if not self.self_loops:
            eye = torch.eye(n, dtype=torch.bool, device=h.device)
            e = e.masked_fill(eye[None, None], float("-inf"))

        return torch.softmax(e, dim=-1), wh

    def forward(self, h: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        alpha, wh = self.attention(h)
        z = torch.matmul(alpha, wh)                      # [B, heads, N, H]
        z = z.permute(0, 2, 1, 3).reshape(h.shape[0], h.shape[1], -1)
        return self.head(F.elu(z)).squeeze(-1), alpha   # [B, N]


class GatInfluenceScorer(InfluenceScorer):
    def __init__(self, config: GatConfig, device: torch.device):
        self.config = config
        self.device = device

    def score(self, train_norm: np.ndarray, seq_len: int, target_idx: Sequence[int], seed: int) -> InfluenceResult:
        cfg = self.config
        torch.manual_seed(seed)
        gen = torch.Generator()
        gen.manual_seed(seed)

        xs, ys = self._next_step_samples(train_norm, seq_len)
        n = xs.shape[0]
        net = GraphAttentionNextStep(seq_len, cfg.hidden_dim, cfg.heads, cfg.leaky_slope, cfg.self_loops)
        net = net.to(self.device)
        opt = torch.optim.Adam(net.parameters(), lr=cfg.learning_rate)
        tgt = torch.as_tensor(list(target_idx), dtype=torch.long, device=self.device)

        history = []
        for epoch in range(cfg.epochs):
            net.train()
            perm = torch.randperm(n, generator=gen)
            total = 0.0
            for s in range(0, n, cfg.batch_size):
                idx = perm[s:s + cfg.batch_size]
                pred, _ = net(xs[idx].to(self.device))
                loss = F.mse_loss(pred[:, tgt], ys[idx].to(self.device)[:, tgt])
                opt.zero_grad()
                loss.backward()
                opt.step()
                total += float(loss) * len(idx)

            history.append({"epoch": epoch + 1, "train_mse_next_step": total / n})

        return InfluenceResult(self._influence(net, xs, tgt), history)

    @staticmethod
    def _next_step_samples(train_norm: np.ndarray, seq_len: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Đầu vào [s, s + L) của từng kênh, nhãn là giá trị tại s + L. Trả về xs [n, C, L], ys [n, C]."""
        series = torch.as_tensor(train_norm, dtype=torch.float32)
        n = series.shape[0] - seq_len
        rows = torch.arange(n)[:, None] + torch.arange(seq_len)[None, :]
        xs = series[rows].permute(0, 2, 1).contiguous()
        ys = series[seq_len:seq_len + n]
        return xs, ys

    def _influence(self, net: GraphAttentionNextStep, xs: torch.Tensor, tgt: torch.Tensor) -> np.ndarray:
        net.eval()
        acc = torch.zeros(xs.shape[1], dtype=torch.float64)
        with torch.no_grad():
            for s in range(0, xs.shape[0], 1024):
                alpha, _ = net.attention(xs[s:s + 1024].to(self.device))
                mean_alpha = alpha.mean(dim=1)                 # trung bình theo head: [B, j, i]
                acc += mean_alpha[:, tgt, :].sum(dim=1).sum(dim=0).double().cpu()

        return (acc / xs.shape[0]).numpy()

In [ ]:
%%writefile badtime_repro/attack/injection.py
"""Bản tham chiếu numpy của phép gắn trigger (Công thức 12-13) và sửa nhãn (Công thức 14).

Bản torch (`attack.trigger.Trigger`) được đối chiếu với bản này sau mỗi lần huấn luyện.
"""

from __future__ import annotations

import numpy as np

from .plan import AttackPlan


class ReferenceInjector:
    def __init__(self, plan: AttackPlan):
        self.plan = plan

    def inject(self, x: np.ndarray, trigger: np.ndarray) -> np.ndarray:
        """x: [n, T, C]. Trả về bản sao có trigger; không sửa x."""
        out = np.array(x, dtype=np.float64, copy=True)
        lam = self.plan.interp_lambda
        for (t_s, t_e), ch, off in self.plan.segments():
            length = t_e - t_s
            seg = out[:, off:off + length, ch]
            out[:, off:off + length, ch] = (1.0 - lam) * seg + lam * np.asarray(trigger)[None, t_s:t_e]

        return out

    def poison_labels(self, y: np.ndarray) -> np.ndarray:
        """y: [n, P, C]. Thay [0, t_pin) của kênh đích bằng p; phần còn lại giữ nguyên."""
        out = np.array(y, dtype=np.float64, copy=True)
        for k, ch in enumerate(self.plan.target_idx):
            out[:, :self.plan.t_pin, ch] = self.plan.pattern[None, :, k]

        return out

    def input_mask(self) -> np.ndarray:
        """Mặt nạ [T, C] các ô đầu vào nhận thay đổi."""
        mask = np.zeros((self.plan.seq_len, len(self.plan.channels)), dtype=bool)
        for (t_s, t_e), ch, off in self.plan.segments():
            mask[off:off + (t_e - t_s), ch] = True

        return mask

In [ ]:
%%writefile badtime_repro/attack/lag.py
"""Vị trí trigger theo độ trễ (Thuật toán 3, Công thức 6-7)."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Sequence, Tuple

import numpy as np

from ..common.errors import PlanError


@dataclass(frozen=True)
class LagEstimate:
    lag: int
    scores: Dict[int, float]        # delta -> tổng tương quan (chỉ các delta tính được)


class LagEstimator:
    """delta* = argmax_{delta in [L, t_IN]} sum_{v_j in V_t} sum_t
                Corr(X^{v_i}[t : t+t_IN-delta], X^{v_j}[t+delta : t+t_IN]).

    Quy tắc biên của dự án (A11): bỏ delta có t_IN - delta < min_corr_len; bỏ cửa sổ có phương sai ~0
    (đóng góp 0 vào tổng). Hòa điểm thì chọn delta nhỏ hơn.
    """

    def __init__(self, seq_len: int, min_corr_len: int):
        self.seq_len = seq_len
        self.min_corr_len = min_corr_len

    def lagged_corr_sum(self, a: np.ndarray, b: np.ndarray, starts: np.ndarray, lag: int) -> Tuple[float, int]:
        """Tổng theo cửa sổ của Corr(a[s : s+T-lag], b[s+lag : s+T]). Trả về (tổng, số cửa sổ hợp lệ)."""
        return self._lagged_corr_sum(_PairPrefix(a, b), starts, lag)

    def _lagged_corr_sum(self, pp: "_PairPrefix", starts: np.ndarray, lag: int) -> Tuple[float, int]:
        n = self.seq_len - lag
        if n < self.min_corr_len:
            return float("nan"), 0

        s = np.asarray(starts, dtype=np.int64)
        pa, paa, pb, pbb = pp.pa, pp.paa, pp.pb, pp.pbb
        pab = _prefix(pp.a[: len(pp.a) - lag] * pp.b[lag:])

        sum_x = pa[s + n] - pa[s]
        sum_xx = paa[s + n] - paa[s]
        sum_y = pb[s + lag + n] - pb[s + lag]
        sum_yy = pbb[s + lag + n] - pbb[s + lag]
        sum_xy = pab[s + n] - pab[s]

        mx = sum_x / n
        my = sum_y / n
        var_x = sum_xx / n - mx * mx
        var_y = sum_yy / n - my * my
        cov = sum_xy / n - mx * my

        valid = (var_x > 1e-10) & (var_y > 1e-10)
        if not np.any(valid):
            return float("nan"), 0

        corr = cov[valid] / np.sqrt(var_x[valid] * var_y[valid])
        return float(corr.sum()), int(valid.sum())

    def estimate(self, series: np.ndarray, channel: int, target_idx: Sequence[int],
                 starts: np.ndarray, min_lag: int) -> LagEstimate:
        lo = min_lag
        hi = self.seq_len - self.min_corr_len
        if hi < lo:
            raise PlanError(
                f"Không có độ trễ hợp lệ trong [{lo}, {self.seq_len}] với min_corr_len={self.min_corr_len}."
            )

        pairs = [_PairPrefix(series[:, channel], series[:, tch]) for tch in target_idx]   # tổng tiền tố tính một lần
        scores: Dict[int, float] = {}
        for lag in range(lo, hi + 1):
            total = 0.0
            ok = True
            for pp in pairs:
                value, n_valid = self._lagged_corr_sum(pp, starts, lag)
                if n_valid == 0 or not np.isfinite(value):
                    ok = False
                    break

                total += value

            if ok:
                scores[lag] = total

        if not scores:
            raise PlanError(f"Kênh {channel}: không có độ trễ nào đủ dữ liệu để tính tương quan.")

        best = max(scores, key=lambda d: (scores[d], -d))
        return LagEstimate(int(best), scores)


class _PairPrefix:
    """Tổng tiền tố của a, a^2, b, b^2 (không phụ thuộc độ trễ), tính một lần cho mỗi cặp kênh."""

    def __init__(self, a: np.ndarray, b: np.ndarray):
        self.a = np.asarray(a, dtype=np.float64)
        self.b = np.asarray(b, dtype=np.float64)
        self.pa, self.paa = _prefix(self.a), _prefix(self.a * self.a)
        self.pb, self.pbb = _prefix(self.b), _prefix(self.b * self.b)


def _prefix(v: np.ndarray) -> np.ndarray:
    """P[k] = sum_{t < k} v[t]."""
    out = np.zeros(len(v) + 1)
    np.cumsum(v, out=out[1:])
    return out

In [ ]:
%%writefile badtime_repro/attack/objectives.py
"""Hàm mục tiêu (torch).

    SquaredPlusAbsolute       ||d||_2^2 + ||d||_1 với phép tổng hợp 'sum' (theo mẫu) hoặc 'mean' (theo phần tử)
    CleanObjective            loss của nhánh sạch: 'mse' (TSLib) hoặc 'mse_mae' (Công thức 16)
    BackdoorModelObjective    Công thức 15-18: a1 L_c + a2 L_p^{V_t} + a3 L_p^{~V_t}
    TriggerObjective          Công thức 20: b1 L_p^{V_t} + b2 L_p^{~V_t} + b3 L_smooth + b4 L_range
Mỗi objective trả về (tổng, {tên_thành_phần: giá_trị}).
"""

from __future__ import annotations

from typing import Dict, List, Sequence, Tuple

import torch
import torch.nn.functional as F

from .trigger import Trigger

LossParts = Dict[str, torch.Tensor]


class SquaredPlusAbsolute:
    def __init__(self, reduction: str):
        if reduction not in ("sum", "mean"):
            raise ValueError(f"reduction không hợp lệ: {reduction}")

        self.reduction = reduction

    def __call__(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        d = pred - target
        if self.reduction == "mean":
            return d.pow(2).mean() + d.abs().mean()

        return (d.pow(2).sum() + d.abs().sum()) / pred.shape[0]


class CleanObjective:
    KINDS = ("mse", "mse_mae")

    def __init__(self, kind: str, reduction: str):
        if kind not in self.KINDS:
            raise ValueError(f"clean_loss không hợp lệ: {kind}")

        self.kind = kind
        self.sq_abs = SquaredPlusAbsolute(reduction)

    def __call__(self, pred: torch.Tensor, y: torch.Tensor) -> Tuple[torch.Tensor, LossParts]:
        loss = F.mse_loss(pred, y) if self.kind == "mse" else self.sq_abs(pred, y)
        return loss, {"clean": loss}


class _PoisonTerms:
    """L_p^{V_t} (Công thức 17) và L_p^{~V_t} (Công thức 18) trên các mẫu bị chọn."""

    def __init__(self, n_channels: int, target_idx: Sequence[int], reduction: str):
        self.target: List[int] = list(target_idx)
        self.other: List[int] = [c for c in range(n_channels) if c not in self.target]
        self.sq_abs = SquaredPlusAbsolute(reduction)

    def __call__(self, pred_p: torch.Tensor, y_poisoned_p: torch.Tensor,
                 y_orig_p: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        l_target = self.sq_abs(pred_p[:, :, self.target], y_poisoned_p[:, :, self.target])
        if self.other:
            l_other = self.sq_abs(pred_p[:, :, self.other], y_orig_p[:, :, self.other])
        else:
            l_other = pred_p.new_zeros(())

        return l_target, l_other


class BackdoorModelObjective:
    """Nhóm rỗng trong lô bị bỏ qua (không chia cho 0, không lấy thêm mẫu)."""

    def __init__(self, n_channels: int, target_idx: Sequence[int], weights: Sequence[float], reduction: str):
        self.a1, self.a2, self.a3 = (float(w) for w in weights)
        self.sq_abs = SquaredPlusAbsolute(reduction)
        self.poison_terms = _PoisonTerms(n_channels, target_idx, reduction)

    def __call__(self, pred: torch.Tensor, y: torch.Tensor, y_poisoned: torch.Tensor,
                 is_poison: torch.Tensor) -> Tuple[torch.Tensor, LossParts]:
        parts: LossParts = {}
        total = pred.new_zeros(())
        clean = ~is_poison
        if bool(clean.any()):
            parts["clean"] = self.sq_abs(pred[clean], y[clean])
            total = total + self.a1 * parts["clean"]

        if bool(is_poison.any()):
            l_t, l_o = self.poison_terms(pred[is_poison], y_poisoned[is_poison], y[is_poison])
            parts["poison_target"] = l_t
            parts["poison_other"] = l_o
            total = total + self.a2 * l_t + self.a3 * l_o

        return total, parts


class TriggerObjective:
    def __init__(self, n_channels: int, target_idx: Sequence[int], weights: Sequence[float],
                 reduction: str, smooth_eps: float):
        self.b1, self.b2, self.b3, self.b4 = (float(w) for w in weights)
        self.smooth_eps = float(smooth_eps)
        self.poison_terms = _PoisonTerms(n_channels, target_idx, reduction)

    def __call__(self, pred_p: torch.Tensor, y_poisoned_p: torch.Tensor, y_orig_p: torch.Tensor,
                 trigger: Trigger) -> Tuple[torch.Tensor, LossParts]:
        l_t, l_o = self.poison_terms(pred_p, y_poisoned_p, y_orig_p)
        l_s = trigger.smooth_loss(self.smooth_eps)
        l_r = trigger.range_loss()
        total = self.b1 * l_t + self.b2 * l_o + self.b3 * l_s + self.b4 * l_r
        return total, {"poison_target": l_t, "poison_other": l_o, "smooth": l_s, "range": l_r}

In [ ]:
%%writefile badtime_repro/attack/patterns.py
"""Mẫu mục tiêu p (Bảng III: Hump, t_PIN = 53). Công thức số là ĐỀ XUẤT của dự án (giả định A06).

Bảng X của v2 dùng ba dạng Hump / Upward / Up-and-down, tương ứng cone / up_trend / up_and_down trong
configs/default_config.yaml của BackTime (xiaolin-cs/BackTime@9e0a1f9).
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Dict, Sequence

import numpy as np

from ..common.errors import PlanError


class TargetPattern(ABC):
    name: str = ""

    @abstractmethod
    def shape(self, length: int) -> np.ndarray:
        """Đường cong chuẩn (trước khi nhân biên độ, cộng nền), dạng [length]."""

    def build(self, length: int, amplitude: float, baseline: float, n_targets: int) -> np.ndarray:
        """p = baseline + amplitude * shape, lặp cho từng kênh đích: [length, n_targets], thang chuẩn hóa."""
        curve = baseline + amplitude * self.shape(length)
        return np.repeat(curve[:, None], n_targets, axis=1)


class InterpolatedPattern(TargetPattern):
    """Nội suy tuyến tính các điểm neo (cách đều) lên `length` điểm."""

    def __init__(self, name: str, anchors: Sequence[float]):
        self.name = name
        self.anchors = np.asarray(anchors, dtype=np.float64)

    def shape(self, length: int) -> np.ndarray:
        grid = np.linspace(0.0, len(self.anchors) - 1.0, length)
        return np.interp(grid, np.arange(len(self.anchors)), self.anchors)


class HalfSinePattern(TargetPattern):
    """sin(pi * (k + 1) / (length + 1)), k in [0, length)."""

    name = "hump_sine"

    def shape(self, length: int) -> np.ndarray:
        k = np.arange(length, dtype=np.float64)
        return np.sin(np.pi * (k + 1.0) / (length + 1.0))


PATTERNS: Dict[str, TargetPattern] = {
    p.name: p
    for p in (
        InterpolatedPattern("hump_backtime_cone", [0.1278, 0.1748, 0.2787, 0.3853, 0.2787, 0.1748, 0.1278]),
        InterpolatedPattern("upward_backtime", [0.1405, 0.2064, 0.2705, 0.2835, 0.2949, 0.3495, 0.3567]),
        InterpolatedPattern("up_and_down_backtime", [0.10, 0.1687, 0.2851, 0.09601, -0.0230, -0.0916, -0.2402]),
        HalfSinePattern(),
    )
}


def get_pattern(name: str) -> TargetPattern:
    if name not in PATTERNS:
        raise PlanError(f"Dạng mục tiêu '{name}' chưa có công thức số.")

    return PATTERNS[name]

In [ ]:
%%writefile badtime_repro/attack/plan.py
"""Kế hoạch tác động của BadTime: kết quả của bốn khối thiết kế (Thuật toán 1-4), cố định suốt quá trình huấn luyện.

`AttackPlanner` điều phối các khối; mỗi khối là một lớp riêng có thể thay thế:
    TargetChannelResolver   -> V_t
    TargetPattern           -> p
    InfluenceScorer + PoisonChannelSelector -> V_p           (Thuật toán 2)
    BudgetProtocol          -> độ dài từng đoạn trigger
    LagEstimator            -> delta*, vị trí đoạn trong đầu vào (Thuật toán 3)
    HybridSampleSelector    -> T_poison                       (Thuật toán 1)
    khởi tạo từ t*          -> tau_0                          (Thuật toán 4)
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, field
from typing import Any, Dict, Iterator, List, Sequence, Tuple

import numpy as np

from ..common.errors import PlanError
from ..common.numeric import ceil_div
from ..config.schema import AttackConfig
from ..data.dataset import PreparedDataset
from .budget import get_budget_protocol
from .channels import InfluenceResult, InfluenceScorer, PoisonChannelSelector, TargetChannelResolver
from .lag import LagEstimator
from .patterns import get_pattern
from .sample_selection import HybridSampleSelector


@dataclass
class AttackPlan:
    channels: List[str]
    target_idx: List[int]
    poison_idx: List[int]
    seg_lens: List[int]
    lags: List[int]
    offsets: List[int]              # đoạn i chiếm [offsets[i], offsets[i] + seg_lens[i]) trong cửa sổ đầu vào
    pattern: np.ndarray             # [t_pin, n_target], thang chuẩn hóa
    poison_starts: np.ndarray       # s tuyệt đối của các cửa sổ huấn luyện bị chọn, tăng dần
    similar_starts: np.ndarray
    different_starts: np.ndarray
    init_start: int                 # s của cửa sổ có nhãn gần mục tiêu nhất (Công thức 8)
    init_trigger: np.ndarray        # [sum(seg_lens)] nối các đoạn theo thứ tự poison_idx (Công thức 10)
    range_min: np.ndarray           # [n_poison] min của kênh trên phần huấn luyện (thang chuẩn hóa)
    range_max: np.ndarray
    interp_lambda: float
    seq_len: int
    pred_len: int
    trace: Dict[str, Any] = field(default_factory=dict)

    # ----------------------------------------------------------------------------- thuộc tính dẫn xuất

    @property
    def t_pin(self) -> int:
        return int(self.pattern.shape[0])

    @property
    def trigger_len(self) -> int:
        return int(sum(self.seg_lens))

    @property
    def seg_bounds(self) -> List[Tuple[int, int]]:
        """[s, e) của từng đoạn trong vector trigger phẳng."""
        out = []
        s = 0
        for length in self.seg_lens:
            out.append((s, s + length))
            s += length

        return out

    def segments(self) -> Iterator[Tuple[Tuple[int, int], int, int]]:
        """Lặp (đoạn [t_s, t_e) trong trigger, kênh, vị trí bắt đầu trong đầu vào)."""
        return zip(self.seg_bounds, self.poison_idx, self.offsets)

    # ----------------------------------------------------------------------------- bất biến

    def validate(self) -> None:
        n_ch = len(self.channels)
        for name, idx in (("target_idx", self.target_idx), ("poison_idx", self.poison_idx)):
            if not idx or any(not (0 <= i < n_ch) for i in idx):
                raise PlanError(f"{name} rỗng hoặc ngoài [0, {n_ch}).")

        if len(set(self.poison_idx)) != len(self.poison_idx) or len(set(self.target_idx)) != len(self.target_idx):
            raise PlanError("Kênh bị sửa hoặc kênh đích bị lặp.")

        if self.pattern.ndim != 2 or self.pattern.shape[1] != len(self.target_idx):
            raise PlanError("Mục tiêu phải có dạng [t_pin, số kênh đích].")

        if len(self.range_min) != len(self.poison_idx) or len(self.range_max) != len(self.poison_idx):
            raise PlanError("range_min/range_max phải có một giá trị cho mỗi kênh bị sửa.")

        if not (0.0 <= self.interp_lambda <= 1.0):
            raise PlanError("interp_lambda phải trong [0, 1].")

        starts = np.asarray(self.poison_starts)
        if len(starts) > 1 and np.any(np.diff(starts) <= 0):
            raise PlanError("poison_starts phải tăng ngặt (không lặp).")

        if len(np.intersect1d(self.similar_starts, self.different_starts)) > 0:
            raise PlanError("Nhóm tương tự và nhóm khác biệt giao nhau.")

        if not (len(self.poison_idx) == len(self.seg_lens) == len(self.lags) == len(self.offsets)):
            raise PlanError("Số kênh, số đoạn, số độ trễ không khớp.")

        for off, length in zip(self.offsets, self.seg_lens):
            if off < 0 or off + length > self.seq_len:
                raise PlanError(f"Đoạn trigger [{off}, {off + length}) nằm ngoài đầu vào [0, {self.seq_len}).")

        if self.t_pin > self.pred_len:
            raise PlanError("Đoạn mục tiêu dài hơn khoảng dự báo.")

        if len(self.init_trigger) != self.trigger_len:
            raise PlanError("Độ dài trigger khởi tạo không khớp ngân sách.")

    # ----------------------------------------------------------------------------- tuần tự hóa

    def to_dict(self) -> Dict[str, Any]:
        d = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in asdict(self).items()}
        d["channel_names"] = {
            "target": [self.channels[i] for i in self.target_idx],
            "poison": [self.channels[i] for i in self.poison_idx],
        }
        d["budget"] = {
            "n_poison_windows": int(len(self.poison_starts)),
            "n_poison_channels": len(self.poison_idx),
            "trigger_cells_per_window": self.trigger_len,
            "total_modified_input_cells": int(self.trigger_len * len(self.poison_starts)),
        }
        return d

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "AttackPlan":
        plan = cls(
            channels=list(d["channels"]),
            target_idx=[int(v) for v in d["target_idx"]],
            poison_idx=[int(v) for v in d["poison_idx"]],
            seg_lens=[int(v) for v in d["seg_lens"]],
            lags=[int(v) for v in d["lags"]],
            offsets=[int(v) for v in d["offsets"]],
            pattern=np.asarray(d["pattern"], dtype=np.float64),
            poison_starts=np.asarray(d["poison_starts"], dtype=np.int64),
            similar_starts=np.asarray(d["similar_starts"], dtype=np.int64),
            different_starts=np.asarray(d["different_starts"], dtype=np.int64),
            init_start=int(d["init_start"]),
            init_trigger=np.asarray(d["init_trigger"], dtype=np.float64),
            range_min=np.asarray(d["range_min"], dtype=np.float64),
            range_max=np.asarray(d["range_max"], dtype=np.float64),
            interp_lambda=float(d["interp_lambda"]),
            seq_len=int(d["seq_len"]),
            pred_len=int(d["pred_len"]),
            trace=dict(d.get("trace", {})),
        )
        plan.validate()
        return plan


class AttackPlanner:
    def __init__(self, config: AttackConfig, scorer: InfluenceScorer):
        self.config = config
        self.scorer = scorer
        self.targets = TargetChannelResolver(config.target_channels, config.target_channel_ratio)
        self.pattern = get_pattern(config.pattern.shape)
        self.channel_selector = PoisonChannelSelector(
            config.poison_channel_ratio, config.exclude_target_from_poison_channels
        )
        self.budget = get_budget_protocol(config.budget_protocol)
        self.sample_selector = HybridSampleSelector(config.poison_ratio, config.similar_ratio)

    def plan(self, dataset: PreparedDataset, seed: int) -> AttackPlan:
        target_idx, _ = self.targets.resolve(dataset.channels)
        influence = self.scorer.score(dataset.train_norm(), dataset.seq_len, target_idx, seed)
        return self.build(dataset, influence)

    def build(self, dataset: PreparedDataset, influence: InfluenceResult) -> AttackPlan:
        """Dựng kế hoạch từ điểm ảnh hưởng đã có (dùng trực tiếp trong kiểm thử và các biến thể bỏ thành phần)."""
        cfg = self.config
        seq_len = dataset.seq_len
        norm = dataset.norm
        b_s, b_e = dataset.bounds.train
        train_starts = dataset.starts["train"]

        # V_t và p
        target_idx, target_note = self.targets.resolve(dataset.channels)
        pattern = self.pattern.build(cfg.pattern.length, cfg.pattern.amplitude, cfg.pattern.baseline, len(target_idx))

        # V_p và ngân sách
        scores = np.asarray(influence.scores, dtype=np.float64)
        poison_idx = self.channel_selector.select(list(scores), target_idx)
        seg_lens = self.budget.split(cfg.trigger_total_len, len(poison_idx))

        # delta* (Thuật toán 3): L = ceil(t_TGR / |V_p|), và đoạn phải nằm trọn trong đầu vào
        min_lag = max(ceil_div(cfg.trigger_total_len, len(poison_idx)), max(seg_lens))
        lag_starts = np.arange(b_s, b_e - seq_len + 1)      # mọi cửa sổ đầu vào nằm trong phần huấn luyện
        estimator = LagEstimator(seq_len, cfg.lag.min_corr_len)
        lags: List[int] = []
        lag_scores: Dict[str, Dict[str, float]] = {}
        for ch in poison_idx:
            est = estimator.estimate(norm, ch, target_idx, lag_starts, min_lag)
            lags.append(est.lag)
            lag_scores[dataset.channels[ch]] = {str(k): float(v) for k, v in est.scores.items()}

        offsets = [seq_len - lag for lag in lags]

        # T_poison (Thuật toán 1)
        distances = self.sample_selector.distances(norm, train_starts, dataset.window, target_idx, pattern)
        selection = self.sample_selector.select(distances)

        # tau_0 (Thuật toán 4)
        best_pos = int(np.lexsort((np.arange(len(distances)), distances))[0])
        init_start = int(train_starts[best_pos])
        init_trigger = self._initial_trigger(norm, init_start + seq_len, poison_idx, seg_lens, lags, (b_s, b_e))

        train_rows = dataset.train_norm()
        plan = AttackPlan(
            channels=list(dataset.channels),
            target_idx=list(target_idx),
            poison_idx=list(poison_idx),
            seg_lens=list(seg_lens),
            lags=lags,
            offsets=offsets,
            pattern=pattern,
            poison_starts=train_starts[selection.all_pos],
            similar_starts=train_starts[selection.similar_pos],
            different_starts=train_starts[selection.different_pos],
            init_start=init_start,
            init_trigger=init_trigger,
            range_min=train_rows[:, poison_idx].min(axis=0),
            range_max=train_rows[:, poison_idx].max(axis=0),
            interp_lambda=float(cfg.interp_lambda),
            seq_len=seq_len,
            pred_len=dataset.pred_len,
            trace={
                "target_note": target_note,
                "influence": [float(v) for v in scores],
                "influence_history": list(influence.history),
                "lag_scores": lag_scores,
                "sample_selection": selection.info,
                "init_distance": float(distances[best_pos]),
                "min_lag": int(min_lag),
                "trigger_total_requested": int(cfg.trigger_total_len),
                "trigger_total_actual": int(sum(seg_lens)),
                "budget_protocol": self.budget.name,
                "pattern_shape": self.pattern.name,
            },
        )
        plan.validate()
        return plan

    @staticmethod
    def _initial_trigger(norm: np.ndarray, label_start: int, poison_idx: Sequence[int], seg_lens: Sequence[int],
                         lags: Sequence[int], train_span: Tuple[int, int]) -> np.ndarray:
        """TGR_i = X^{v_i}[t* - delta_i : t* - delta_i + len_i], với t* = đầu nhãn của cửa sổ gần mục tiêu nhất."""
        b_s, b_e = train_span
        pieces = []
        for ch, seg_len, lag in zip(poison_idx, seg_lens, lags):
            s = label_start - lag
            e = s + seg_len
            if s < b_s or e > b_e:
                raise PlanError("Đoạn khởi tạo trigger nằm ngoài phần huấn luyện.")

            pieces.append(norm[s:e, ch])

        return np.concatenate(pieces)

In [ ]:
%%writefile badtime_repro/attack/sample_selection.py
"""Chọn mẫu bị tác động (Thuật toán 1, Công thức 2-3)."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Sequence

import numpy as np

from ..common.errors import PlanError
from ..common.numeric import floor_count
from ..data.windows import WindowSpec


@dataclass(frozen=True)
class SampleSelection:
    similar_pos: np.ndarray        # vị trí (trong mảng starts) của T_sim, tăng dần
    different_pos: np.ndarray      # vị trí của T_diff, tăng dần
    info: Dict[str, Any]

    @property
    def all_pos(self) -> np.ndarray:
        return np.sort(np.concatenate([self.similar_pos, self.different_pos]))


class HybridSampleSelector:
    """T_sim: k_sim cửa sổ có nhãn gần mục tiêu nhất; T_diff: k_diff cửa sổ xa nhất.

    k_sim = floor(gamma * alpha_T * |D|), k_diff = floor((1 - gamma) * alpha_T * |D|) (Thuật toán 1, dòng 7-8).
    Khoảng cách Euclid ở thang chuẩn hóa (giả định A12).
    """

    def __init__(self, poison_ratio: float, similar_ratio: float):
        self.poison_ratio = poison_ratio
        self.similar_ratio = similar_ratio

    @staticmethod
    def distances(norm: np.ndarray, starts: np.ndarray, window: WindowSpec,
                  target_idx: Sequence[int], pattern: np.ndarray) -> np.ndarray:
        """d(w) = || Y_w[0:t_pin, V_t] - p ||_F cho từng cửa sổ (Công thức 3)."""
        rows = window.label_rows(starts, pattern.shape[0])
        lab = norm[rows][:, :, list(target_idx)]
        diff = lab - pattern[None, :, :]
        return np.sqrt((diff * diff).sum(axis=(1, 2)))

    def select(self, distances: np.ndarray) -> SampleSelection:
        n = len(distances)
        k_sim = floor_count(self.similar_ratio * self.poison_ratio * n)
        k_diff = floor_count((1.0 - self.similar_ratio) * self.poison_ratio * n)
        if k_sim + k_diff > n:
            raise PlanError("Ngân sách mẫu vượt số cửa sổ huấn luyện.")

        order = np.lexsort((np.arange(n), distances))      # tăng dần; hòa thì theo chỉ số
        sim = order[:k_sim]
        diff = order[n - k_diff:][::-1] if k_diff > 0 else order[:0]
        if len(np.intersect1d(sim, diff)) > 0:
            raise PlanError("Nhóm tương tự và nhóm khác biệt giao nhau; ngân sách quá lớn.")

        info = {
            "n_train_windows": int(n),
            "k_total_requested": float(self.poison_ratio * n),
            "k_total": int(k_sim + k_diff),
            "k_similar": int(k_sim),
            "k_different": int(k_diff),
            "distance_similar_max": float(distances[sim].max()) if k_sim else None,
            "distance_different_min": float(distances[diff].min()) if k_diff else None,
        }
        return SampleSelection(np.sort(sim), np.sort(diff), info)

In [ ]:
%%writefile badtime_repro/attack/trigger.py
"""Trigger có thể tối ưu (torch): gắn vào đầu vào (Công thức 12-13), sửa nhãn (Công thức 14),
và hai hàm phạt riêng của trigger (Công thức 21-22)."""

from __future__ import annotations

from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .plan import AttackPlan


class Trigger(nn.Module):
    """Trigger phẳng tau dài sum(seg_lens), cắt thành từng đoạn theo thứ tự plan.poison_idx."""

    def __init__(self, plan: AttackPlan, init: Optional[np.ndarray] = None):
        super().__init__()
        init = plan.init_trigger if init is None else init
        self.tau = nn.Parameter(torch.as_tensor(np.asarray(init), dtype=torch.float32).clone())
        self.seg_bounds = list(plan.seg_bounds)
        self.poison_idx = list(plan.poison_idx)
        self.offsets = list(plan.offsets)
        self.lam = float(plan.interp_lambda)
        self.target_idx = list(plan.target_idx)
        self.t_pin = int(plan.t_pin)
        self.register_buffer("pattern", torch.as_tensor(plan.pattern, dtype=torch.float32))
        self.register_buffer("range_min", torch.as_tensor(plan.range_min, dtype=torch.float32))
        self.register_buffer("range_max", torch.as_tensor(plan.range_max, dtype=torch.float32))

    def inject(self, x: torch.Tensor, detach: bool = False) -> torch.Tensor:
        """x: [B, T, C]. Trả về bản sao có trigger; không sửa x. detach=True khi mô hình đang được cập nhật."""
        tau = self.tau.detach() if detach else self.tau
        out = x.clone()
        for (t_s, t_e), ch, off in zip(self.seg_bounds, self.poison_idx, self.offsets):
            length = t_e - t_s
            seg = x[:, off:off + length, ch]
            out[:, off:off + length, ch] = (1.0 - self.lam) * seg + self.lam * tau[t_s:t_e].unsqueeze(0)

        return out

    def poison_labels(self, y: torch.Tensor) -> torch.Tensor:
        out = y.clone()
        for k, ch in enumerate(self.target_idx):
            out[:, :self.t_pin, ch] = self.pattern[:, k].unsqueeze(0)

        return out

    def smooth_loss(self, eps: float) -> torch.Tensor:
        """Công thức 21: sum [max(|tau_i - tau_{i-1}| - eps, 0)]^2 / t_TGR, chỉ trong từng đoạn."""
        total = self.tau.new_zeros(())
        for t_s, t_e in self.seg_bounds:
            if t_e - t_s < 2:
                continue

            d = self.tau[t_s + 1:t_e] - self.tau[t_s:t_e - 1]
            total = total + F.relu(d.abs() - eps).pow(2).sum()

        return total / self.tau.numel()

    def range_loss(self) -> torch.Tensor:
        """Công thức 22: phạt phần vượt [min, max] của kênh trên tập huấn luyện."""
        total = self.tau.new_zeros(())
        for k, (t_s, t_e) in enumerate(self.seg_bounds):
            seg = self.tau[t_s:t_e]
            total = total + F.relu(self.range_min[k] - seg).sum() + F.relu(seg - self.range_max[k]).sum()

        return total / self.tau.numel()

    def numpy(self) -> np.ndarray:
        return self.tau.detach().cpu().numpy().astype(np.float64)

In [ ]:
%%writefile badtime_repro/common/__init__.py
"""Tiện ích dùng chung, không phụ thuộc phần nghiệp vụ."""

In [ ]:
%%writefile badtime_repro/common/errors.py
"""Phân cấp lỗi của dự án. Mỗi loại lỗi gắn một mã thoát CLI cố định.

    BadTimeError                    1  lỗi chung / vi phạm bất biến nội bộ
    ├── ConfigInvalidError          2  tên không có trong danh mục, khóa lạ, giá trị mâu thuẫn
    │   └── PlanError               2  cấu hình hợp lệ nhưng không lập được kế hoạch tác động trên dữ liệu này
    ├── UnsupportedSelectionError   3  hợp lệ về thiết kế nhưng chưa triển khai (dừng trước huấn luyện)
    ├── DataUnavailableError        4  thiếu dữ liệu hoặc dữ liệu không qua kiểm tra
    └── IntegrityError              1  bất biến thí nghiệm bị vi phạm (vd. trigger đổi trong pha cập nhật mô hình)
"""

from __future__ import annotations

from typing import Iterable, Union


class BadTimeError(Exception):
    exit_code = 1

    def __init__(self, messages: Union[str, Iterable[str]]):
        self.messages = [messages] if isinstance(messages, str) else list(messages)
        super().__init__("\n".join(self.messages))


class ConfigInvalidError(BadTimeError):
    exit_code = 2


class PlanError(ConfigInvalidError):
    exit_code = 2


class UnsupportedSelectionError(BadTimeError):
    exit_code = 3


class DataUnavailableError(BadTimeError):
    exit_code = 4


class IntegrityError(BadTimeError):
    exit_code = 1

In [ ]:
%%writefile badtime_repro/common/jsonio.py
"""Đọc/ghi JSON UTF-8, chấp nhận kiểu numpy."""

from __future__ import annotations

import json
from typing import Any

import numpy as np


def _default(obj: Any) -> Any:
    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if hasattr(obj, "to_dict"):
        return obj.to_dict()

    raise TypeError(f"Không tuần tự hóa được {type(obj)}")


def write_json(path: str, obj: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=_default)


def read_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
%%writefile badtime_repro/common/numeric.py
"""Quy tắc làm tròn dùng chung. Mọi phép đếm từ tỉ lệ đi qua đây để có một quy ước duy nhất."""

from __future__ import annotations

import math

# Bù sai số dấu phẩy động, ví dụ 0.3 * 0.1 * 10000 = 299.99999999999994 phải cho floor = 300.
_EPS = 1e-9


def floor_count(value: float) -> int:
    return int(math.floor(value + _EPS))


def round_half_up(value: float) -> int:
    return int(math.floor(value + 0.5))


def ceil_div(a: int, b: int) -> int:
    return -(-a // b)

In [ ]:
%%writefile badtime_repro/common/registry.py
"""Registry nạp lười: ánh xạ tên -> "module:thuộc_tính", chỉ import khi cần.

Dùng cho các phần cài đặt phụ thuộc torch (mô hình, phương pháp huấn luyện, bộ chấm điểm kênh), để các lệnh
không cần torch (check, kiểm tra numpy) vẫn chạy được.
"""

from __future__ import annotations

import importlib
from typing import Any, Dict, Iterator


class LazyRegistry:
    def __init__(self, kind: str):
        self.kind = kind
        self._targets: Dict[str, str] = {}

    def register(self, name: str, target: str) -> None:
        if name in self._targets:
            raise ValueError(f"{self.kind} '{name}' đã được đăng ký.")

        if ":" not in target:
            raise ValueError(f"Đích phải có dạng 'module:thuộc_tính', nhận '{target}'.")

        self._targets[name] = target

    def __contains__(self, name: str) -> bool:
        return name in self._targets

    def __iter__(self) -> Iterator[str]:
        return iter(self._targets)

    def target(self, name: str) -> str:
        return self._targets[name]

    def resolve(self, name: str) -> Any:
        if name not in self._targets:
            raise KeyError(f"Chưa có cài đặt cho {self.kind} '{name}'.")

        module_name, attr = self._targets[name].split(":", 1)
        return getattr(importlib.import_module(module_name), attr)

In [ ]:
%%writefile badtime_repro/config/__init__.py
"""Cấu hình: schema có kiểu và bộ nạp JSON + ghi đè. Kiểm tra hỗ trợ nằm ở `badtime_repro.support`."""

from .loader import ConfigLoader, load_config
from .schema import (
    AttackConfig,
    DataConfig,
    ExperimentConfig,
    GatConfig,
    LagConfig,
    PatternConfig,
    StateSelectionConfig,
    TimesNetConfig,
    TrainConfig,
)

__all__ = [
    "AttackConfig",
    "ConfigLoader",
    "DataConfig",
    "ExperimentConfig",
    "GatConfig",
    "LagConfig",
    "PatternConfig",
    "StateSelectionConfig",
    "TimesNetConfig",
    "TrainConfig",
    "load_config",
]

In [ ]:
%%writefile badtime_repro/config/loader.py
"""Nạp cấu hình: mặc định của schema <- file JSON <- các ghi đè `--set khoa.con=giatri` (theo thứ tự đó)."""

from __future__ import annotations

import copy
import json
from typing import Any, Dict, Iterable, Optional, Tuple

from ..common.errors import ConfigInvalidError
from .schema import ExperimentConfig


class ConfigLoader:
    def load(self, path: Optional[str] = None, overrides: Iterable[str] = ()) -> ExperimentConfig:
        raw = ExperimentConfig().to_dict()
        if path is not None:
            with open(path, "r", encoding="utf-8") as f:
                raw = self.merge(raw, json.load(f))

        for text in overrides:
            key, value = self.parse_override(text)
            self.apply_override(raw, key, value)

        return ExperimentConfig.from_dict(raw)

    @staticmethod
    def merge(base: Dict[str, Any], override: Dict[str, Any]) -> Dict[str, Any]:
        out = copy.deepcopy(base)
        for key, value in override.items():
            if isinstance(value, dict) and isinstance(out.get(key), dict):
                out[key] = ConfigLoader.merge(out[key], value)
            else:
                out[key] = copy.deepcopy(value)

        return out

    @staticmethod
    def parse_override(text: str) -> Tuple[str, Any]:
        """'a.b=3' -> ('a.b', 3). Giá trị được đọc như JSON nếu được, ngược lại là chuỗi."""
        if "=" not in text:
            raise ConfigInvalidError(f"Ghi đè phải có dạng khoa=giatri: {text}")

        key, raw = text.split("=", 1)
        try:
            value = json.loads(raw)
        except json.JSONDecodeError:
            value = raw

        return key.strip(), value

    @staticmethod
    def apply_override(raw: Dict[str, Any], key: str, value: Any) -> None:
        """Khóa chưa tồn tại vẫn được ghi vào; schema sẽ báo 'không có trong schema' kèm đường dẫn."""
        parts = key.split(".")
        node = raw
        for part in parts[:-1]:
            if not isinstance(node.get(part), dict):
                node[part] = {}

            node = node[part]

        node[parts[-1]] = value


def load_config(path: Optional[str] = None, overrides: Iterable[str] = ()) -> ExperimentConfig:
    return ConfigLoader().load(path, overrides)

In [ ]:
%%writefile badtime_repro/config/schema.py
"""Schema cấu hình có kiểu. Cấu trúc trùng khớp với configs/*.json.

- Mỗi mục cấu hình là một dataclass bất biến (frozen).
- `from_dict` kiểm tra nghiêm: khóa lạ, sai kiểu, sai độ dài bộ trọng số đều bị báo lỗi kèm đường dẫn khóa.
  Mọi lỗi được gom và báo cùng lúc.
- Kiểm tra ngữ nghĩa (tỉ lệ trong [0, 1], tên giao thức hợp lệ...) nằm ở `badtime_repro.support`, không nằm ở đây.
"""

from __future__ import annotations

import dataclasses
import typing
from dataclasses import dataclass, field, fields
from typing import Any, Dict, List, Optional, Tuple, Type, TypeVar

from ..common.errors import ConfigInvalidError

S = TypeVar("S", bound="ConfigSection")


class ConfigSection:
    """Lớp cơ sở của mọi mục cấu hình: dựng từ dict có kiểm tra và xuất ngược ra dict."""

    @classmethod
    def from_dict(cls: Type[S], data: Any) -> S:
        errors: List[str] = []
        obj = _build(cls, data, "", errors)
        if errors:
            raise ConfigInvalidError(["Cấu hình không hợp lệ:"] + errors)

        return obj

    def to_dict(self) -> Dict[str, Any]:
        return {f.name: _export(getattr(self, f.name)) for f in fields(self)}

    def replace(self: S, **changes: Any) -> S:
        return dataclasses.replace(self, **changes)


# ---------------------------------------------------------------------------------------------
# Các mục cấu hình
# ---------------------------------------------------------------------------------------------

@dataclass(frozen=True)
class DataConfig(ConfigSection):
    # "ratio_6_2_2": tỉ lệ 6/2/2 trên toàn chuỗi (nguồn: v2 Bảng XI).
    # "tslib_ett_hour": ranh giới 12/4/4 tháng của Time-Series-Library (cấu hình đối chiếu riêng).
    split_protocol: str = "ratio_6_2_2"
    # Cho cửa sổ đầu vào của val/test dùng lịch sử thuộc phân vùng trước (docs/03 mục 3).
    allow_history_from_previous_split: bool = True


@dataclass(frozen=True)
class TimesNetConfig(ConfigSection):
    label_len: int = 48
    d_model: int = 16
    d_ff: int = 32
    e_layers: int = 2
    top_k: int = 5
    num_kernels: int = 6
    dropout: float = 0.1
    embed: str = "timeF"


@dataclass(frozen=True)
class TrainConfig(ConfigSection):
    epochs: int = 10
    batch_size: int = 32
    learning_rate: float = 1e-4
    lradj: str = "type1"
    # "mse" = loss mặc định TSLib cho Benign; "mse_mae" = (||.||_2^2 + ||.||_1) như Công thức 16,
    # cùng phép tổng hợp attack.loss_reduction (dùng cho kiểm tra đường xử lý).
    clean_loss: str = "mse"
    early_stopping_patience: int = 3
    # TimesNet chọn chu kỳ bằng FFT trung bình trên CẢ LÔ -> dự báo phụ thuộc thành phần lô.
    # Giữ 32 như bộ nạp test của TSLib; không đổi giữa các nhánh.
    eval_batch_size: int = 32
    device: str = "auto"
    deterministic: bool = False


@dataclass(frozen=True)
class PatternConfig(ConfigSection):
    # Bảng III: Hump, t_PIN = 53 (= mu * t_OUT = 0.55 * 96). Công thức số là đề xuất (attack.patterns).
    shape: str = "hump_backtime_cone"
    length: int = 53
    amplitude: float = 1.0
    baseline: float = 0.0
    scale: str = "normalized"


@dataclass(frozen=True)
class LagConfig(ConfigSection):
    min_corr_len: int = 8


@dataclass(frozen=True)
class GatConfig(ConfigSection):
    hidden_dim: int = 64
    heads: int = 4
    epochs: int = 20
    learning_rate: float = 1e-3
    batch_size: int = 256
    self_loops: bool = True
    leaky_slope: float = 0.2


@dataclass(frozen=True)
class StateSelectionConfig(ConfigSection):
    clean_tolerance: float = 0.05


@dataclass(frozen=True)
class AttackConfig(ConfigSection):
    target_channels: Tuple[str, ...] = ("OT",)
    target_channel_ratio: float = 0.1
    pattern: PatternConfig = field(default_factory=PatternConfig)
    poison_ratio: float = 0.1
    similar_ratio: float = 0.3
    poison_channel_ratio: float = 0.3
    exclude_target_from_poison_channels: bool = False
    trigger_total_len: int = 29
    # t_TGR / |V_p| = 14.5 không nguyên. Ba giao thức có nhãn riêng:
    # "keep_total": giữ tổng 29, các đoạn chênh nhau tối đa 1 bước (15 + 14)
    # "equal_segments": các đoạn bằng nhau floor(29/2)=14, tổng thực tế 28
    # "ceil_segments": các đoạn bằng nhau ceil(29/2)=15 (= L của Thuật toán 3), tổng thực tế 30
    budget_protocol: str = "keep_total"
    interp_lambda: float = 1.0
    trigger_update_every: int = 2
    lag: LagConfig = field(default_factory=LagConfig)
    gat: GatConfig = field(default_factory=GatConfig)
    model_loss_weights: Tuple[float, float, float] = (1.0, 1.0, 1.0)
    trigger_loss_weights: Tuple[float, float, float, float] = (1.0, 1.0, 1.0, 1.0)
    # "sum": (||.||_2^2 + ||.||_1) cộng trên mọi phần tử của mẫu rồi chia số mẫu, đúng cách viết Công thức 16-18.
    # "mean": trung bình theo phần tử trong mỗi nhóm (cấu hình đối chiếu).
    loss_reduction: str = "sum"
    smooth_eps: float = 0.8
    trigger_learning_rate: float = 0.01
    trigger_passes_per_update: int = 1
    trigger_batch_size: int = 32
    state_selection: StateSelectionConfig = field(default_factory=StateSelectionConfig)


@dataclass(frozen=True)
class ExperimentConfig(ConfigSection):
    dataset: str = "ETTh1"
    model: str = "TimesNet"
    method: str = "clean"
    seq_len: int = 96
    pred_len: int = 96
    runs_root: str = "runs"
    data: DataConfig = field(default_factory=DataConfig)
    timesnet: TimesNetConfig = field(default_factory=TimesNetConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    attack: AttackConfig = field(default_factory=AttackConfig)

    @property
    def slot(self) -> str:
        return f"{self.dataset}_{self.model}_{self.seq_len}_{self.pred_len}"


# ---------------------------------------------------------------------------------------------
# Dựng / xuất
# ---------------------------------------------------------------------------------------------

def _build(cls: type, data: Any, path: str, errors: List[str]) -> Optional[Any]:
    where = path.rstrip(".") or "cấu hình gốc"
    if not isinstance(data, dict):
        errors.append(f"{where} phải là object, nhận {type(data).__name__}.")
        return None

    hints = typing.get_type_hints(cls)
    known = {f.name for f in fields(cls)}
    for key in sorted(set(data) - known):
        errors.append(f"Khóa '{path}{key}' không có trong schema.")

    kwargs = {}
    for f in fields(cls):
        if f.name in data:
            kwargs[f.name] = _coerce(hints[f.name], data[f.name], f"{path}{f.name}", errors)

    if any(v is _INVALID for v in kwargs.values()):
        return None

    return cls(**kwargs)


_INVALID = object()


def _coerce(tp: Any, value: Any, path: str, errors: List[str]) -> Any:
    if dataclasses.is_dataclass(tp):
        obj = _build(tp, value, path + ".", errors)
        return _INVALID if obj is None else obj

    origin = typing.get_origin(tp)
    if origin is tuple:
        args = typing.get_args(tp)
        if not isinstance(value, (list, tuple)):
            errors.append(f"'{path}' phải là danh sách.")
            return _INVALID

        variadic = len(args) == 2 and args[1] is Ellipsis
        if not variadic and len(value) != len(args):
            errors.append(f"'{path}' cần đúng {len(args)} phần tử, nhận {len(value)}.")
            return _INVALID

        item_types = [args[0]] * len(value) if variadic else list(args)
        items = [_coerce(t, v, f"{path}[{i}]", errors) for i, (t, v) in enumerate(zip(item_types, value))]
        return _INVALID if any(v is _INVALID for v in items) else tuple(items)

    if tp is bool:
        if isinstance(value, bool):
            return value

    elif tp is int:
        if isinstance(value, int) and not isinstance(value, bool):
            return value

    elif tp is float:
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            return float(value)

    elif tp is str:
        if isinstance(value, str):
            return value

    else:
        raise TypeError(f"Schema dùng kiểu chưa hỗ trợ: {tp}")

    errors.append(f"'{path}' phải có kiểu {tp.__name__}, nhận {value!r}.")
    return _INVALID


def _export(value: Any) -> Any:
    if isinstance(value, ConfigSection):
        return value.to_dict()

    if isinstance(value, tuple):
        return [_export(v) for v in value]

    return value

In [ ]:
%%writefile badtime_repro/data/__init__.py
"""Tầng dữ liệu (thuần numpy/pandas, không phụ thuộc torch)."""

In [ ]:
%%writefile badtime_repro/data/dataset.py
"""Dataset đã chuẩn bị: chuỗi gốc, chuỗi chuẩn hóa, đặc trưng thời gian, ranh giới và chỉ số cửa sổ."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

from .scaling import StandardScaler
from .sources import DataQualityReport
from .splits import SplitBounds
from .windows import WindowSpec


@dataclass(frozen=True)
class PreparedDataset:
    report: Optional[DataQualityReport]
    raw: np.ndarray                 # [n_rows, C] đơn vị gốc
    norm: np.ndarray                # [n_rows, C] thang chuẩn hóa
    marks: np.ndarray               # [n_rows, n_time_features]
    scaler: StandardScaler
    bounds: SplitBounds
    starts: Dict[str, np.ndarray]   # split -> các s hợp lệ, tăng dần
    channels: List[str]
    window: WindowSpec

    @property
    def seq_len(self) -> int:
        return self.window.seq_len

    @property
    def pred_len(self) -> int:
        return self.window.pred_len

    @property
    def n_channels(self) -> int:
        return len(self.channels)

    def windows(self, split: str, starts: Optional[np.ndarray] = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """(x, y, x_mark) dạng [n, seq_len, C], [n, pred_len, C], [n, seq_len, F]; thang chuẩn hóa; là bản sao."""
        if starts is None:
            starts = self.starts[split]

        x_rows = self.window.input_rows(starts)
        y_rows = self.window.label_rows(starts)
        return self.norm[x_rows], self.norm[y_rows], self.marks[x_rows]

    def train_norm(self) -> np.ndarray:
        b_s, b_e = self.bounds.train
        return self.norm[b_s:b_e]

    def profile(self) -> Dict[str, Any]:
        return {
            "report": self.report.to_dict() if self.report is not None else None,
            "bounds": self.bounds.to_dict(),
            "window_counts": {k: int(len(v)) for k, v in self.starts.items()},
            "window_first_last_start": {k: [int(v[0]), int(v[-1])] for k, v in self.starts.items()},
            "scaler": self.scaler.to_dict(),
            "channels": list(self.channels),
        }

In [ ]:
%%writefile badtime_repro/data/pipeline.py
"""Định vị thư mục dataset -> đọc & kiểm tra -> phân chia -> chuẩn hóa bằng train -> chỉ số cửa sổ."""

from __future__ import annotations

import os
from typing import Tuple

from ..catalog import CATALOG, Catalog, DatasetSpec
from ..common.errors import DataUnavailableError
from ..config.schema import ExperimentConfig
from .dataset import PreparedDataset
from .scaling import StandardScaler
from .sources import DATA_SOURCES, RawSeries, TimeSeriesSource
from .splits import SPLIT_NAMES, get_split_protocol
from .time_features import TIME_FEATURES, TimeFeatureEncoder
from .windows import WindowSpec


class DataPipeline:
    def __init__(self, project_root: str, catalog: Catalog = CATALOG):
        self.project_root = project_root
        self.catalog = catalog

    def dataset_dir(self, spec: DatasetSpec) -> str:
        return os.path.join(self.project_root, *spec.folder.split("/"))

    def locate(self, spec: DatasetSpec) -> Tuple[TimeSeriesSource, str]:
        """Tạo thư mục chuẩn nếu thiếu (giữ nguyên nội dung sẵn có). Thiếu file -> DataUnavailableError."""
        directory = self.dataset_dir(spec)
        created = not os.path.isdir(directory)
        os.makedirs(directory, exist_ok=True)

        source = DATA_SOURCES[spec.source](spec)
        path = source.file_path(directory)
        if not os.path.isfile(path):
            prefix = "Đã chuẩn bị" if created else "Đã có"
            raise DataUnavailableError(
                f"{prefix} thư mục {spec.folder}/. Cần bổ sung dữ liệu {spec.name} hợp lệ trước khi chạy "
                f"(file kỳ vọng: {spec.folder}/{spec.file_name})."
            )

        return source, path

    def prepare(self, cfg: ExperimentConfig) -> PreparedDataset:
        """`cfg` phải đã qua SupportChecker (tên chuẩn hóa, tổ hợp được hỗ trợ)."""
        spec = self.catalog.datasets[cfg.dataset]
        if spec.freq not in TIME_FEATURES:
            raise DataUnavailableError(
                f"Chưa có bộ mã hóa đặc trưng thời gian cho tần suất '{spec.freq}' của {spec.name}."
            )

        source, path = self.locate(spec)
        try:
            raw = source.load(path)
        except DataUnavailableError as exc:
            raise DataUnavailableError(
                [f"Dữ liệu {spec.name} không hợp lệ tại {spec.folder}/: {m}" for m in exc.messages]
            )

        return self.build(raw, cfg, TIME_FEATURES[spec.freq])

    @staticmethod
    def build(raw: RawSeries, cfg: ExperimentConfig, time_features: TimeFeatureEncoder) -> PreparedDataset:
        """Phân chia -> chuẩn hóa bằng thống kê của train -> đặc trưng thời gian -> chỉ số cửa sổ."""
        values = raw.values
        bounds = get_split_protocol(cfg.data.split_protocol).boundaries(len(values))
        b_s, b_e = bounds.train
        scaler = StandardScaler.fit(values[b_s:b_e])
        window = WindowSpec(cfg.seq_len, cfg.pred_len)

        starts = {
            split: window.starts(bounds, split, cfg.data.allow_history_from_previous_split)
            for split in SPLIT_NAMES
        }
        for split, arr in starts.items():
            if len(arr) == 0:
                raise DataUnavailableError(f"Phân vùng {split} không có cửa sổ hợp lệ.")

        return PreparedDataset(
            report=raw.report,
            raw=values,
            norm=scaler.transform(values),
            marks=time_features.encode(raw.dates),
            scaler=scaler,
            bounds=bounds,
            starts=starts,
            channels=list(raw.report.channels),
            window=window,
        )

In [ ]:
%%writefile badtime_repro/data/scaling.py
"""Chuẩn hóa từng kênh bằng thống kê của phần huấn luyện (ddof=0 như sklearn)."""

from __future__ import annotations

from typing import Dict, List, Optional, Sequence

import numpy as np

from ..common.errors import DataUnavailableError


class StandardScaler:
    def __init__(self, mean: Sequence[float], std: Sequence[float]):
        self.mean = np.asarray(mean, dtype=np.float64)
        self.std = np.asarray(std, dtype=np.float64)

    @classmethod
    def fit(cls, values: np.ndarray) -> "StandardScaler":
        mean = values.mean(axis=0)
        std = values.std(axis=0)
        if np.any(std == 0):
            raise DataUnavailableError("Có kênh hằng trong tập huấn luyện; không chuẩn hóa được.")

        return cls(mean, std)

    @classmethod
    def from_dict(cls, d: Dict[str, List[float]]) -> "StandardScaler":
        return cls(d["mean"], d["std"])

    def transform(self, values: np.ndarray) -> np.ndarray:
        return (values - self.mean) / self.std

    def inverse(self, values: np.ndarray, channels: Optional[Sequence[int]] = None) -> np.ndarray:
        if channels is None:
            return values * self.std + self.mean

        idx = list(channels)
        return values * self.std[idx] + self.mean[idx]

    def to_dict(self) -> Dict[str, List[float]]:
        return {"mean": self.mean.tolist(), "std": self.std.tolist()}

In [ ]:
%%writefile badtime_repro/data/sources.py
"""Nguồn dữ liệu: đọc file và kiểm tra chất lượng. Không sửa dữ liệu."""

from __future__ import annotations

import hashlib
import os
from abc import ABC, abstractmethod
from dataclasses import asdict, dataclass
from typing import Any, Callable, Dict, List, Tuple

import numpy as np
import pandas as pd

from ..catalog import DatasetSpec
from ..common.errors import DataUnavailableError


@dataclass(frozen=True)
class DataQualityReport:
    path: str
    sha256: str
    n_rows: int
    columns: List[str]
    channels: List[str]
    start: str
    end: str
    freq: str
    n_missing: int
    n_duplicate_timestamps: int
    n_irregular_steps: int

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class RawSeries:
    dates: pd.Series
    values: np.ndarray          # [n_rows, n_channels], float64, đơn vị gốc
    report: DataQualityReport


def sha256_of(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)

    return h.hexdigest()


class TimeSeriesSource(ABC):
    def __init__(self, spec: DatasetSpec):
        self.spec = spec

    def file_path(self, dataset_dir: str) -> str:
        return os.path.join(dataset_dir, self.spec.file_name)

    @abstractmethod
    def load(self, path: str) -> RawSeries:
        """Đọc và kiểm tra; ném DataUnavailableError nếu dữ liệu không hợp lệ."""


class EttCsvSource(TimeSeriesSource):
    """CSV dạng ETT: cột 'date' rồi các kênh số, bước thời gian đều."""

    STEPS = {"h": pd.Timedelta(hours=1), "t": pd.Timedelta(minutes=15)}

    def load(self, path: str) -> RawSeries:
        if not os.path.isfile(path):
            raise DataUnavailableError(f"Thiếu file dữ liệu: {path}")

        df = pd.read_csv(path)
        columns = list(df.columns)
        if tuple(columns) != tuple(self.spec.expected_columns):
            raise DataUnavailableError(
                f"Cột không khớp. Kỳ vọng {list(self.spec.expected_columns)}, nhận {columns}"
            )

        dates = pd.to_datetime(df["date"])
        values = df[columns[1:]].to_numpy(dtype=np.float64)
        n_missing, n_dup, n_irregular = self._quality(dates, values)

        report = DataQualityReport(
            path=os.path.abspath(path),
            sha256=sha256_of(path),
            n_rows=int(len(df)),
            columns=columns,
            channels=columns[1:],
            start=str(dates.iloc[0]),
            end=str(dates.iloc[-1]),
            freq=self.spec.freq,
            n_missing=n_missing,
            n_duplicate_timestamps=n_dup,
            n_irregular_steps=n_irregular,
        )
        self._raise_on_problems(report)
        return RawSeries(dates=dates, values=values, report=report)

    def _quality(self, dates: pd.Series, values: np.ndarray) -> Tuple[int, int, int]:
        n_missing = int(np.isnan(values).sum()) + int(dates.isna().sum())
        n_dup = int(dates.duplicated().sum())
        diffs = dates.diff().iloc[1:]
        n_irregular = int((diffs != self.STEPS[self.spec.freq]).sum())
        return n_missing, n_dup, n_irregular

    @staticmethod
    def _raise_on_problems(report: DataQualityReport) -> None:
        problems = []
        if report.n_missing:
            problems.append(f"{report.n_missing} giá trị thiếu")

        if report.n_duplicate_timestamps:
            problems.append(f"{report.n_duplicate_timestamps} mốc thời gian trùng")

        if report.n_irregular_steps:
            problems.append(f"{report.n_irregular_steps} bước thời gian không đều")

        if problems:
            raise DataUnavailableError("Dữ liệu không hợp lệ: " + "; ".join(problems) + ". Không tự sửa dữ liệu.")


DATA_SOURCES: Dict[str, Callable[[DatasetSpec], TimeSeriesSource]] = {"ett_csv": EttCsvSource}

In [ ]:
%%writefile badtime_repro/data/splits.py
"""Giao thức phân chia theo thời gian. Mọi khoảng là half-open [s, e) theo chỉ số hàng của chuỗi gốc."""

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Dict, Iterator, List, Tuple

from ..common.errors import DataUnavailableError

SPLIT_NAMES = ("train", "val", "test")

Span = Tuple[int, int]


@dataclass(frozen=True)
class SplitBounds:
    train: Span
    val: Span
    test: Span

    def __getitem__(self, name: str) -> Span:
        if name not in SPLIT_NAMES:
            raise KeyError(name)

        return getattr(self, name)

    def __iter__(self) -> Iterator[str]:
        return iter(SPLIT_NAMES)

    def items(self) -> Iterator[Tuple[str, Span]]:
        return ((name, self[name]) for name in SPLIT_NAMES)

    def to_dict(self) -> Dict[str, List[int]]:
        return {name: list(span) for name, span in self.items()}

    def validate(self, n_rows: int) -> None:
        s_prev = 0
        for name, (s, e) in self.items():
            if not (s_prev <= s <= e <= n_rows):
                raise DataUnavailableError(f"Ranh giới {name} [{s}, {e}) không hợp lệ với {n_rows} hàng.")

            s_prev = e


class SplitProtocol(ABC):
    name: str = ""

    @abstractmethod
    def boundaries(self, n_rows: int) -> SplitBounds:
        ...


class RatioSplit(SplitProtocol):
    """train = int(0.6 N), test = int(0.2 N), val = phần còn lại (như Dataset_Custom của TSLib)."""

    name = "ratio_6_2_2"

    def __init__(self, train_ratio: float = 0.6, test_ratio: float = 0.2):
        self.train_ratio = train_ratio
        self.test_ratio = test_ratio

    def boundaries(self, n_rows: int) -> SplitBounds:
        n_train = int(n_rows * self.train_ratio)
        n_test = int(n_rows * self.test_ratio)
        n_val = n_rows - n_train - n_test
        bounds = SplitBounds(
            train=(0, n_train),
            val=(n_train, n_train + n_val),
            test=(n_train + n_val, n_rows),
        )
        bounds.validate(n_rows)
        return bounds


class TslibEttHourSplit(SplitProtocol):
    """Ranh giới cố định 12/4/4 tháng của Dataset_ETT_hour trong TSLib (bỏ phần đuôi)."""

    name = "tslib_ett_hour"
    month = 30 * 24

    def boundaries(self, n_rows: int) -> SplitBounds:
        e_train = 12 * self.month
        e_val = e_train + 4 * self.month
        e_test = e_val + 4 * self.month
        if e_test > n_rows:
            raise DataUnavailableError(f"tslib_ett_hour cần ít nhất {e_test} hàng, có {n_rows}.")

        return SplitBounds(train=(0, e_train), val=(e_train, e_val), test=(e_val, e_test))


SPLIT_PROTOCOLS: Dict[str, SplitProtocol] = {p.name: p for p in (RatioSplit(), TslibEttHourSplit())}


def get_split_protocol(name: str) -> SplitProtocol:
    if name not in SPLIT_PROTOCOLS:
        raise DataUnavailableError(f"split_protocol không hợp lệ: {name}")

    return SPLIT_PROTOCOLS[name]

In [ ]:
%%writefile badtime_repro/data/time_features.py
"""Đặc trưng thời gian dạng 'timeF' của TSLib."""

from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Dict

import numpy as np
import pandas as pd


class TimeFeatureEncoder(ABC):
    n_features: int = 0

    @abstractmethod
    def encode(self, dates: pd.Series) -> np.ndarray:
        """Trả về [len(dates), n_features]."""


class HourlyTimeFeatures(TimeFeatureEncoder):
    """freq='h': HourOfDay, DayOfWeek, DayOfMonth, DayOfYear, mỗi đặc trưng trong [-0.5, 0.5]."""

    n_features = 4

    def encode(self, dates: pd.Series) -> np.ndarray:
        idx = pd.DatetimeIndex(dates)
        feats = [
            idx.hour / 23.0 - 0.5,
            idx.dayofweek / 6.0 - 0.5,
            (idx.day - 1) / 30.0 - 0.5,
            (idx.dayofyear - 1) / 365.0 - 0.5,
        ]
        return np.stack([np.asarray(f, dtype=np.float64) for f in feats], axis=1)


TIME_FEATURES: Dict[str, TimeFeatureEncoder] = {"h": HourlyTimeFeatures()}

In [ ]:
%%writefile badtime_repro/data/windows.py
"""Cửa sổ trượt.

Cửa sổ có đầu vào bắt đầu ở s: đầu vào [s, s + seq_len), nhãn [s + seq_len, s + seq_len + pred_len).
Cửa sổ thuộc phân vùng p khi TOÀN BỘ đoạn nhãn nằm trong [b_s, b_e) của p.
Nếu cho phép dùng lịch sử từ phân vùng trước (không áp dụng cho train): s >= max(0, b_s - seq_len); ngược lại s >= b_s.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import numpy as np

from .splits import SplitBounds


@dataclass(frozen=True)
class WindowSpec:
    seq_len: int
    pred_len: int

    def starts(self, bounds: SplitBounds, split: str, allow_history: bool) -> np.ndarray:
        b_s, b_e = bounds[split]
        if split == "train" or not allow_history:
            first = b_s
        else:
            first = max(0, b_s - self.seq_len)

        last = b_e - self.seq_len - self.pred_len     # s + seq_len + pred_len <= b_e
        if last < first:
            return np.zeros(0, dtype=np.int64)

        return np.arange(first, last + 1, dtype=np.int64)

    def input_rows(self, starts: np.ndarray) -> np.ndarray:
        return np.asarray(starts, dtype=np.int64)[:, None] + np.arange(self.seq_len)[None, :]

    def label_rows(self, starts: np.ndarray, length: Optional[int] = None) -> np.ndarray:
        """Hàng của [0, length) bước đầu của nhãn (mặc định toàn bộ pred_len)."""
        n = self.pred_len if length is None else length
        return np.asarray(starts, dtype=np.int64)[:, None] + self.seq_len + np.arange(n)[None, :]

In [ ]:
%%writefile badtime_repro/evaluation/__init__.py
"""Phân tích sau huấn luyện: đối chứng bốn điều kiện, tính lại chỉ số, báo cáo so với bài báo.

Hợp đồng phép đo (numpy) nằm ở `badtime_repro.metrics`, tầng thấp hơn, để training dùng được mà không tạo vòng phụ thuộc.
"""

In [ ]:
%%writefile badtime_repro/evaluation/controls.py
"""Đối chứng bốn điều kiện: {mô hình sạch, mô hình bị tấn công} x {đầu vào sạch, đầu vào có trigger}.

Cùng một trigger đã khóa, cùng cửa sổ, cùng mục tiêu cho cả hai mô hình (docs/03 mục 'Đánh giá theo cặp').
"""

from __future__ import annotations

import logging
from typing import Any, Dict, Tuple

from ..attack.plan import AttackPlan
from ..common.errors import ConfigInvalidError, DataUnavailableError
from ..common.jsonio import write_json
from ..experiment import Experiment
from ..metrics import AttackTarget, MetricCalculator
from ..runs.store import RunFiles, RunReader

log = logging.getLogger(__name__)

CONDITIONS = (
    ("clean_model", "clean_input"),
    ("clean_model", "trigger_input"),
    ("attacked_model", "clean_input"),
    ("attacked_model", "trigger_input"),
)


class ControlExperiment:
    def __init__(self, project_root: str):
        self.experiment = Experiment(project_root)

    def run(self, clean_run: str, attack_run: str) -> Dict[str, Any]:
        import torch

        from ..attack.trigger import Trigger
        from ..models import build_model
        from ..training.context import resolve_device
        from ..training.inference import Predictor
        from ..training.tensors import SplitTensors

        clean, attack = RunReader(clean_run), RunReader(attack_run)
        self._check_compatible(clean, attack)

        cfg, dataset = self.experiment.resolve(attack.config())
        if dataset.report.sha256 != attack.profile()["data"]["report"]["sha256"]:
            raise DataUnavailableError("Dữ liệu hiện tại khác dữ liệu lúc huấn luyện (sha256).")

        device = resolve_device(cfg.train.device)
        pairs = {"clean_model": clean.state_pair(), "attacked_model": attack.state_pair()}
        readers = {"clean_model": clean, "attacked_model": attack}
        models = {}
        for name, pair in pairs.items():
            model = build_model(readers[name].config(), dataset.n_channels).to(device)
            model.load_state_dict(pair.model_state)
            models[name] = model

        att_pair = pairs["attacked_model"]
        plan = AttackPlan.from_dict(att_pair.plan)
        trigger = Trigger(plan, init=att_pair.tau.numpy()).to(device)
        target = AttackTarget.of(plan)
        predictor = Predictor(device, cfg.train.eval_batch_size)
        calculator = MetricCalculator(dataset.scaler)

        out: Dict[str, Any] = {
            "clean_run": clean.path,
            "attack_run": attack.path,
            "attack_epoch": att_pair.epoch,
            "note": "Cùng trigger đã khóa, cùng cửa sổ, cùng mục tiêu cho cả hai mô hình.",
        }
        with torch.no_grad():
            for split_name in ("val", "test"):
                split = SplitTensors.from_dataset(dataset, split_name)
                y = split.y_numpy()
                res = {}
                for model_name, input_name in CONDITIONS:
                    use_trigger = trigger if input_name == "trigger_input" else None
                    pred = predictor.predict(models[model_name], split, use_trigger)
                    res[f"{model_name}|{input_name}"] = calculator.forecast(pred, y, pred, target)

                out[split_name] = res

        out_path = attack.file(RunFiles.controls(clean.name))
        write_json(out_path, out)
        log.info("Đã ghi %s", out_path)
        for cond, met in out["test"].items():
            log.info("test %-32s MAE_all %.4f  MAE_pa %.4f  MAE_pn %.4f",
                     cond, met["mae_c_norm"], met["mae_pa_norm"], met["mae_pn_norm"])

        return out

    @staticmethod
    def _check_compatible(clean: RunReader, attack: RunReader) -> None:
        p_clean, p_att = clean.profile(), attack.profile()
        if p_clean["method"] != "clean":
            raise ConfigInvalidError(f"--clean-run phải là lần chạy clean, nhận '{p_clean['method']}'.")

        if not attack.has(RunFiles.PLAN):
            raise ConfigInvalidError(f"--attack-run phải là lần chạy có kế hoạch tác động, nhận '{p_att['method']}'.")

        for reader in (clean, attack):
            if not reader.is_completed():
                raise ConfigInvalidError(f"Lần chạy {reader.path} chưa hoàn thành.")

        pairs: Tuple[Tuple[str, Any, Any], ...] = (
            ("sha256", p_clean["data"]["report"]["sha256"], p_att["data"]["report"]["sha256"]),
            ("n_rows", p_clean["data"]["report"]["n_rows"], p_att["data"]["report"]["n_rows"]),
            ("bounds", p_clean["data"]["bounds"], p_att["data"]["bounds"]),
            ("timesnet", p_clean["config"]["timesnet"], p_att["config"]["timesnet"]),
            ("seq_len", p_clean["config"]["seq_len"], p_att["config"]["seq_len"]),
            ("pred_len", p_clean["config"]["pred_len"], p_att["config"]["pred_len"]),
            ("eval_batch_size", p_clean["config"]["train"]["eval_batch_size"],
             p_att["config"]["train"]["eval_batch_size"]),
        )
        for key, a, b in pairs:
            if a != b:
                raise ConfigInvalidError(f"Hai lần chạy khác nhau ở '{key}'; không đối chứng được.")

In [ ]:
%%writefile badtime_repro/evaluation/recompute.py
"""Tính lại chỉ số từ dự báo đã lưu và so với metrics.json (docs/06: kiểm tra tính lại chỉ số)."""

from __future__ import annotations

from typing import Any, Dict

from ..data.scaling import StandardScaler
from ..metrics import AttackTarget, MetricCalculator
from ..runs.store import RunReader


class MetricRecomputer:
    def __init__(self, tolerance: float = 1e-9):
        self.tolerance = tolerance

    def run(self, run_dir: str) -> Dict[str, Dict[str, Any]]:
        reader = RunReader(run_dir)
        saved = reader.metrics()
        calculator = MetricCalculator(StandardScaler.from_dict(reader.profile()["data"]["scaler"]))
        plan = reader.plan()
        target = None if plan is None else AttackTarget.of(plan)

        report = {}
        for split in ("val", "test"):
            z = reader.predictions(split)
            pred_trig = z["pred_trig"].astype("float64") if target is not None else None
            met = calculator.forecast(z["pred_clean"].astype("float64"), z["y"].astype("float64"), pred_trig, target)
            missing = sorted(set(met) - set(saved[split]))
            diffs = {k: abs(met[k] - saved[split][k]) for k in met if k in saved[split]}
            worst = max(diffs.values()) if diffs else float("inf")
            report[split] = {
                "max_abs_diff": worst,
                "n_metrics": len(diffs),
                "missing_in_saved": missing,
                "ok": bool(diffs) and not missing and worst <= self.tolerance,
            }

        return report

In [ ]:
%%writefile badtime_repro/evaluation/report.py
"""Tổng hợp các lần chạy của một slot và so với số của bài báo (docs/06 'Quy tắc báo cáo')."""

from __future__ import annotations

import os
from dataclasses import dataclass, field
from typing import Any, Dict, List, Mapping, Optional, Tuple

import numpy as np

from ..common.errors import DataUnavailableError
from ..common.jsonio import read_json, write_json
from ..runs.store import RunIndex

METRICS = ("mae_c", "mae_pa", "mae_pn")


@dataclass(frozen=True)
class PaperReference:
    slot: str
    source: str
    rows: Mapping[str, Mapping[str, float]]


# Số của bài báo v2 (Bảng V, horizon 96). Thang chưa xác nhận (A20).
BADTIME_V2_TABLE_V = PaperReference(
    slot="ETTh1_TimesNet_96_96",
    source="BadTime v2: Bảng V (horizon 96), trùng cột TimesNet/ETTh1 của Bảng IV",
    rows={
        "Benign": {"mae_c": 0.424, "mae_pa": 0.781, "mae_pn": 0.432},
        "BackTime": {"mae_c": 0.448, "mae_pa": 0.750, "mae_pn": 0.439},
        "BadTime": {"mae_c": 0.421, "mae_pa": 0.376, "mae_pn": 0.429},
    },
)


@dataclass(frozen=True)
class MetricSummary:
    runs: List[float]
    paper: Optional[float]
    warn_rel: float

    @property
    def mean(self) -> float:
        return float(np.mean(self.runs))

    @property
    def std(self) -> float:
        return float(np.std(self.runs, ddof=1)) if len(self.runs) > 1 else float("nan")

    @property
    def abs_diff(self) -> Optional[float]:
        return None if self.paper is None else self.mean - self.paper

    @property
    def rel_diff(self) -> Optional[float]:
        if self.paper is None or self.paper == 0:
            return None

        return (self.mean - self.paper) / self.paper

    @property
    def warn(self) -> bool:
        return self.rel_diff is not None and abs(self.rel_diff) > self.warn_rel

    def to_dict(self) -> Dict[str, Any]:
        return {"runs": self.runs, "mean": self.mean, "std": self.std, "paper": self.paper,
                "abs_diff": self.abs_diff, "rel_diff": self.rel_diff, "warn_over_5pct": self.warn}


@dataclass
class ReportBuilder:
    slot_dir: str
    slot: str
    reference: PaperReference = BADTIME_V2_TABLE_V
    warn_rel: float = 0.05
    splits: Tuple[str, ...] = field(default=("val", "test"))

    def build(self) -> Dict[str, Any]:
        index = RunIndex(self.slot_dir)
        bad_runs = index.completed("badtime")
        clean_runs = index.completed("clean")
        if not bad_runs and not clean_runs:
            raise DataUnavailableError(f"Chưa có lần chạy hoàn thành nào trong {self.slot_dir}.")

        refs = self.reference.rows if self.slot == self.reference.slot else {}

        report: Dict[str, Any] = {
            "slot": self.slot, "reference": self.reference.source, "warn_rel": self.warn_rel, "rows": {},
        }
        for split in self.splits:
            rows: Dict[str, Any] = {}

            badtime = {m: [r.metrics()[split][f"{m}_norm"] for r in bad_runs] for m in METRICS}
            if bad_runs:
                rows["BadTime"] = self._row(badtime, refs.get("BadTime", {}), len(bad_runs))

            # Benign MAE_pa / MAE_pn = mô hình sạch nhận trigger BadTime đã khóa (từ ControlExperiment).
            benign: Dict[str, List[float]] = {"mae_c": [r.metrics()[split]["mae_c_norm"] for r in clean_runs],
                                              "mae_pa": [], "mae_pn": []}
            for r in bad_runs:
                for f in r.controls_files():
                    ctl = read_json(f)[split]["clean_model|trigger_input"]
                    benign["mae_pa"].append(ctl["mae_pa_norm"])
                    benign["mae_pn"].append(ctl["mae_pn_norm"])

            if clean_runs:
                rows["Benign"] = self._row({k: v for k, v in benign.items() if v}, refs.get("Benign", {}),
                                           len(clean_runs))

            report["rows"][split] = rows

        return report

    def _row(self, values: Dict[str, List[float]], refs: Mapping[str, float], n_runs: int) -> Dict[str, Any]:
        row: Dict[str, Any] = {k: MetricSummary(v, refs.get(k), self.warn_rel).to_dict() for k, v in values.items()}
        row["n_runs"] = n_runs
        return row

    def write(self, report: Dict[str, Any]) -> str:
        write_json(os.path.join(self.slot_dir, "report.json"), report)
        path = os.path.join(self.slot_dir, "report.md")
        with open(path, "w", encoding="utf-8") as f:
            f.write(MarkdownReportRenderer().render(report))

        return path


class MarkdownReportRenderer:
    def render(self, report: Dict[str, Any]) -> str:
        lines = [
            f"# Báo cáo {report['slot']}",
            "",
            f"Mốc nguồn: {report['reference']}. Thang so sánh: chuẩn hóa (giả định A20, chưa xác nhận).",
            "Benign MAE_pa / MAE_pn = mô hình sạch nhận trigger BadTime đã khóa (từ lệnh controls).",
            f"Cảnh báo kỹ thuật khi |sai lệch tương đối| > {int(report['warn_rel'] * 100)}%; "
            "không phải kiểm định thống kê.",
            "",
        ]
        for split, rows in report["rows"].items():
            lines += [
                f"## Tập {split}",
                "",
                "| Nhánh | Chỉ số | Bài báo | Các lần chạy | TB | Độ lệch chuẩn "
                "| Chênh tuyệt đối | Chênh tương đối | Cảnh báo |",
                "|---|---|---:|---|---:|---:|---:|---:|---|",
            ]
            for method, cols in rows.items():
                for metric, r in cols.items():
                    if metric == "n_runs":
                        continue

                    lines.append(self._line(method, metric, r))

            lines.append("")

        return "\n".join(lines)

    @staticmethod
    def _line(method: str, metric: str, r: Dict[str, Any]) -> str:
        runs = ", ".join(f"{v:.4f}" for v in r["runs"])
        paper = f"{r['paper']:.3f}" if r["paper"] is not None else "—"
        absd = f"{r['abs_diff']:+.4f}" if r["abs_diff"] is not None else "—"
        reld = f"{r['rel_diff'] * 100:+.1f}%" if r["rel_diff"] is not None else "—"
        warn = "⚠" if r["warn_over_5pct"] else ""
        cells = [method, metric, paper, runs, f"{r['mean']:.4f}", f"{r['std']:.4f}", absd, reld, warn]
        return "| " + " | ".join(cells) + " |"

In [ ]:
%%writefile badtime_repro/models/__init__.py
"""Cài đặt mô hình. Tên khớp với `catalog.CATALOG.models`; chỉ import torch khi resolve."""

from typing import TYPE_CHECKING

from ..common.registry import LazyRegistry

if TYPE_CHECKING:
    from ..config.schema import ExperimentConfig
    from .base import Forecaster

MODEL_FACTORIES = LazyRegistry("mô hình")
MODEL_FACTORIES.register("TimesNet", "badtime_repro.models.timesnet:TimesNetFactory")


def build_model(cfg: "ExperimentConfig", n_channels: int) -> "Forecaster":
    return MODEL_FACTORIES.resolve(cfg.model)().build(cfg, n_channels)

In [ ]:
%%writefile badtime_repro/models/base.py
"""Giao diện mô hình dự báo."""

from __future__ import annotations

from abc import ABC, abstractmethod

import torch
import torch.nn as nn

from ..config.schema import ExperimentConfig


class Forecaster(nn.Module):
    """Bọc mô hình: chỉ nhận lịch sử x [B, seq_len, C] và đặc trưng thời gian của lịch sử [B, seq_len, F].

    Đầu vào bộ giải mã (x_dec, x_mark_dec) luôn là None, nên nhãn tương lai không thể lọt vào mô hình.
    Trả về dự báo [B, pred_len, C].
    """

    def __init__(self, inner: nn.Module):
        super().__init__()
        self.inner = inner

    def forward(self, x: torch.Tensor, x_mark: torch.Tensor) -> torch.Tensor:
        return self.inner(x, x_mark, None, None)


class ModelFactory(ABC):
    @abstractmethod
    def build(self, cfg: ExperimentConfig, n_channels: int) -> Forecaster:
        ...

In [ ]:
%%writefile badtime_repro/models/timesnet.py
"""TimesNet từ Time-Series-Library (vendored, commit ghim trong TSLIB_COMMIT)."""

from __future__ import annotations

from types import SimpleNamespace

from ..catalog import CATALOG
from ..config.schema import ExperimentConfig
from .base import Forecaster, ModelFactory
from .tslib.timesnet import Model as TslibTimesNet

TSLIB_COMMIT = "4e938a1767106324dd753b2a44832bf870a0252e"


class TimesNetFactory(ModelFactory):
    def build(self, cfg: ExperimentConfig, n_channels: int) -> Forecaster:
        tn = cfg.timesnet
        args = SimpleNamespace(
            task_name="long_term_forecast",
            seq_len=cfg.seq_len,
            label_len=tn.label_len,
            pred_len=cfg.pred_len,
            e_layers=tn.e_layers,
            enc_in=n_channels,
            c_out=n_channels,
            d_model=tn.d_model,
            d_ff=tn.d_ff,
            top_k=tn.top_k,
            num_kernels=tn.num_kernels,
            dropout=tn.dropout,
            embed=tn.embed,
            freq=CATALOG.datasets[cfg.dataset].freq,     # chọn số đặc trưng thời gian của nhúng timeF
        )
        return Forecaster(TslibTimesNet(args))

In [ ]:
%%writefile badtime_repro/models/tslib/LICENSE_TSLIB
MIT License

Copyright (c) 2021 THUML @ Tsinghua University

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
%%writefile badtime_repro/models/tslib/__init__.py
"""Mã vendored từ thuml/Time-Series-Library (MIT, xem LICENSE_TSLIB). Không sửa ngoài đường import."""

In [ ]:
%%writefile badtime_repro/models/tslib/conv_blocks.py
# Vendored from thuml/Time-Series-Library@4e938a1767106324dd753b2a44832bf870a0252e layers/Conv_Blocks.py (MIT). Unchanged.
import torch
import torch.nn as nn


class Inception_Block_V1(nn.Module):
    def __init__(self, in_channels, out_channels, num_kernels=6, init_weight=True):
        super(Inception_Block_V1, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_kernels = num_kernels
        kernels = []
        for i in range(self.num_kernels):
            kernels.append(nn.Conv2d(in_channels, out_channels, kernel_size=2 * i + 1, padding=i))
        self.kernels = nn.ModuleList(kernels)
        if init_weight:
            self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        res_list = []
        for i in range(self.num_kernels):
            res_list.append(self.kernels[i](x))
        res = torch.stack(res_list, dim=-1).mean(-1)
        return res


class Inception_Block_V2(nn.Module):
    def __init__(self, in_channels, out_channels, num_kernels=6, init_weight=True):
        super(Inception_Block_V2, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_kernels = num_kernels
        kernels = []
        for i in range(self.num_kernels // 2):
            kernels.append(nn.Conv2d(in_channels, out_channels, kernel_size=[1, 2 * i + 3], padding=[0, i + 1]))
            kernels.append(nn.Conv2d(in_channels, out_channels, kernel_size=[2 * i + 3, 1], padding=[i + 1, 0]))
        kernels.append(nn.Conv2d(in_channels, out_channels, kernel_size=1))
        self.kernels = nn.ModuleList(kernels)
        if init_weight:
            self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        res_list = []
        for i in range(self.num_kernels // 2 * 2 + 1):
            res_list.append(self.kernels[i](x))
        res = torch.stack(res_list, dim=-1).mean(-1)
        return res

In [ ]:
%%writefile badtime_repro/models/tslib/embed.py
# Vendored from thuml/Time-Series-Library@4e938a1767106324dd753b2a44832bf870a0252e layers/Embed.py (MIT). Unchanged.
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm
import math


class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEmbedding, self).__init__()
        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model).float()
        pe.require_grad = False

        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = (torch.arange(0, d_model, 2).float()
                    * -(math.log(10000.0) / d_model)).exp()

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.pe[:, :x.size(1)]


class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(TokenEmbedding, self).__init__()
        padding = 1 if torch.__version__ >= '1.5.0' else 2
        self.tokenConv = nn.Conv1d(in_channels=c_in, out_channels=d_model,
                                   kernel_size=3, padding=padding, padding_mode='circular', bias=False)
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(
                    m.weight, mode='fan_in', nonlinearity='leaky_relu')

    def forward(self, x):
        x = self.tokenConv(x.permute(0, 2, 1)).transpose(1, 2)
        return x


class FixedEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(FixedEmbedding, self).__init__()

        w = torch.zeros(c_in, d_model).float()
        w.require_grad = False

        position = torch.arange(0, c_in).float().unsqueeze(1)
        div_term = (torch.arange(0, d_model, 2).float()
                    * -(math.log(10000.0) / d_model)).exp()

        w[:, 0::2] = torch.sin(position * div_term)
        w[:, 1::2] = torch.cos(position * div_term)

        self.emb = nn.Embedding(c_in, d_model)
        self.emb.weight = nn.Parameter(w, requires_grad=False)

    def forward(self, x):
        return self.emb(x).detach()


class TemporalEmbedding(nn.Module):
    def __init__(self, d_model, embed_type='fixed', freq='h'):
        super(TemporalEmbedding, self).__init__()

        minute_size = 4
        hour_size = 24
        weekday_size = 7
        day_size = 32
        month_size = 13

        Embed = FixedEmbedding if embed_type == 'fixed' else nn.Embedding
        if freq == 't':
            self.minute_embed = Embed(minute_size, d_model)
        self.hour_embed = Embed(hour_size, d_model)
        self.weekday_embed = Embed(weekday_size, d_model)
        self.day_embed = Embed(day_size, d_model)
        self.month_embed = Embed(month_size, d_model)

    def forward(self, x):
        x = x.long()
        minute_x = self.minute_embed(x[:, :, 4]) if hasattr(
            self, 'minute_embed') else 0.
        hour_x = self.hour_embed(x[:, :, 3])
        weekday_x = self.weekday_embed(x[:, :, 2])
        day_x = self.day_embed(x[:, :, 1])
        month_x = self.month_embed(x[:, :, 0])

        return hour_x + weekday_x + day_x + month_x + minute_x


class TimeFeatureEmbedding(nn.Module):
    def __init__(self, d_model, embed_type='timeF', freq='h'):
        super(TimeFeatureEmbedding, self).__init__()

        freq_map = {'h': 4, 't': 5, 's': 6,
                    'm': 1, 'a': 1, 'w': 2, 'd': 3, 'b': 3}
        d_inp = freq_map[freq]
        self.embed = nn.Linear(d_inp, d_model, bias=False)

    def forward(self, x):
        return self.embed(x)


class DataEmbedding(nn.Module):
    def __init__(self, c_in, d_model, embed_type='fixed', freq='h', dropout=0.1):
        super(DataEmbedding, self).__init__()

        self.value_embedding = TokenEmbedding(c_in=c_in, d_model=d_model)
        self.position_embedding = PositionalEmbedding(d_model=d_model)
        self.temporal_embedding = TemporalEmbedding(d_model=d_model, embed_type=embed_type,
                                                    freq=freq) if embed_type != 'timeF' else TimeFeatureEmbedding(
            d_model=d_model, embed_type=embed_type, freq=freq)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, x_mark):
        if x_mark is None:
            x = self.value_embedding(x) + self.position_embedding(x)
        else:
            x = self.value_embedding(
                x) + self.temporal_embedding(x_mark) + self.position_embedding(x)
        return self.dropout(x)


class DataEmbedding_inverted(nn.Module):
    def __init__(self, c_in, d_model, embed_type='fixed', freq='h', dropout=0.1):
        super(DataEmbedding_inverted, self).__init__()
        self.value_embedding = nn.Linear(c_in, d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, x_mark):
        x = x.permute(0, 2, 1)
        # x: [Batch Variate Time]
        if x_mark is None:
            x = self.value_embedding(x)
        else:
            x = self.value_embedding(torch.cat([x, x_mark.permute(0, 2, 1)], 1))
        # x: [Batch Variate d_model]
        return self.dropout(x)


class DataEmbedding_wo_pos(nn.Module):
    def __init__(self, c_in, d_model, embed_type='fixed', freq='h', dropout=0.1):
        super(DataEmbedding_wo_pos, self).__init__()

        self.value_embedding = TokenEmbedding(c_in=c_in, d_model=d_model)
        self.position_embedding = PositionalEmbedding(d_model=d_model)
        self.temporal_embedding = TemporalEmbedding(d_model=d_model, embed_type=embed_type,
                                                    freq=freq) if embed_type != 'timeF' else TimeFeatureEmbedding(
            d_model=d_model, embed_type=embed_type, freq=freq)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, x_mark):
        if x_mark is None:
            x = self.value_embedding(x)
        else:
            x = self.value_embedding(x) + self.temporal_embedding(x_mark)
        return self.dropout(x)


class PatchEmbedding(nn.Module):
    def __init__(self, d_model, patch_len, stride, padding, dropout):
        super(PatchEmbedding, self).__init__()
        # Patching
        self.patch_len = patch_len
        self.stride = stride
        self.padding_patch_layer = nn.ReplicationPad1d((0, padding))

        # Backbone, Input encoding: projection of feature vectors onto a d-dim vector space
        self.value_embedding = nn.Linear(patch_len, d_model, bias=False)

        # Positional embedding
        self.position_embedding = PositionalEmbedding(d_model)

        # Residual dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # do patching
        n_vars = x.shape[1]
        x = self.padding_patch_layer(x)
        x = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)
        x = torch.reshape(x, (x.shape[0] * x.shape[1], x.shape[2], x.shape[3]))
        # Input encoding
        x = self.value_embedding(x) + self.position_embedding(x)
        return self.dropout(x), n_vars

In [ ]:
%%writefile badtime_repro/models/tslib/timesnet.py
# Vendored from thuml/Time-Series-Library@4e938a1767106324dd753b2a44832bf870a0252e models/TimesNet.py (MIT).
# Only change: import paths.
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft
from .embed import DataEmbedding
from .conv_blocks import Inception_Block_V1


def FFT_for_Period(x, k=2):
    # [B, T, C]
    xf = torch.fft.rfft(x, dim=1)
    # find period by amplitudes
    frequency_list = abs(xf).mean(0).mean(-1)
    frequency_list[0] = 0
    _, top_list = torch.topk(frequency_list, k)
    top_list = top_list.detach().cpu().numpy()
    period = x.shape[1] // top_list
    return period, abs(xf).mean(-1)[:, top_list]


class TimesBlock(nn.Module):
    def __init__(self, configs):
        super(TimesBlock, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.k = configs.top_k
        # parameter-efficient design
        self.conv = nn.Sequential(
            Inception_Block_V1(configs.d_model, configs.d_ff,
                               num_kernels=configs.num_kernels),
            nn.GELU(),
            Inception_Block_V1(configs.d_ff, configs.d_model,
                               num_kernels=configs.num_kernels)
        )

    def forward(self, x):
        B, T, N = x.size()
        period_list, period_weight = FFT_for_Period(x, self.k)

        res = []
        for i in range(self.k):
            period = period_list[i]
            # padding
            if (self.seq_len + self.pred_len) % period != 0:
                length = (
                                 ((self.seq_len + self.pred_len) // period) + 1) * period
                padding = torch.zeros([x.shape[0], (length - (self.seq_len + self.pred_len)), x.shape[2]]).to(x.device)
                out = torch.cat([x, padding], dim=1)
            else:
                length = (self.seq_len + self.pred_len)
                out = x
            # reshape
            out = out.reshape(B, length // period, period,
                              N).permute(0, 3, 1, 2).contiguous()
            # 2D conv: from 1d Variation to 2d Variation
            out = self.conv(out)
            # reshape back
            out = out.permute(0, 2, 3, 1).reshape(B, -1, N)
            res.append(out[:, :(self.seq_len + self.pred_len), :])
        res = torch.stack(res, dim=-1)
        # adaptive aggregation
        period_weight = F.softmax(period_weight, dim=1)
        period_weight = period_weight.unsqueeze(
            1).unsqueeze(1).repeat(1, T, N, 1)
        res = torch.sum(res * period_weight, -1)
        # residual connection
        res = res + x
        return res


class Model(nn.Module):
    """
    Paper link: https://openreview.net/pdf?id=ju_Uqw384Oq
    """

    def __init__(self, configs):
        super(Model, self).__init__()
        self.configs = configs
        self.task_name = configs.task_name
        self.seq_len = configs.seq_len
        self.label_len = configs.label_len
        self.pred_len = configs.pred_len
        self.model = nn.ModuleList([TimesBlock(configs)
                                    for _ in range(configs.e_layers)])
        self.enc_embedding = DataEmbedding(configs.enc_in, configs.d_model, configs.embed, configs.freq,
                                           configs.dropout)
        self.layer = configs.e_layers
        self.layer_norm = nn.LayerNorm(configs.d_model)
        if self.task_name == 'long_term_forecast' or self.task_name == 'short_term_forecast':
            self.predict_linear = nn.Linear(
                self.seq_len, self.pred_len + self.seq_len)
            self.projection = nn.Linear(
                configs.d_model, configs.c_out, bias=True)
        if self.task_name == 'imputation' or self.task_name == 'anomaly_detection':
            self.projection = nn.Linear(
                configs.d_model, configs.c_out, bias=True)
        if self.task_name == 'classification':
            self.act = F.gelu
            self.dropout = nn.Dropout(configs.dropout)
            self.projection = nn.Linear(
                configs.d_model * configs.seq_len, configs.num_class)

    def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
        # Normalization from Non-stationary Transformer
        means = x_enc.mean(1, keepdim=True).detach()
        x_enc = x_enc.sub(means)
        stdev = torch.sqrt(
            torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
        x_enc = x_enc.div(stdev)

        # embedding
        enc_out = self.enc_embedding(x_enc, x_mark_enc)  # [B,T,C]
        enc_out = self.predict_linear(enc_out.permute(0, 2, 1)).permute(
            0, 2, 1)  # align temporal dimension
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))
        # project back
        dec_out = self.projection(enc_out)

        # De-Normalization from Non-stationary Transformer
        dec_out = dec_out.mul(
                  (stdev[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        dec_out = dec_out.add(
                  (means[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        return dec_out

    def imputation(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask):
        # Normalization from Non-stationary Transformer
        means = torch.sum(x_enc, dim=1) / torch.sum(mask == 1, dim=1)
        means = means.unsqueeze(1).detach()
        x_enc = x_enc.sub(means)
        x_enc = x_enc.masked_fill(mask == 0, 0)
        stdev = torch.sqrt(torch.sum(x_enc * x_enc, dim=1) /
                           torch.sum(mask == 1, dim=1) + 1e-5)
        stdev = stdev.unsqueeze(1).detach()
        x_enc = x_enc.div(stdev)

        # embedding
        enc_out = self.enc_embedding(x_enc, x_mark_enc)  # [B,T,C]
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))
        # project back
        dec_out = self.projection(enc_out)

        # De-Normalization from Non-stationary Transformer
        dec_out = dec_out.mul(
                  (stdev[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        dec_out = dec_out.add(
                  (means[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        return dec_out

    def anomaly_detection(self, x_enc):
        # Normalization from Non-stationary Transformer
        means = x_enc.mean(1, keepdim=True).detach()
        x_enc = x_enc.sub(means)
        stdev = torch.sqrt(
            torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
        x_enc = x_enc.div(stdev)

        # embedding
        enc_out = self.enc_embedding(x_enc, None)  # [B,T,C]
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))
        # project back
        dec_out = self.projection(enc_out)

        # De-Normalization from Non-stationary Transformer
        dec_out = dec_out.mul(
                  (stdev[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        dec_out = dec_out.add(
                  (means[:, 0, :].unsqueeze(1).repeat(
                      1, self.pred_len + self.seq_len, 1)))
        return dec_out

    def classification(self, x_enc, x_mark_enc):
        # embedding
        enc_out = self.enc_embedding(x_enc, None)  # [B,T,C]
        # TimesNet
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))

        # Output
        # the output transformer encoder/decoder embeddings don't include non-linearity
        output = self.act(enc_out)
        output = self.dropout(output)
        # zero-out padding embeddings
        output = output * x_mark_enc.unsqueeze(-1)
        # (batch_size, seq_length * d_model)
        output = output.reshape(output.shape[0], -1)
        output = self.projection(output)  # (batch_size, num_classes)
        return output

    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):
        if self.task_name == 'long_term_forecast' or self.task_name == 'short_term_forecast':
            dec_out = self.forecast(x_enc, x_mark_enc, x_dec, x_mark_dec)
            return dec_out[:, -self.pred_len:, :]  # [B, L, D]
        if self.task_name == 'imputation':
            dec_out = self.imputation(
                x_enc, x_mark_enc, x_dec, x_mark_dec, mask)
            return dec_out  # [B, L, D]
        if self.task_name == 'anomaly_detection':
            dec_out = self.anomaly_detection(x_enc)
            return dec_out  # [B, L, D]
        if self.task_name == 'classification':
            dec_out = self.classification(x_enc, x_mark_enc)
            return dec_out  # [B, N]
        return None

In [ ]:
%%writefile badtime_repro/runs/__init__.py
"""Quản lý thư mục lần chạy: ghi (RunStore), đọc (RunReader), liệt kê (RunIndex)."""

In [ ]:
%%writefile badtime_repro/runs/environment.py
"""Thông tin môi trường ghi vào hồ sơ lần chạy."""

from __future__ import annotations

import platform
import sys
from typing import Any, Dict, Optional

import numpy as np


def software_versions() -> Dict[str, Any]:
    out: Dict[str, Any] = {"python": sys.version.split()[0], "platform": platform.platform(), "numpy": np.__version__}
    try:
        import pandas

        out["pandas"] = pandas.__version__
    except ImportError:
        pass

    try:
        import torch

        out["torch"] = torch.__version__
        out["cuda"] = torch.version.cuda
        out["cudnn"] = torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None
    except ImportError:
        pass

    return out


def peak_memory_mb(device: Any) -> Optional[float]:
    """GPU: bộ nhớ cấp phát tối đa; CPU: max RSS của tiến trình (không có trên Windows)."""
    try:
        import torch

        if getattr(device, "type", None) == "cuda":
            return torch.cuda.max_memory_allocated(device) / 2**20
    except ImportError:
        pass

    try:
        import resource

        rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return rss / 1024.0 if sys.platform != "darwin" else rss / 2**20
    except ImportError:
        return None

In [ ]:
%%writefile badtime_repro/runs/store.py
"""Hiện vật của một lần chạy. Mỗi lần chạy một thư mục riêng; không bao giờ ghi đè lần chạy khác.

runs/<slot>/<method>/seed<seed>_<YYYYmmdd-HHMMSS>[_k]/
    status.json        running | completed | failed (+ traceback)
    profile.json       cấu hình đã chốt, dữ liệu, phiên bản, giả định
    plan.json          (phương pháp có tác động) kế hoạch tác động
    history.json       từng epoch
    state_pair.pt      mô hình + trigger của CÙNG một epoch
    preds_{val,test}.npz
    metrics.json
    controls_vs_<clean_run>.json
"""

from __future__ import annotations

import datetime
import glob
import os
import traceback
from dataclasses import dataclass
from enum import Enum
from typing import Any, Dict, List, Optional

import numpy as np

from ..attack.plan import AttackPlan
from ..common.jsonio import read_json, write_json
from ..config.schema import ExperimentConfig


class RunStatus(str, Enum):
    RUNNING = "running"
    COMPLETED = "completed"
    FAILED = "failed"


class RunFiles:
    STATUS = "status.json"
    PROFILE = "profile.json"
    PLAN = "plan.json"
    HISTORY = "history.json"
    METRICS = "metrics.json"
    STATE_PAIR = "state_pair.pt"

    @staticmethod
    def predictions(split: str) -> str:
        return f"preds_{split}.npz"

    @staticmethod
    def controls(clean_run_name: str) -> str:
        return f"controls_vs_{clean_run_name}.json"


@dataclass
class StatePair:
    """Mô hình và trigger của cùng một epoch. Khóa khi lưu giữ nguyên định dạng của các phiên bản trước."""

    epoch: int
    model_state: Dict[str, Any]
    tau: Optional[Any]                  # torch.Tensor hoặc None
    plan: Optional[Dict[str, Any]]
    method: str
    seed: int

    def to_payload(self) -> Dict[str, Any]:
        return {"epoch": self.epoch, "model": self.model_state, "tau": self.tau, "plan": self.plan,
                "method": self.method, "seed": self.seed}

    @classmethod
    def from_payload(cls, d: Dict[str, Any]) -> "StatePair":
        return cls(d["epoch"], d["model"], d["tau"], d["plan"], d["method"], d["seed"])


class RunReader:
    def __init__(self, path: str):
        self.path = os.path.abspath(path)

    @property
    def name(self) -> str:
        return os.path.basename(os.path.normpath(self.path))

    def file(self, name: str) -> str:
        return os.path.join(self.path, name)

    def has(self, name: str) -> bool:
        return os.path.isfile(self.file(name))

    def status(self) -> Dict[str, Any]:
        return read_json(self.file(RunFiles.STATUS))

    def is_completed(self) -> bool:
        return self.has(RunFiles.STATUS) and self.status().get("state") == RunStatus.COMPLETED.value

    def profile(self) -> Dict[str, Any]:
        return read_json(self.file(RunFiles.PROFILE))

    def config(self) -> ExperimentConfig:
        return ExperimentConfig.from_dict(self.profile()["config"])

    def metrics(self) -> Dict[str, Any]:
        return read_json(self.file(RunFiles.METRICS))

    def history(self) -> List[Dict[str, Any]]:
        return read_json(self.file(RunFiles.HISTORY))

    def plan(self) -> Optional[AttackPlan]:
        return AttackPlan.from_dict(read_json(self.file(RunFiles.PLAN))) if self.has(RunFiles.PLAN) else None

    def predictions(self, split: str) -> Dict[str, np.ndarray]:
        with np.load(self.file(RunFiles.predictions(split))) as z:
            return {k: z[k] for k in z.files}

    def state_pair(self) -> StatePair:
        """Nạp với weights_only=True: file chỉ chứa tensor và kiểu dữ liệu cơ bản, không thực thi mã tùy ý."""
        import torch

        path = self.file(RunFiles.STATE_PAIR)
        try:
            payload = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:                       # torch < 1.13 chưa có tham số weights_only
            payload = torch.load(path, map_location="cpu")

        return StatePair.from_payload(payload)

    def controls_files(self) -> List[str]:
        return sorted(glob.glob(self.file(RunFiles.controls("*"))))


class RunStore(RunReader):
    """RunReader + quyền ghi. Chỉ được tạo qua `create`, nên luôn trỏ tới một thư mục mới."""

    @classmethod
    def create(cls, project_root: str, cfg: ExperimentConfig, method: str, seed: int) -> "RunStore":
        stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        base = os.path.join(project_root, cfg.runs_root, cfg.slot, method, f"seed{seed}_{stamp}")
        path = base
        k = 1
        while os.path.exists(path):
            path = f"{base}_{k}"
            k += 1

        os.makedirs(path)
        return cls(path)

    def write_json(self, name: str, obj: Any) -> None:
        write_json(self.file(name), obj)

    def save_predictions(self, split: str, **arrays: np.ndarray) -> None:
        np.savez_compressed(self.file(RunFiles.predictions(split)), **arrays)

    def save_state_pair(self, pair: StatePair) -> None:
        import torch

        torch.save(pair.to_payload(), self.file(RunFiles.STATE_PAIR))

    # ----------------------------------------------------------------------------- vòng đời

    def mark_running(self, method: str, seed: int) -> None:
        self.write_json(RunFiles.STATUS, {"state": RunStatus.RUNNING.value, "method": method, "seed": seed})

    def mark_completed(self, selected_epoch: int) -> None:
        status = self.status()
        status.update(state=RunStatus.COMPLETED.value, selected_epoch=selected_epoch)
        self.write_json(RunFiles.STATUS, status)

    def mark_failed(self, exc: BaseException) -> None:
        status = self.status()
        status.update(state=RunStatus.FAILED.value, error=repr(exc), traceback=traceback.format_exc())
        self.write_json(RunFiles.STATUS, status)


class RunIndex:
    """Liệt kê các lần chạy của một slot: <project>/<runs_root>/<slot>/<method>/*."""

    def __init__(self, slot_dir: str):
        self.slot_dir = slot_dir

    def completed(self, method: str) -> List[RunReader]:
        runs = [RunReader(d) for d in sorted(glob.glob(os.path.join(self.slot_dir, method, "*")))]
        return [r for r in runs if r.is_completed()]

    def latest(self, method: str, seed: int) -> Optional[RunReader]:
        runs = [r for r in self.completed(method) if r.name.startswith(f"seed{seed}_")]
        return runs[-1] if runs else None

In [ ]:
%%writefile badtime_repro/training/__init__.py
"""Tầng huấn luyện (torch): Trainer chung + các TrainingMethod."""

In [ ]:
%%writefile badtime_repro/training/context.py
"""Mọi thứ một lần chạy cần, gom vào một chỗ để truyền cho Trainer và TrainingMethod."""

from __future__ import annotations

import random
from dataclasses import dataclass, field
from typing import Dict

import numpy as np
import torch

from ..config.schema import ExperimentConfig
from ..data.dataset import PreparedDataset
from ..runs.store import RunStore
from .tensors import SplitTensors


def resolve_device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    return torch.device(name)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class SeedPlan:
    """Mỗi nguồn ngẫu nhiên có seed riêng suy ra từ seed của lần chạy, để các nhánh khớp nhau khi so sánh."""

    MODEL_OFFSET = 0          # khởi tạo mô hình + thứ tự lô
    INFLUENCE_OFFSET = 1000   # huấn luyện GAT
    TRIGGER_OFFSET = 2000     # thứ tự mẫu khi cập nhật trigger

    def __init__(self, seed: int):
        self.seed = seed

    @property
    def model(self) -> int:
        return self.seed + self.MODEL_OFFSET

    @property
    def influence(self) -> int:
        return self.seed + self.INFLUENCE_OFFSET

    @property
    def trigger(self) -> int:
        return self.seed + self.TRIGGER_OFFSET

    @staticmethod
    def generator(seed: int) -> torch.Generator:
        gen = torch.Generator()
        gen.manual_seed(seed)
        return gen


@dataclass
class RunContext:
    config: ExperimentConfig
    dataset: PreparedDataset
    device: torch.device
    seed: int
    store: RunStore
    splits: Dict[str, SplitTensors] = field(default_factory=dict)

    def __post_init__(self) -> None:
        if not self.splits:
            self.splits = {name: SplitTensors.from_dataset(self.dataset, name) for name in ("train", "val", "test")}

    @property
    def seeds(self) -> SeedPlan:
        return SeedPlan(self.seed)

    @property
    def train(self) -> SplitTensors:
        return self.splits["train"]

    @property
    def val(self) -> SplitTensors:
        return self.splits["val"]

    @property
    def test(self) -> SplitTensors:
        return self.splits["test"]

In [ ]:
%%writefile badtime_repro/training/inference.py
"""Dự báo và đánh giá một phân vùng.

TimesNet chọn chu kỳ bằng FFT lấy trung bình trên cả lô, nên dự báo phụ thuộc thành phần lô (A21):
luôn dự báo theo thứ tự thời gian, không xáo, cùng kích thước lô cho mọi nhánh.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import numpy as np
import torch

from ..attack.trigger import Trigger
from ..data.scaling import StandardScaler
from ..metrics import AttackTarget, MetricCalculator, Metrics
from .tensors import SplitTensors


class Predictor:
    def __init__(self, device: torch.device, batch_size: int):
        self.device = device
        self.batch_size = batch_size

    def predict(self, model: torch.nn.Module, split: SplitTensors, trigger: Optional[Trigger] = None) -> np.ndarray:
        """Trả về [n, pred_len, C] float64 (giá trị float32 của mô hình, đổi kiểu không mất mát)."""
        model.eval()
        outs = []
        with torch.no_grad():
            for s in range(0, len(split), self.batch_size):
                e = s + self.batch_size
                xb = split.x[s:e].to(self.device)
                if trigger is not None:
                    xb = trigger.inject(xb, detach=True)

                outs.append(model(xb, split.m[s:e].to(self.device)).cpu())

        return torch.cat(outs).numpy().astype(np.float64)


@dataclass(frozen=True)
class EvaluationResult:
    metrics: Metrics
    pred_clean: np.ndarray
    pred_trig: Optional[np.ndarray]


class Evaluator:
    def __init__(self, scaler: StandardScaler, predictor: Predictor):
        self.calculator = MetricCalculator(scaler)
        self.predictor = predictor

    def evaluate(self, model: torch.nn.Module, split: SplitTensors,
                 trigger: Optional[Trigger] = None, target: Optional[AttackTarget] = None) -> EvaluationResult:
        y = split.y_numpy()
        pred_clean = self.predictor.predict(model, split)
        if trigger is None:
            return EvaluationResult(self.calculator.forecast(pred_clean, y), pred_clean, None)

        pred_trig = self.predictor.predict(model, split, trigger)
        return EvaluationResult(self.calculator.forecast(pred_clean, y, pred_trig, target), pred_clean, pred_trig)

In [ ]:
%%writefile badtime_repro/training/schedule.py
"""Lịch learning rate theo epoch (epoch đếm từ 0)."""

from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Dict, Type


class LearningRateSchedule(ABC):
    def __init__(self, base_lr: float):
        self.base_lr = base_lr

    @abstractmethod
    def lr(self, epoch: int) -> float:
        ...


class HalvingSchedule(LearningRateSchedule):
    """TSLib 'type1': lr = base * 0.5^epoch."""

    def lr(self, epoch: int) -> float:
        return self.base_lr * (0.5 ** epoch)


class ConstantSchedule(LearningRateSchedule):
    def lr(self, epoch: int) -> float:
        return self.base_lr


SCHEDULES: Dict[str, Type[LearningRateSchedule]] = {"type1": HalvingSchedule, "constant": ConstantSchedule}

In [ ]:
%%writefile badtime_repro/training/selection.py
"""Chọn cặp trạng thái (mô hình, trigger) cuối cùng từ các epoch."""

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

from ..metrics import Metrics


@dataclass(frozen=True)
class StateSnapshot:
    epoch: int                          # đếm từ 1
    val: Metrics
    val_loss_mse: float
    model_state: Dict[str, Any]         # bản sao CPU của state_dict
    tau: Optional[Any]                  # bản sao CPU của trigger, hoặc None


Selection = Tuple[StateSnapshot, Dict[str, Any]]


class StateSelector(ABC):
    @abstractmethod
    def observe(self, snapshot: StateSnapshot) -> bool:
        """Ghi nhận trạng thái cuối epoch. Trả về True nếu nên dừng huấn luyện."""

    @abstractmethod
    def select(self) -> Selection:
        ...


class EarlyStoppingSelector(StateSelector):
    """Như TSLib: giữ trạng thái có MSE val nhỏ nhất; dừng khi `patience` epoch liên tiếp không cải thiện."""

    def __init__(self, patience: int):
        self.patience = patience
        self.best: Optional[StateSnapshot] = None
        self.bad_epochs = 0

    def observe(self, snapshot: StateSnapshot) -> bool:
        if self.best is None or snapshot.val_loss_mse < self.best.val_loss_mse:
            self.best = snapshot
            self.bad_epochs = 0
            return False

        self.bad_epochs += 1
        return self.bad_epochs >= self.patience

    def select(self) -> Selection:
        return self.best, {"rule": "min val MSE (TSLib early stopping)", "chosen_epoch": self.best.epoch}


class CleanConstrainedAttackSelector(StateSelector):
    """Ứng viên: MAE_c(val) <= (1 + tol) * min MAE_c(val); trong đó chọn MAE_pa(val) nhỏ nhất,
    hòa thì epoch sớm hơn (A17)."""

    def __init__(self, tolerance: float):
        self.tolerance = tolerance
        self.snapshots: List[StateSnapshot] = []

    def observe(self, snapshot: StateSnapshot) -> bool:
        self.snapshots.append(snapshot)
        return False

    def select(self) -> Selection:
        best_c = min(s.val["mae_c_norm"] for s in self.snapshots)
        limit = (1.0 + self.tolerance) * best_c
        feasible = [s for s in self.snapshots if s.val["mae_c_norm"] <= limit]
        chosen = min(feasible, key=lambda s: (s.val["mae_pa_norm"], s.epoch))
        return chosen, {
            "rule": f"MAE_c(val) <= (1 + {self.tolerance}) * min MAE_c(val); argmin MAE_pa(val)",
            "min_val_mae_c": best_c,
            "limit": limit,
            "feasible_epochs": [s.epoch for s in feasible],
            "chosen_epoch": chosen.epoch,
        }

In [ ]:
%%writefile badtime_repro/training/tensors.py
"""Tensor của một phân vùng (CPU, float32). Lô được chuyển lên thiết bị khi dùng."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import torch

from ..data.dataset import PreparedDataset


def as_float_tensor(a: np.ndarray) -> torch.Tensor:
    return torch.as_tensor(np.ascontiguousarray(a), dtype=torch.float32)


@dataclass(frozen=True)
class SplitTensors:
    name: str
    starts: np.ndarray        # [n] s của từng cửa sổ, tăng dần
    x: torch.Tensor           # [n, seq_len, C]
    y: torch.Tensor           # [n, pred_len, C]
    m: torch.Tensor           # [n, seq_len, F]

    @classmethod
    def from_dataset(cls, dataset: PreparedDataset, split: str) -> "SplitTensors":
        x, y, m = dataset.windows(split)
        return cls(split, dataset.starts[split], as_float_tensor(x), as_float_tensor(y), as_float_tensor(m))

    def __len__(self) -> int:
        return len(self.starts)

    def y_numpy(self) -> np.ndarray:
        return self.y.numpy().astype(np.float64)


@dataclass(frozen=True)
class Batch:
    x: torch.Tensor
    y: torch.Tensor
    m: torch.Tensor
    idx: torch.Tensor         # vị trí của các mẫu trong SplitTensors (CPU, long)

In [ ]:
%%writefile badtime_repro/training/trainer.py
"""Vòng huấn luyện chung cho mọi TrainingMethod.

Thứ tự dùng RNG được cố định để nhánh sạch và BadTime đã vô hiệu hóa (0% mẫu) có cùng quỹ đạo huấn luyện:
    method.prepare()  (RNG riêng của khâu lập kế hoạch)
    seed_everything(seed) -> khởi tạo mô hình
    torch.Generator(seed) -> thứ tự lô
"""

from __future__ import annotations

import logging
import time
from dataclasses import dataclass
from typing import Any, Dict, List

import numpy as np
import torch

from ..assumptions import ASSUMPTIONS
from ..catalog import CATALOG
from ..common.errors import IntegrityError
from ..models import build_model
from ..runs.environment import peak_memory_mb, software_versions
from ..runs.store import RunFiles, StatePair
from .context import RunContext, seed_everything
from .inference import Evaluator, Predictor
from .methods.base import TrainingMethod
from .schedule import SCHEDULES
from .selection import StateSnapshot
from .tensors import Batch

log = logging.getLogger(__name__)


@dataclass(frozen=True)
class TrainingOutcome:
    selected_epoch: int
    metrics: Dict[str, Any]


class CostClock:
    """Cộng dồn thời gian theo hạng mục."""

    def __init__(self) -> None:
        self.costs: Dict[str, float] = {}
        self._t0 = time.time()

    def add(self, key: str, since: float) -> None:
        self.costs[key] = self.costs.get(key, 0.0) + (time.time() - since)

    def total(self) -> float:
        return time.time() - self._t0


class Trainer:
    def __init__(self, ctx: RunContext, method: TrainingMethod):
        self.ctx = ctx
        self.method = method
        self.config = ctx.config
        self.clock = CostClock()
        self.evaluator = Evaluator(ctx.dataset.scaler, Predictor(ctx.device, self.config.train.eval_batch_size))
        self.history: List[Dict[str, Any]] = []

    # ----------------------------------------------------------------------------- điều phối

    def run(self) -> TrainingOutcome:
        ctx, method, tr = self.ctx, self.method, self.config.train
        ctx.store.write_json(RunFiles.PROFILE, self._profile())

        t = time.time()
        method.prepare()
        self.clock.add("planning_sec", t)

        seed_everything(ctx.seeds.model)
        model = build_model(self.config, ctx.dataset.n_channels).to(ctx.device)
        optimizer = torch.optim.Adam(model.parameters(), lr=tr.learning_rate)
        schedule = SCHEDULES[tr.lradj](tr.learning_rate)
        batch_order = ctx.seeds.generator(ctx.seeds.model)
        method.bind(model)
        selector = method.make_selector()

        for epoch in range(tr.epochs):
            snapshot = self._run_epoch(epoch, model, optimizer, schedule.lr(epoch), batch_order)
            if selector.observe(snapshot):
                log.info("Dừng sớm.")
                break

        chosen, selection = selector.select()
        model.load_state_dict(chosen.model_state)
        method.load_trigger_state(chosen.tau)
        ctx.store.save_state_pair(StatePair(
            epoch=chosen.epoch, model_state=chosen.model_state, tau=chosen.tau, plan=method.plan_dict(),
            method=method.name, seed=ctx.seed,
        ))

        results = self._final_evaluation(model, selection)
        log.info("Chọn epoch %d. Test: %s", chosen.epoch, ", ".join(
            f"{k}={v:.4f}" for k, v in results["test"].items() if k.startswith("mae") and k.endswith("norm")
        ))
        return TrainingOutcome(chosen.epoch, results)

    # ----------------------------------------------------------------------------- một epoch

    def _run_epoch(self, epoch: int, model: torch.nn.Module, optimizer: torch.optim.Optimizer,
                   lr: float, batch_order: torch.Generator) -> StateSnapshot:
        ctx, method = self.ctx, self.method
        for group in optimizer.param_groups:
            group["lr"] = lr

        rec: Dict[str, Any] = {"epoch": epoch + 1, "lr": lr}

        t = time.time()
        tau_before = method.trigger_state()
        rec["train"] = self._model_phase(model, optimizer, batch_order)
        tau_after = method.trigger_state()
        if tau_before is not None and not torch.equal(tau_before, tau_after):
            raise IntegrityError("Trigger thay đổi trong giai đoạn cập nhật mô hình.")

        self.clock.add("model_train_sec", t)

        t = time.time()
        extra = method.after_model_epoch(epoch, model)
        if extra is not None:
            rec["trigger_update"] = extra
            self.clock.add("trigger_update_sec", t)

        t = time.time()
        val = self.evaluator.evaluate(model, ctx.val, method.trigger, method.attack_target)
        rec["val"] = val.metrics
        rec["val_loss_mse"] = float(np.mean((val.pred_clean - ctx.val.y_numpy()) ** 2))
        self.clock.add("eval_sec", t)

        log.info("epoch %d: train %.4f | %s", epoch + 1, rec["train"]["total"], method.epoch_summary(val.metrics))
        self.history.append(rec)
        ctx.store.write_json(RunFiles.HISTORY, self.history)

        return StateSnapshot(
            epoch=epoch + 1,
            val=val.metrics,
            val_loss_mse=rec["val_loss_mse"],
            model_state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            tau=method.trigger_state(),
        )

    def _model_phase(self, model: torch.nn.Module, optimizer: torch.optim.Optimizer,
                     batch_order: torch.Generator) -> Dict[str, float]:
        ctx = self.ctx
        train = ctx.train
        bs = self.config.train.batch_size
        model.train()
        perm = torch.randperm(len(train), generator=batch_order)
        sums: Dict[str, float] = {}
        n_seen = 0
        for s in range(0, len(train), bs):
            idx = perm[s:s + bs]
            batch = Batch(train.x[idx].to(ctx.device), train.y[idx].to(ctx.device), train.m[idx].to(ctx.device), idx)
            loss, parts = self.method.batch_loss(model, batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            n_seen += len(idx)
            sums["total"] = sums.get("total", 0.0) + float(loss) * len(idx)
            for key, value in parts.items():
                sums[key] = sums.get(key, 0.0) + float(value) * len(idx)

        return {k: v / n_seen for k, v in sums.items()}

    # ----------------------------------------------------------------------------- kết thúc

    def _final_evaluation(self, model: torch.nn.Module, selection: Dict[str, Any]) -> Dict[str, Any]:
        ctx, method = self.ctx, self.method
        t = time.time()
        results: Dict[str, Any] = {"selection": selection}
        for split in (ctx.val, ctx.test):
            res = self.evaluator.evaluate(model, split, method.trigger, method.attack_target)
            results[split.name] = res.metrics
            arrays = {"pred_clean": res.pred_clean.astype(np.float32), "y": split.y.numpy(), "starts": split.starts}
            if res.pred_trig is not None:
                arrays["pred_trig"] = res.pred_trig.astype(np.float32)

            ctx.store.save_predictions(split.name, **arrays)

        method.finalize(results)
        self.clock.add("final_eval_sec", t)

        costs = dict(self.clock.costs)
        costs["total_sec"] = self.clock.total()
        costs["peak_memory_mb"] = peak_memory_mb(ctx.device)
        results["costs"] = costs
        ctx.store.write_json(RunFiles.METRICS, results)
        return results

    def _profile(self) -> Dict[str, Any]:
        ctx = self.ctx
        return {
            "method": self.method.name,
            "seed": ctx.seed,
            "device": str(ctx.device),
            "config": self.config.to_dict(),
            "data": ctx.dataset.profile(),
            "model_source": CATALOG.models[self.config.model].source,
            "versions": software_versions(),
            "assumptions": [a.to_dict() for a in ASSUMPTIONS],
        }

In [ ]:
%%writefile badtime_repro/training/methods/__init__.py
"""Cài đặt phương pháp huấn luyện. Tên khớp với `catalog.CATALOG.methods`.

Kế thừa:
    TrainingMethod (base)
    ├── CleanMethod                 Benign
    └── PoisoningMethod             khung đầu độc + tối ưu xen kẽ mô hình/trigger
        └── BadTimeMethod
"""

from ...common.registry import LazyRegistry

TRAINING_METHODS = LazyRegistry("phương pháp")
TRAINING_METHODS.register("clean", "badtime_repro.training.methods.clean:CleanMethod")
TRAINING_METHODS.register("badtime", "badtime_repro.training.methods.badtime:BadTimeMethod")

In [ ]:
%%writefile badtime_repro/training/methods/badtime.py
"""BadTime (v2): kế hoạch từ bốn khối thiết kế, GAT chấm điểm kênh, objective Công thức 15-22."""

from __future__ import annotations

from ...attack.gat import GatInfluenceScorer
from ...attack.objectives import BackdoorModelObjective, TriggerObjective
from ...attack.plan import AttackPlan, AttackPlanner
from .poisoning import PoisoningMethod


class BadTimeMethod(PoisoningMethod):
    name = "badtime"

    def build_plan(self) -> AttackPlan:
        att = self.config.attack
        scorer = GatInfluenceScorer(att.gat, self.ctx.device)
        return AttackPlanner(att, scorer).plan(self.ctx.dataset, self.ctx.seeds.influence)

    def make_model_objective(self) -> BackdoorModelObjective:
        att = self.config.attack
        return BackdoorModelObjective(
            self.ctx.dataset.n_channels, self.plan.target_idx, att.model_loss_weights, att.loss_reduction
        )

    def make_trigger_objective(self) -> TriggerObjective:
        att = self.config.attack
        return TriggerObjective(
            self.ctx.dataset.n_channels, self.plan.target_idx, att.trigger_loss_weights, att.loss_reduction,
            att.smooth_eps,
        )

In [ ]:
%%writefile badtime_repro/training/methods/base.py
"""Giao diện phương pháp huấn luyện. Trainer chỉ làm việc với giao diện này.

Vòng đời do Trainer gọi, theo thứ tự:
    prepare()                    trước khi khởi tạo mô hình (lập kế hoạch; dùng RNG riêng)
    bind(model)                  sau khi có mô hình (tạo optimizer phụ)
    mỗi epoch:
        batch_loss(model, batch) cho từng lô của pha cập nhật mô hình
        after_model_epoch(...)   việc làm thêm sau pha mô hình (vd. cập nhật trigger)
    make_selector()              quy tắc chọn cặp trạng thái
    finalize(results)            hiện vật bổ sung sau đánh giá cuối
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Any, Dict, Optional, Tuple

import torch

from ...attack.objectives import LossParts
from ...attack.trigger import Trigger
from ...metrics import AttackTarget, Metrics
from ..context import RunContext
from ..selection import StateSelector
from ..tensors import Batch


class TrainingMethod(ABC):
    name: str = ""

    def __init__(self, ctx: RunContext):
        self.ctx = ctx
        self.config = ctx.config

    # ----------------------------------------------------------------------------- vòng đời

    def prepare(self) -> None:
        """Mặc định không cần chuẩn bị."""

    def bind(self, model: torch.nn.Module) -> None:
        """Mặc định không có optimizer phụ."""

    @abstractmethod
    def batch_loss(self, model: torch.nn.Module, batch: Batch) -> Tuple[torch.Tensor, LossParts]:
        ...

    def after_model_epoch(self, epoch: int, model: torch.nn.Module) -> Optional[Dict[str, float]]:
        """epoch đếm từ 0. Trả về bản ghi để đưa vào history, hoặc None."""
        return None

    @abstractmethod
    def make_selector(self) -> StateSelector:
        ...

    def finalize(self, results: Dict[str, Any]) -> None:
        """Bổ sung hiện vật vào metrics.json sau đánh giá cuối."""

    # ----------------------------------------------------------------------------- trigger (nếu có)

    @property
    def trigger(self) -> Optional[Trigger]:
        return None

    @property
    def attack_target(self) -> Optional[AttackTarget]:
        return None

    def plan_dict(self) -> Optional[Dict[str, Any]]:
        return None

    def trigger_state(self) -> Optional[torch.Tensor]:
        trigger = self.trigger
        return None if trigger is None else trigger.tau.detach().cpu().clone()

    def load_trigger_state(self, tau: Optional[torch.Tensor]) -> None:
        trigger = self.trigger
        if trigger is not None and tau is not None:
            with torch.no_grad():
                trigger.tau.copy_(tau.to(trigger.tau.device))

    # ----------------------------------------------------------------------------- hiển thị

    def epoch_summary(self, val: Metrics) -> str:
        return f"val MAE_c {val['mae_c_norm']:.4f}"

In [ ]:
%%writefile badtime_repro/training/methods/clean.py
"""Huấn luyện sạch (Benign)."""

from __future__ import annotations

from typing import Tuple

import torch

from ...attack.objectives import CleanObjective, LossParts
from ..context import RunContext
from ..selection import EarlyStoppingSelector, StateSelector
from ..tensors import Batch
from .base import TrainingMethod


class CleanMethod(TrainingMethod):
    name = "clean"

    def __init__(self, ctx: RunContext):
        super().__init__(ctx)
        self.objective = CleanObjective(self.config.train.clean_loss, self.config.attack.loss_reduction)

    def batch_loss(self, model: torch.nn.Module, batch: Batch) -> Tuple[torch.Tensor, LossParts]:
        return self.objective(model(batch.x, batch.m), batch.y)

    def make_selector(self) -> StateSelector:
        return EarlyStoppingSelector(self.config.train.early_stopping_patience)

In [ ]:
%%writefile badtime_repro/training/methods/poisoning.py
"""Khung chung cho các phương pháp đầu độc dữ liệu có trigger tối ưu được và tối ưu xen kẽ mô hình/trigger.

Lớp con chỉ cần cung cấp:
    build_plan()               -> AttackPlan (chọn mẫu, kênh, vị trí, trigger khởi tạo)
    make_model_objective()     -> objective của pha mô hình
    make_trigger_objective()   -> objective của pha trigger
BadTime là một lớp con; một bản chuyển đổi BackTime về sau cũng có thể là lớp con.
"""

from __future__ import annotations

import hashlib
import logging
from abc import abstractmethod
from typing import Any, Dict, Optional, Tuple

import numpy as np
import torch

from ...attack.injection import ReferenceInjector
from ...attack.objectives import BackdoorModelObjective, LossParts, TriggerObjective
from ...attack.plan import AttackPlan
from ...attack.trigger import Trigger
from ...common.errors import IntegrityError
from ...metrics import AttackTarget, MetricCalculator, Metrics
from ...runs.store import RunFiles
from ..context import RunContext
from ..selection import CleanConstrainedAttackSelector, StateSelector
from ..tensors import Batch
from .base import TrainingMethod

log = logging.getLogger(__name__)


def params_digest(module: torch.nn.Module) -> str:
    h = hashlib.sha256()
    for p in module.parameters():
        h.update(p.detach().cpu().numpy().tobytes())

    return h.hexdigest()


class PoisoningMethod(TrainingMethod):
    def __init__(self, ctx: RunContext):
        super().__init__(ctx)
        self.plan: Optional[AttackPlan] = None
        self._trigger: Optional[Trigger] = None
        self.flags = torch.zeros(len(ctx.train), dtype=torch.bool)    # flags[i]: mẫu i của train bị chọn
        self.poison_pos = torch.zeros(0, dtype=torch.long)
        self.model_objective: Optional[BackdoorModelObjective] = None
        self.trigger_objective: Optional[TriggerObjective] = None
        self.trigger_optimizer: Optional[torch.optim.Optimizer] = None
        self.trigger_generator: Optional[torch.Generator] = None

    # ----------------------------------------------------------------------------- điểm mở rộng

    @abstractmethod
    def build_plan(self) -> AttackPlan:
        ...

    @abstractmethod
    def make_model_objective(self) -> BackdoorModelObjective:
        ...

    @abstractmethod
    def make_trigger_objective(self) -> TriggerObjective:
        ...

    # ----------------------------------------------------------------------------- vòng đời

    def prepare(self) -> None:
        ctx = self.ctx
        self.plan = self.build_plan()
        ctx.store.write_json(RunFiles.PLAN, self.plan.to_dict())
        log.info(
            "Kênh bị sửa: %s, đoạn %s, độ trễ %s, số cửa sổ bị chọn %d",
            [ctx.dataset.channels[i] for i in self.plan.poison_idx], self.plan.seg_lens, self.plan.lags,
            len(self.plan.poison_starts),
        )

        pos = np.searchsorted(ctx.train.starts, self.plan.poison_starts)
        if not np.array_equal(ctx.train.starts[pos], self.plan.poison_starts):
            raise IntegrityError("Cửa sổ bị chọn không thuộc tập huấn luyện.")

        self.poison_pos = torch.as_tensor(pos, dtype=torch.long)
        self.flags[self.poison_pos] = True
        self._trigger = Trigger(self.plan).to(ctx.device)
        self.model_objective = self.make_model_objective()
        self.trigger_objective = self.make_trigger_objective()

    def bind(self, model: torch.nn.Module) -> None:
        lr = self.config.attack.trigger_learning_rate
        self.trigger_optimizer = torch.optim.Adam(self._trigger.parameters(), lr=lr)
        self.trigger_generator = self.ctx.seeds.generator(self.ctx.seeds.trigger)

    def batch_loss(self, model: torch.nn.Module, batch: Batch) -> Tuple[torch.Tensor, LossParts]:
        """Pha mô hình: trigger cố định (detach), mẫu bị chọn được gắn trigger và sửa nhãn (Công thức 11-14)."""
        is_poison = self.flags[batch.idx].to(self.ctx.device)
        x, y_poisoned = batch.x, batch.y
        if bool(is_poison.any()):
            x = x.clone()
            x[is_poison] = self._trigger.inject(x[is_poison], detach=True)
            y_poisoned = batch.y.clone()
            y_poisoned[is_poison] = self._trigger.poison_labels(batch.y[is_poison])

        return self.model_objective(model(x, batch.m), batch.y, y_poisoned, is_poison)

    def after_model_epoch(self, epoch: int, model: torch.nn.Module) -> Optional[Dict[str, float]]:
        if (epoch + 1) % self.config.attack.trigger_update_every != 0 or len(self.poison_pos) == 0:
            return None

        return self._update_trigger(model)

    def _update_trigger(self, model: torch.nn.Module) -> Dict[str, float]:
        """Pha trigger: cố định mô hình (eval, không cập nhật tham số), duyệt các mẫu bị chọn."""
        att = self.config.attack
        ctx = self.ctx
        model.eval()
        saved = [p.requires_grad for p in model.parameters()]
        for p in model.parameters():
            p.requires_grad_(False)

        before = params_digest(model)
        sums: Dict[str, float] = {}
        n_seen = 0
        for _ in range(att.trigger_passes_per_update):
            perm = self.poison_pos[torch.randperm(len(self.poison_pos), generator=self.trigger_generator)]
            for s in range(0, len(perm), att.trigger_batch_size):
                idx = perm[s:s + att.trigger_batch_size]
                x = ctx.train.x[idx].to(ctx.device)
                y = ctx.train.y[idx].to(ctx.device)
                m = ctx.train.m[idx].to(ctx.device)
                pred = model(self._trigger.inject(x), m)
                loss, parts = self.trigger_objective(pred, self._trigger.poison_labels(y), y, self._trigger)
                self.trigger_optimizer.zero_grad()
                loss.backward()
                self.trigger_optimizer.step()

                n_seen += len(idx)
                _accumulate(sums, "total", loss, len(idx))
                for key, value in parts.items():
                    _accumulate(sums, key, value, len(idx))

        for p, flag in zip(model.parameters(), saved):
            p.requires_grad_(flag)

        if params_digest(model) != before:
            raise IntegrityError("Tham số mô hình thay đổi trong giai đoạn cập nhật trigger.")

        return {k: v / max(n_seen, 1) for k, v in sums.items()}

    def make_selector(self) -> StateSelector:
        return CleanConstrainedAttackSelector(self.config.attack.state_selection.clean_tolerance)

    def finalize(self, results: Dict[str, Any]) -> None:
        """Đối chiếu phép gắn torch với bản numpy tham chiếu, rồi đo mức sửa đầu vào trên test."""
        test = self.ctx.test
        tau = self._trigger.numpy()
        x_test = test.x.numpy().astype(np.float64)
        x_trig_ref = ReferenceInjector(self.plan).inject(x_test, tau)
        x_trig_torch = self._trigger.inject(test.x.to(self.ctx.device), detach=True).cpu().numpy()
        if not np.allclose(x_trig_ref, x_trig_torch, atol=1e-5):
            raise IntegrityError("Gắn trigger bằng torch và numpy không khớp.")

        results["input_change_test"] = MetricCalculator.input_change(x_test, x_trig_ref, self.plan)
        results["trigger_final"] = tau.tolist()

    # ----------------------------------------------------------------------------- trigger

    @property
    def trigger(self) -> Optional[Trigger]:
        return self._trigger

    @property
    def attack_target(self) -> Optional[AttackTarget]:
        return None if self.plan is None else AttackTarget.of(self.plan)

    def plan_dict(self) -> Optional[Dict[str, Any]]:
        return None if self.plan is None else self.plan.to_dict()

    def epoch_summary(self, val: Metrics) -> str:
        return (f"{super().epoch_summary(val)} MAE_pa {val['mae_pa_norm']:.4f} "
                f"MAE_pn {val.get('mae_pn_norm', float('nan')):.4f}")


def _accumulate(sums: Dict[str, float], key: str, value: torch.Tensor, n: int) -> None:
    sums[key] = sums.get(key, 0.0) + float(value) * n

In [ ]:
%%writefile tests/conftest.py
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))   # gốc dự án -> badtime_repro
sys.path.insert(0, HERE)                    # helpers

In [ ]:
%%writefile tests/helpers.py
"""Dữ liệu và đối tượng dựng sẵn cho test."""

from __future__ import annotations

import os

import numpy as np

from badtime_repro.attack.plan import AttackPlan
from badtime_repro.config import load_config
from badtime_repro.data.dataset import PreparedDataset
from badtime_repro.data.scaling import StandardScaler
from badtime_repro.data.splits import RatioSplit
from badtime_repro.data.windows import WindowSpec

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
ETTH1_CSV = os.path.join(ROOT, "dataset", "ETTh1", "ETTh1.csv")
HAS_ETTH1 = os.path.isfile(ETTH1_CSV)


def synthetic_dataset(n: int = 40, seq: int = 4, pred: int = 3, channels: int = 3) -> PreparedDataset:
    raw = np.arange(n * channels, dtype=np.float64).reshape(n, channels)
    raw[:, 1] = np.sin(np.arange(n))
    bounds = RatioSplit().boundaries(n)
    scaler = StandardScaler.fit(raw[bounds.train[0]:bounds.train[1]])
    window = WindowSpec(seq, pred)
    starts = {k: window.starts(bounds, k, True) for k in bounds}
    return PreparedDataset(None, raw, scaler.transform(raw), np.zeros((n, 4)), scaler, bounds, starts,
                           [f"c{i}" for i in range(channels)], window)


def toy_plan(seq: int = 10, pred: int = 6, channels: int = 3, lam: float = 1.0) -> AttackPlan:
    return AttackPlan(
        channels=[f"c{i}" for i in range(channels)], target_idx=[channels - 1], poison_idx=[0, 1], seg_lens=[3, 2],
        lags=[4, 7], offsets=[seq - 4, seq - 7], pattern=np.full((4, 1), 5.0),
        poison_starts=np.array([0]), similar_starts=np.array([0]), different_starts=np.array([], dtype=np.int64),
        init_start=0, init_trigger=np.array([1.0, 2.0, 3.0, 4.0, 5.0]),
        range_min=np.array([-1.0, -1.0]), range_max=np.array([1.0, 1.0]), interp_lambda=lam,
        seq_len=seq, pred_len=pred,
    )


def etth1_toy_plan(lam: float = 1.0) -> AttackPlan:
    """Kế hoạch có kích thước thật của slot PoC (96 -> 96, 7 kênh, 29 = 15 + 14)."""
    return AttackPlan(
        channels=["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"], target_idx=[6], poison_idx=[3, 1],
        seg_lens=[15, 14], lags=[59, 58], offsets=[37, 38], pattern=np.ones((53, 1)),
        poison_starts=np.array([0]), similar_starts=np.array([0]), different_starts=np.array([], dtype=np.int64),
        init_start=0, init_trigger=np.linspace(-1, 1, 29),
        range_min=np.array([-3.0, -3.0]), range_max=np.array([3.0, 3.0]), interp_lambda=lam,
        seq_len=96, pred_len=96,
    )


def default_config(*overrides: str):
    return load_config(overrides=list(overrides))

In [ ]:
%%writefile tests/test_architecture.py
"""Ràng buộc kiến trúc, kiểm bằng phân tích tĩnh các câu import (không cần chạy mã).

1. Phân tầng: một module chỉ được import module cùng tầng hoặc tầng thấp hơn, kể cả import bên trong hàm.
2. Không torch: các module trong TORCH_FREE không được import torch ở cấp module (import trong hàm được phép,
   vì chỉ chạy khi gọi hàm đó), để lệnh `check` và phần lớn test chạy khi chưa cài torch.
"""

import ast
import os

PACKAGE = "badtime_repro"
ROOT = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), PACKAGE)

# Tầng thấp -> cao. Khóa là tiền tố tên module (sau "badtime_repro.").
LAYERS = [
    ("common", "assumptions"),                      # 0  tiện ích, dữ liệu tĩnh
    ("catalog", "config"),                          # 1  danh mục, schema cấu hình
    ("data",),                                      # 2  nguồn, phân chia, chuẩn hóa, cửa sổ
    ("attack",),                                    # 3  khối thiết kế BadTime
    ("metrics", "runs", "support"),                 # 4  hợp đồng phép đo, hiện vật lần chạy, kiểm tra hỗ trợ
    ("models",),                                    # 5  mô hình dự báo
    ("training",),                                  # 6  Trainer + TrainingMethod
    ("experiment",),                                # 7  mặt tiền ứng dụng
    ("evaluation",),                                # 8  đối chứng, tính lại, báo cáo
    ("cli", "__main__"),                            # 9  giao diện dòng lệnh
]

TORCH_FREE = (
    "common", "assumptions", "catalog", "config", "data", "metrics", "runs", "support", "experiment",
    "evaluation.report", "evaluation.recompute", "cli",
    "attack.budget", "attack.channels", "attack.injection", "attack.lag", "attack.patterns", "attack.plan",
    "attack.sample_selection", "models.__init__", "training.methods.__init__", "training.schedule",
)


def _layer_of(name):
    for i, prefixes in enumerate(LAYERS):
        for p in prefixes:
            if name == p or name.startswith(p + "."):
                return i

    raise AssertionError(f"Module '{name}' chưa được xếp tầng trong LAYERS.")


def _modules():
    for dirpath, dirnames, filenames in os.walk(ROOT):
        dirnames[:] = [d for d in dirnames if d not in ("__pycache__", "tslib")]
        for f in filenames:
            if f.endswith(".py"):
                path = os.path.join(dirpath, f)
                rel = os.path.relpath(path, ROOT)[:-3].replace(os.sep, ".")
                yield rel, path


def _imports(rel, path):
    """(tên module nội bộ được import, có phải import torch không) cho mọi câu import trong file."""
    tree = ast.parse(open(path, encoding="utf-8").read())
    is_pkg = rel.endswith("__init__")
    here = rel.split(".")[:-1]                        # gói chứa module
    in_function = {id(n) for f in ast.walk(tree) if isinstance(f, (ast.FunctionDef, ast.AsyncFunctionDef))
                   for n in ast.walk(f)}
    internal, torch_used = [], False                  # torch_used: chỉ tính import ở cấp module (khi nạp module)
    for node in ast.walk(tree):
        top = id(node) not in in_function
        if isinstance(node, ast.Import):
            for alias in node.names:
                torch_used |= top and alias.name.split(".")[0] == "torch"
                if alias.name.startswith(PACKAGE + "."):
                    internal.append(alias.name[len(PACKAGE) + 1:])

        elif isinstance(node, ast.ImportFrom):
            if node.level == 0:
                torch_used |= top and (node.module or "").split(".")[0] == "torch"
                if (node.module or "").startswith(PACKAGE + "."):
                    internal.append(node.module[len(PACKAGE) + 1:])
            else:
                base = here[: len(here) - (node.level - 1)] if not is_pkg or node.level > 1 else here
                target = ".".join(base + ([node.module] if node.module else []))
                if target:
                    internal.append(target)

    return internal, torch_used


def test_every_module_has_a_layer():
    for rel, _ in _modules():
        _layer_of(rel.replace(".__init__", "") if rel != "__init__" else "common")


def test_imports_respect_layers():
    violations = []
    for rel, path in _modules():
        if rel == "__init__":
            continue

        src_layer = _layer_of(rel.replace(".__init__", ""))
        for target in _imports(rel, path)[0]:
            if _layer_of(target) > src_layer:
                violations.append(f"{rel} (tầng {src_layer}) -> {target} (tầng {_layer_of(target)})")

    assert not violations, "Import ngược tầng:\n" + "\n".join(violations)


def test_torch_free_modules_do_not_import_torch():
    offenders = []
    for rel, path in _modules():
        name = rel.replace(".__init__", "") if rel.endswith(".__init__") and rel != "__init__" else rel
        declared = any(name == p or name.startswith(p + ".") or rel == p for p in TORCH_FREE)
        if declared and _imports(rel, path)[1]:
            offenders.append(rel)

    assert not offenders, f"Module khai báo không dùng torch nhưng import torch: {offenders}"

In [ ]:
%%writefile tests/test_attack.py
"""Các khối thiết kế BadTime, phần numpy (Thuật toán 1-4, Công thức 12-14)."""

import numpy as np
import pytest

from badtime_repro.attack.budget import BUDGET_PROTOCOLS, get_budget_protocol
from badtime_repro.attack.channels import FixedInfluenceScorer, PoisonChannelSelector, TargetChannelResolver
from badtime_repro.attack.injection import ReferenceInjector
from badtime_repro.attack.lag import LagEstimator
from badtime_repro.attack.patterns import get_pattern
from badtime_repro.attack.plan import AttackPlan, AttackPlanner
from badtime_repro.attack.sample_selection import HybridSampleSelector
from badtime_repro.common.errors import PlanError
from badtime_repro.support import SupportChecker
from badtime_repro.data.pipeline import DataPipeline
from helpers import HAS_ETTH1, ROOT, default_config, toy_plan

INFLUENCE = [0.1, 0.2, 0.05, 0.3, 0.1, 0.1, 0.15]


# ------------------------------------------------------------------ ngân sách, mục tiêu, kênh

def test_budget_protocols():
    assert get_budget_protocol("keep_total").split(29, 2) == [15, 14]
    assert get_budget_protocol("equal_segments").split(29, 2) == [14, 14]
    assert get_budget_protocol("ceil_segments").split(29, 2) == [15, 15]
    for protocol in BUDGET_PROTOCOLS.values():
        with pytest.raises(PlanError):
            protocol.split(1, 2)


def test_patterns():
    p = get_pattern("hump_backtime_cone").build(53, 1.0, 0.0, 1)
    assert p.shape == (53, 1)
    assert p[0, 0] == pytest.approx(0.1278) and p[-1, 0] == pytest.approx(0.1278)
    assert p[26, 0] == pytest.approx(0.3853)
    q = get_pattern("hump_sine").build(53, 2.0, 0.5, 2)
    assert q.shape == (53, 2) and q.max() <= 2.5 and q.min() > 0.5
    with pytest.raises(PlanError):
        get_pattern("star")


def test_channel_resolution_and_selection():
    idx, note = TargetChannelResolver(["OT"], 0.1).resolve(["a", "b", "c", "d", "e", "f", "OT"])
    assert idx == [6] and note is None
    with pytest.raises(PlanError):
        TargetChannelResolver(["XX"], 0.1).resolve(["a", "OT"])

    sel = PoisonChannelSelector(0.3, exclude_targets=False)
    assert sel.count(7) == 2
    assert sel.select(INFLUENCE, [6]) == [3, 1]
    assert PoisonChannelSelector(0.3, exclude_targets=True).select([0, 0, 0, 0, 0, 0, 1.0], [6]) == [0, 1]


# ------------------------------------------------------------------ Thuật toán 1

def test_sample_selection_groups():
    d = np.array([5.0, 1.0, 9.0, 3.0, 7.0, 2.0, 8.0, 4.0, 6.0, 0.0])
    sel = HybridSampleSelector(0.5, 0.4).select(d)      # k_sim = floor(2.0) = 2, k_diff = floor(3.0) = 3
    assert sel.info["k_similar"] == 2 and sel.info["k_different"] == 3
    assert set(sel.similar_pos.tolist()) == {9, 1}
    assert set(sel.different_pos.tolist()) == {2, 6, 4}
    assert len(sel.all_pos) == 5


# ------------------------------------------------------------------ Thuật toán 3

def test_lag_recovers_shift():
    rng = np.random.default_rng(0)
    n = 3000
    a = rng.standard_normal(n)
    series = np.zeros((n, 2))
    series[:, 0] = a
    series[20:, 1] = a[:-20]           # b(t) = a(t - 20): a dẫn trước 20 bước
    est = LagEstimator(96, 8).estimate(series, 0, [1], np.arange(0, n - 96), 10)
    assert est.lag == 20


def test_lag_skips_constant_segments():
    series = np.zeros((500, 2))
    series[:, 0] = np.arange(500)
    with pytest.raises(PlanError):
        LagEstimator(96, 8).estimate(series, 0, [1], np.arange(0, 300), 10)


# ------------------------------------------------------------------ Công thức 12-14

def test_inject_exact_cells_and_no_side_effects():
    plan = toy_plan()
    inj = ReferenceInjector(plan)
    x = np.zeros((2, 10, 3))
    x_t = inj.inject(x, plan.init_trigger)
    assert np.all(x == 0)
    assert inj.input_mask().sum() == 5
    assert np.array_equal(x_t[0] != 0, inj.input_mask())
    assert x_t[0, 6:9, 0].tolist() == [1.0, 2.0, 3.0]
    assert x_t[0, 3:5, 1].tolist() == [4.0, 5.0]


def test_inject_disabled_is_identity():
    plan = toy_plan(lam=0.0)
    x = np.random.default_rng(1).standard_normal((3, 10, 3))
    assert np.array_equal(ReferenceInjector(plan).inject(x, plan.init_trigger), x)


def test_poison_labels_only_target_segment():
    plan = toy_plan()
    y = np.zeros((2, 6, 3))
    yp = ReferenceInjector(plan).poison_labels(y)
    assert np.all(yp[:, :4, 2] == 5.0)
    assert np.all(yp[:, 4:, 2] == 0.0) and np.all(yp[:, :, :2] == 0.0)
    assert np.all(y == 0)


def test_plan_validation_and_round_trip():
    plan = toy_plan()
    again = AttackPlan.from_dict(plan.to_dict())
    assert again.to_dict() == plan.to_dict()
    plan.offsets = [9, 0]                 # đoạn [9, 12) vượt ra ngoài đầu vào [0, 10)
    with pytest.raises(PlanError):
        plan.validate()


# ------------------------------------------------------------------ dữ liệu thật

@pytest.mark.skipif(not HAS_ETTH1, reason="chưa có dataset/ETTh1/ETTh1.csv")
def test_real_plan_consistency():
    cfg = SupportChecker().check(default_config("method=badtime")).raise_if_blocked()
    ds = DataPipeline(ROOT).prepare(cfg)
    plan = AttackPlanner(cfg.attack, FixedInfluenceScorer(INFLUENCE)).plan(ds, seed=0)

    assert plan.target_idx == [6] and plan.poison_idx == [3, 1]
    assert plan.seg_lens == [15, 14] and plan.trigger_len == 29
    assert len(plan.poison_starts) == 1025                    # floor(0.03 x 10261) + floor(0.07 x 10261)
    assert len(plan.similar_starts) == 307 and len(plan.different_starts) == 718
    assert all(15 <= lag <= 96 for lag in plan.lags)          # [ceil(29/2), t_IN]
    assert set(plan.poison_starts.tolist()) <= set(ds.starts["train"].tolist())

    # Trigger khởi tạo = đúng đoạn đầu vào của cửa sổ có nhãn gần mục tiêu nhất (Công thức 9)
    x, _, _ = ds.windows("train", np.array([plan.init_start]))
    for (t_s, t_e), ch, off in plan.segments():
        assert np.allclose(plan.init_trigger[t_s:t_e], x[0, off:off + (t_e - t_s), ch])

    # Mọi dữ liệu dùng để lựa chọn thuộc phần huấn luyện
    b_e = ds.bounds.train[1]
    assert plan.poison_starts.max() + 96 + 96 <= b_e
    assert plan.init_start + 96 + 96 <= b_e

In [ ]:
%%writefile tests/test_catalog.py
"""Danh mục và các registry cài đặt phải nhất quán: mục nào chạy được thì phải có cài đặt."""

from badtime_repro.catalog import CATALOG
from badtime_repro.data.sources import DATA_SOURCES
from badtime_repro.models import MODEL_FACTORIES
from badtime_repro.training.methods import TRAINING_METHODS


def test_runnable_entries_have_implementations():
    for name, spec in CATALOG.datasets.items():
        if spec.status.runnable:
            assert spec.source in DATA_SOURCES, name

    for name, spec in CATALOG.models.items():
        if spec.status.runnable:
            assert name in MODEL_FACTORIES, name

    for name, spec in CATALOG.methods.items():
        if spec.status.runnable:
            assert name in TRAINING_METHODS, name


def test_described_entries_are_not_registered():
    for name, spec in CATALOG.methods.items():
        if not spec.status.runnable:
            assert name not in TRAINING_METHODS, name

    for name, spec in CATALOG.models.items():
        if not spec.status.runnable:
            assert name not in MODEL_FACTORIES, name


def test_supported_slots_are_runnable():
    for dataset, model in CATALOG.supported_slots:
        assert CATALOG.datasets[dataset].status.runnable
        assert CATALOG.models[model].status.runnable

In [ ]:
%%writefile tests/test_config.py
"""Schema cấu hình, bộ nạp và kiểm tra hỗ trợ (docs/07)."""

import json
import os
import shutil
import tempfile

import pytest

from badtime_repro.catalog import CATALOG
from badtime_repro.common.errors import ConfigInvalidError, DataUnavailableError, UnsupportedSelectionError
from badtime_repro.config import ExperimentConfig, load_config
from badtime_repro.experiment import Experiment
from badtime_repro.support import SupportChecker, SupportVerdict
from helpers import ROOT, default_config


def check(*overrides):
    return SupportChecker().check(default_config(*overrides))


# ------------------------------------------------------------------ schema

def test_default_config_matches_shipped_file():
    with open(os.path.join(ROOT, "configs", "etth1_timesnet_96.json"), encoding="utf-8") as f:
        assert ExperimentConfig().to_dict() == json.load(f)


def test_round_trip():
    cfg = default_config("attack.pattern.amplitude=2", "train.epochs=3")
    again = ExperimentConfig.from_dict(cfg.to_dict())
    assert again == cfg
    assert again.attack.pattern.amplitude == 2.0 and isinstance(again.attack.pattern.amplitude, float)


def test_unknown_key_is_rejected_with_path():
    with pytest.raises(ConfigInvalidError) as exc:
        default_config("attack.gatt.epochs=3")

    assert "attack.gatt" in str(exc.value)


def test_wrong_type_and_length_rejected():
    for ov, needle in (("train.epochs=ten", "train.epochs"),
                       ("train.deterministic=1", "train.deterministic"),
                       ("attack.model_loss_weights=[1,2]", "model_loss_weights")):
        with pytest.raises(ConfigInvalidError) as exc:
            default_config(ov)

        assert needle in str(exc.value), ov


def test_all_errors_reported_together():
    with pytest.raises(ConfigInvalidError) as exc:
        default_config("train.epochs=ten", "foo=1")

    assert len(exc.value.messages) == 3   # tiêu đề + 2 lỗi


def test_config_is_immutable():
    cfg = default_config()
    with pytest.raises(Exception):
        cfg.dataset = "ETTh2"


# ------------------------------------------------------------------ hỗ trợ

def test_poc_slot_ok():
    assert check().verdict is SupportVerdict.OK


def test_case_insensitive_then_unsupported():
    r = check("dataset=ETTH2")
    assert r.verdict is SupportVerdict.UNSUPPORTED
    assert r.config.dataset == "ETTh2"
    assert "dataset/ETTh2/" in r.messages[0]


def test_unknown_name_is_invalid_not_unsupported():
    assert check("dataset=ETTh9").verdict is SupportVerdict.INVALID


def test_out_of_scope_selections_rejected():
    for ov in ("model=Autoformer", "pred_len=720", "pred_len=192", "method=backtime"):
        assert check(ov).verdict is SupportVerdict.UNSUPPORTED, ov


def test_raise_if_blocked_maps_to_error_types():
    with pytest.raises(UnsupportedSelectionError):
        check("dataset=ETTh2").raise_if_blocked()

    with pytest.raises(ConfigInvalidError):
        check("dataset=Foo").raise_if_blocked()


def test_semantic_contradictions_invalid():
    for ov in ("attack.target_channels=[]", "attack.poison_ratio=1.5", "attack.budget_protocol=foo",
               "attack.pattern.shape=star", "data.split_protocol=random", "train.lradj=cosine"):
        assert check(ov).verdict is SupportVerdict.INVALID, ov


def test_exit_codes():
    assert ConfigInvalidError.exit_code == 2
    assert UnsupportedSelectionError.exit_code == 3
    assert DataUnavailableError.exit_code == 4


# ------------------------------------------------------------------ dữ liệu theo cấu hình

def test_existing_folder_does_not_make_dataset_supported():
    tmp = tempfile.mkdtemp()
    try:
        os.makedirs(os.path.join(tmp, "dataset", "ETTh2"))
        with pytest.raises(UnsupportedSelectionError):
            Experiment(tmp).resolve(default_config("dataset=ETTh2"))
    finally:
        shutil.rmtree(tmp)


def test_missing_data_creates_folder_and_reports():
    tmp = tempfile.mkdtemp()
    try:
        with pytest.raises(DataUnavailableError) as exc:
            Experiment(tmp).resolve(default_config())

        assert os.path.isdir(os.path.join(tmp, "dataset", "ETTh1"))
        assert not os.path.isdir(os.path.join(tmp, "dataset", "ETTh2"))
        assert "Đã chuẩn bị thư mục dataset/ETTh1/" in exc.value.messages[0]
    finally:
        shutil.rmtree(tmp)


def test_load_config_from_file_and_override_order():
    tmp = tempfile.mkdtemp()
    try:
        path = os.path.join(tmp, "c.json")
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"train": {"epochs": 4}}, f)

        cfg = load_config(path, ["train.epochs=5"])
        assert cfg.train.epochs == 5 and cfg.train.batch_size == 32
    finally:
        shutil.rmtree(tmp)


def test_catalog_poc_slot():
    assert CATALOG.slot_supported("ETTh1", "TimesNet")
    assert CATALOG.canonical("etth1", CATALOG.datasets) == "ETTh1"
    assert CATALOG.canonical(3, CATALOG.datasets) is None


def test_more_semantic_checks():
    for ov in ("train.device=gpu", "attack.trigger_total_len=200", 'attack.target_channels=["OT","OT"]',
               "attack.model_loss_weights=[1,-1,1]", "attack.lag.min_corr_len=1", "timesnet.dropout=1.5",
               "timesnet.embed=fixed", "attack.gat.heads=0", "train.learning_rate=0"):
        assert check(ov).verdict is SupportVerdict.INVALID, ov

    assert check("train.device=cuda:1").verdict is SupportVerdict.OK


def test_errors_are_collected_not_first_only():
    r = check("train.device=gpu", "attack.gat.heads=0")
    assert len(r.messages) == 3       # tiêu đề + 2 lỗi


def test_unverified_components_are_announced():
    notes = " ".join(check().messages)
    assert "TimesNet" in notes and "clean" in notes      # chưa chạy kiểm tra torch -> chưa kiểm chứng


def test_schedule_names_match_implementation():
    from badtime_repro.support import LR_SCHEDULES
    from badtime_repro.training.schedule import SCHEDULES

    assert set(LR_SCHEDULES) == set(SCHEDULES)

In [ ]:
%%writefile tests/test_data.py
"""Phân chia, cửa sổ, chuẩn hóa, nguồn dữ liệu (docs/03, docs/06)."""

import os
import shutil
import tempfile

import numpy as np
import pytest

from badtime_repro.catalog import CATALOG
from badtime_repro.common.errors import DataUnavailableError
from badtime_repro.data.pipeline import DataPipeline
from badtime_repro.data.scaling import StandardScaler
from badtime_repro.data.splits import RatioSplit, TslibEttHourSplit
from badtime_repro.data.windows import WindowSpec
from helpers import ETTH1_CSV, HAS_ETTH1, ROOT, default_config, synthetic_dataset


def test_split_ratio_boundaries():
    assert RatioSplit().boundaries(17420).to_dict() == {
        "train": [0, 10452], "val": [10452, 13936], "test": [13936, 17420]}
    assert TslibEttHourSplit().boundaries(17420).to_dict() == {
        "train": [0, 8640], "val": [8640, 11520], "test": [11520, 14400]}
    with pytest.raises(DataUnavailableError):
        TslibEttHourSplit().boundaries(1000)


def test_window_boundaries_small_example():
    bounds = RatioSplit().boundaries(20)        # train [0,12) val [12,16) test [16,20)
    w = WindowSpec(3, 2)
    assert w.starts(bounds, "train", True).tolist() == list(range(0, 8))    # s + 3 + 2 <= 12
    assert w.starts(bounds, "val", True).tolist() == [9, 10, 11]            # nhãn [12,14) ... [14,16)
    assert w.starts(bounds, "val", False).tolist() == []


def test_windows_exact_rows_and_labels_inside_partition():
    ds = synthetic_dataset()
    x, y, _ = ds.windows("val")
    s = ds.starts["val"][0]
    assert np.allclose(x[0], ds.norm[s:s + 4])
    assert np.allclose(y[0], ds.norm[s + 4:s + 7])
    v_s, v_e = ds.bounds.val
    for st in ds.starts["val"]:
        assert v_s <= st + 4 and st + 7 <= v_e


def test_scaler_uses_train_only():
    ds = synthetic_dataset()
    raw = ds.raw.copy()
    raw[30:] += 1000.0
    b_s, b_e = ds.bounds.train
    sc = StandardScaler.fit(raw[b_s:b_e])
    assert np.allclose(sc.mean, ds.scaler.mean) and np.allclose(sc.std, ds.scaler.std)


def test_scaler_rejects_constant_channel_and_round_trips():
    with pytest.raises(DataUnavailableError):
        StandardScaler.fit(np.ones((10, 2)))

    sc = StandardScaler([1.0, 2.0], [3.0, 4.0])
    v = np.array([[0.5, -1.0]])
    assert np.allclose(sc.inverse(sc.transform(v)), v)
    assert StandardScaler.from_dict(sc.to_dict()).to_dict() == sc.to_dict()


def test_windows_are_independent_copies():
    ds = synthetic_dataset()
    x, _, _ = ds.windows("train")
    before = ds.norm.copy()
    x[0, -1, 0] = 999.0
    assert np.array_equal(ds.norm, before)
    assert x[1, -2, 0] != 999.0          # cửa sổ kế tiếp dùng chung hàng đó không bị đổi


def test_invalid_csv_is_reported_not_fixed():
    tmp = tempfile.mkdtemp()
    try:
        d = os.path.join(tmp, "dataset", "ETTh1")
        os.makedirs(d)
        with open(os.path.join(d, "ETTh1.csv"), "w") as f:
            f.write("date,a\n2016-07-01 00:00:00,1\n")

        with pytest.raises(DataUnavailableError) as exc:
            DataPipeline(tmp).prepare(default_config())

        assert "Cột không khớp" in str(exc.value)
    finally:
        shutil.rmtree(tmp)


@pytest.mark.skipif(not HAS_ETTH1, reason="chưa có dataset/ETTh1/ETTh1.csv")
def test_etth1_profile():
    ds = DataPipeline(ROOT).prepare(default_config())
    assert ds.report.n_rows == 17420 and ds.report.n_missing == 0
    assert ds.report.channels == list(CATALOG.datasets["ETTh1"].expected_columns[1:])
    assert {k: len(v) for k, v in ds.starts.items()} == {"train": 10261, "val": 3389, "test": 3389}
    assert ds.report.path == os.path.abspath(ETTH1_CSV)

In [ ]:
%%writefile tests/test_evaluation.py
"""Phép đo, thư mục lần chạy, tính lại chỉ số và báo cáo (không cần torch)."""

import shutil
import tempfile

import numpy as np
import pytest

from badtime_repro.attack.injection import ReferenceInjector
from badtime_repro.common.errors import DataUnavailableError
from badtime_repro.common.jsonio import write_json
from badtime_repro.data.scaling import StandardScaler
from badtime_repro.metrics import AttackTarget, MetricCalculator
from badtime_repro.evaluation.recompute import MetricRecomputer
from badtime_repro.evaluation.report import MarkdownReportRenderer, ReportBuilder
from badtime_repro.runs.store import RunFiles, RunIndex, RunStore
from helpers import default_config, etth1_toy_plan, toy_plan


def test_metrics_hand_example():
    calc = MetricCalculator(StandardScaler([0.0, 10.0], [1.0, 2.0]))
    y = np.zeros((1, 2, 2))
    pc = np.array([[[1.0, 0.0], [0.0, -1.0]]])
    pt = np.array([[[3.0, 2.0], [1.0, 2.0]]])
    m = calc.forecast(pc, y, pt, AttackTarget((1,), np.array([[1.0]])))   # t_pin = 1, kênh đích 1
    assert m["mae_c_norm"] == pytest.approx(0.5)
    assert m["mae_c_orig"] == pytest.approx((1 + 0 + 0 + 2) / 4)
    assert m["mae_pa_norm"] == pytest.approx(1.0)       # |2 - 1|
    assert m["mae_pa_orig"] == pytest.approx(2.0)
    assert m["mae_pn_norm"] == pytest.approx(2.0)       # kênh 0: |3|, |1|
    assert m["mae_rest_norm"] == pytest.approx(2.0)     # kênh 1, bước 1: |2|


def test_input_change_counts():
    plan = toy_plan()
    x = np.zeros((4, 10, 3))
    st = MetricCalculator.input_change(x, ReferenceInjector(plan).inject(x, plan.init_trigger), plan)
    assert st["changed_cells_per_window_mean"] == 5
    assert st["registered_cells_per_window"] == 5


def _fake_run(root, method, seed, rng, plan=None):
    """Một lần chạy giả hoàn chỉnh: profile, (plan), preds, metrics đúng như Trainer ghi."""
    cfg = default_config(f"method={method}")
    store = RunStore.create(root, cfg, method, seed)
    store.mark_running(method, seed)
    scaler = StandardScaler(rng.standard_normal(7), rng.random(7) + 0.5)
    store.write_json(RunFiles.PROFILE, {"method": method, "seed": seed, "config": cfg.to_dict(),
                                        "data": {"scaler": scaler.to_dict()}})
    calc = MetricCalculator(scaler)
    target = None
    if plan is not None:
        store.write_json(RunFiles.PLAN, plan.to_dict())
        target = AttackTarget.of(plan)

    metrics = {}
    for split in ("val", "test"):
        y = rng.standard_normal((20, 96, 7)).astype(np.float32)
        pc = rng.standard_normal((20, 96, 7)).astype(np.float32)
        pt = rng.standard_normal((20, 96, 7)).astype(np.float32) if plan is not None else None
        metrics[split] = calc.forecast(pc.astype(np.float64), y.astype(np.float64),
                                       None if pt is None else pt.astype(np.float64), target)
        arrays = {"pred_clean": pc, "y": y, "starts": np.arange(20)}
        if pt is not None:
            arrays["pred_trig"] = pt

        store.save_predictions(split, **arrays)

    store.write_json(RunFiles.METRICS, metrics)
    store.mark_completed(1)
    return store


def test_run_store_recompute_and_report():
    tmp = tempfile.mkdtemp()
    try:
        rng = np.random.default_rng(0)
        plan = etth1_toy_plan()
        clean = _fake_run(tmp, "clean", 1, rng)
        bad = _fake_run(tmp, "badtime", 1, rng, plan)
        _fake_run(tmp, "badtime", 2, rng, plan)

        for run in (clean, bad):
            res = MetricRecomputer().run(run.path)
            assert all(v["ok"] for v in res.values()), res

        slot_dir = f"{tmp}/runs/ETTh1_TimesNet_96_96"
        index = RunIndex(slot_dir)
        assert len(index.completed("badtime")) == 2
        assert index.latest("badtime", 1).path == bad.path

        ctl = {s: {"clean_model|trigger_input": {"mae_pa_norm": 0.8, "mae_pn_norm": 0.43}} for s in ("val", "test")}
        write_json(bad.file(RunFiles.controls(clean.name)), ctl)

        builder = ReportBuilder(slot_dir, "ETTh1_TimesNet_96_96")
        report = builder.build()
        row = report["rows"]["test"]
        assert row["BadTime"]["n_runs"] == 2 and row["BadTime"]["mae_pa"]["paper"] == 0.376
        assert row["Benign"]["mae_pa"]["runs"] == [0.8]
        assert row["Benign"]["mae_pa"]["warn_over_5pct"] is False       # |0.8 - 0.781| / 0.781 < 5%
        text = MarkdownReportRenderer().render(report)
        assert "| BadTime | mae_pa | 0.376 |" in text
        builder.write(report)
    finally:
        shutil.rmtree(tmp)


def test_report_without_runs_is_data_error():
    tmp = tempfile.mkdtemp()
    try:
        with pytest.raises(DataUnavailableError):
            ReportBuilder(f"{tmp}/runs/x", "x").build()
    finally:
        shutil.rmtree(tmp)


def test_failed_run_is_not_listed():
    tmp = tempfile.mkdtemp()
    try:
        store = RunStore.create(tmp, default_config(), "clean", 5)
        store.mark_running("clean", 5)
        try:
            raise RuntimeError("boom")
        except RuntimeError as exc:
            store.mark_failed(exc)

        assert store.status()["state"] == "failed" and "boom" in store.status()["error"]
        assert RunIndex(f"{tmp}/runs/ETTh1_TimesNet_96_96").completed("clean") == []
    finally:
        shutil.rmtree(tmp)

In [ ]:
%%writefile tests/test_torch_units.py
"""Thành phần torch, không huấn luyện thật. Bỏ qua nếu chưa cài torch."""

import numpy as np
import pytest

torch = pytest.importorskip("torch")

from badtime_repro.attack.gat import GatInfluenceScorer, GraphAttentionNextStep  # noqa: E402
from badtime_repro.attack.injection import ReferenceInjector  # noqa: E402
from badtime_repro.attack.objectives import (  # noqa: E402
    BackdoorModelObjective,
    CleanObjective,
    SquaredPlusAbsolute,
    TriggerObjective,
)
from badtime_repro.attack.trigger import Trigger  # noqa: E402
from badtime_repro.catalog import CATALOG  # noqa: E402
from badtime_repro.config import GatConfig  # noqa: E402
from badtime_repro.models import MODEL_FACTORIES, build_model  # noqa: E402
from badtime_repro.training.methods import TRAINING_METHODS  # noqa: E402
from badtime_repro.training.selection import (  # noqa: E402
    CleanConstrainedAttackSelector,
    EarlyStoppingSelector,
    StateSnapshot,
)
from helpers import default_config, etth1_toy_plan  # noqa: E402


def test_registries_resolve():
    for name in TRAINING_METHODS:
        assert TRAINING_METHODS.resolve(name).name == name

    for name in MODEL_FACTORIES:
        assert CATALOG.models[name].status.runnable
        MODEL_FACTORIES.resolve(name)


def test_clean_loss_names_match_config():
    from badtime_repro.support import CLEAN_LOSSES

    assert set(CLEAN_LOSSES) == set(CleanObjective.KINDS)
    with pytest.raises(ValueError):
        CleanObjective("l1", "sum")


def test_inject_matches_reference_and_keeps_input():
    plan = etth1_toy_plan(lam=0.7)
    x = np.random.default_rng(0).standard_normal((4, 96, 7))
    trig = Trigger(plan)
    xt = torch.as_tensor(x, dtype=torch.float32)
    before = xt.clone()
    out = trig.inject(xt).detach().numpy()
    assert torch.equal(xt, before)
    ref = ReferenceInjector(plan).inject(x.astype(np.float32).astype(np.float64), plan.init_trigger.astype(np.float32))
    assert np.allclose(out, ref, atol=1e-6)


def test_disabled_interpolation_is_identity():
    plan = etth1_toy_plan(lam=0.0)
    x = torch.randn(3, 96, 7)
    assert torch.equal(Trigger(plan).inject(x), x)


def test_poison_labels_torch():
    plan = etth1_toy_plan()
    y = torch.zeros(2, 96, 7)
    yp = Trigger(plan).poison_labels(y)
    assert torch.all(yp[:, :53, 6] == 1) and torch.all(yp[:, 53:, 6] == 0) and torch.all(yp[:, :, :6] == 0)
    assert torch.all(y == 0)


def test_objectives_and_empty_groups():
    pred, y = torch.randn(4, 96, 7), torch.randn(4, 96, 7)
    obj = BackdoorModelObjective(7, [6], (1, 1, 1), "sum")
    for flags in (torch.zeros(4, dtype=torch.bool), torch.ones(4, dtype=torch.bool)):
        loss, _ = obj(pred, y, y, flags)
        assert torch.isfinite(loss)

    _, parts = obj(pred, y, y, torch.zeros(4, dtype=torch.bool))
    assert set(parts) == {"clean"}

    # 'sum' = tổng phần tử / số mẫu; 'mean' = trung bình phần tử
    d = pred - y
    assert float(SquaredPlusAbsolute("sum")(pred, y)) == pytest.approx(float((d.pow(2).sum() + d.abs().sum()) / 4))
    assert float(SquaredPlusAbsolute("mean")(pred, y)) == pytest.approx(float(d.pow(2).mean() + d.abs().mean()))

    # Nhánh sạch 'mse_mae' và BadTime khi lô không có mẫu bị chọn cho cùng giá trị loss
    clean_loss, _ = CleanObjective("mse_mae", "sum")(pred, y)
    bad_loss, _ = obj(pred, y, y, torch.zeros(4, dtype=torch.bool))
    assert torch.equal(clean_loss, bad_loss)


def test_smooth_and_range_loss():
    trig = Trigger(etth1_toy_plan(), init=np.zeros(29))
    assert float(trig.smooth_loss(0.8)) == 0.0 and float(trig.range_loss()) == 0.0
    with torch.no_grad():
        trig.tau[0] = 5.0                  # vượt max 3 và nhảy 5 > 0.8
    assert float(trig.range_loss()) == pytest.approx(2.0 / 29)
    assert float(trig.smooth_loss(0.8)) == pytest.approx((5.0 - 0.8) ** 2 / 29)


def _tiny_model():
    torch.manual_seed(0)
    return build_model(default_config("timesnet.d_model=8", "timesnet.d_ff=8", "timesnet.e_layers=1"), 7)


def test_model_shape_and_no_decoder_input():
    out = _tiny_model()(torch.randn(5, 96, 7), torch.randn(5, 96, 4))
    assert out.shape == (5, 96, 7)


def test_trigger_gradient_with_frozen_model():
    model = _tiny_model()
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    trig = Trigger(etth1_toy_plan())
    x, y = torch.randn(8, 96, 7), torch.randn(8, 96, 7)
    pred = model(trig.inject(x), torch.zeros(8, 96, 4))
    loss, _ = TriggerObjective(7, [6], (1, 1, 1, 1), "sum", 0.8)(pred, trig.poison_labels(y), y, trig)
    loss.backward()
    assert trig.tau.grad is not None and torch.any(trig.tau.grad != 0)
    assert all(p.grad is None for p in model.parameters())


def test_gat_attention_and_influence():
    net = GraphAttentionNextStep(96, 8, 4, 0.2, True)
    alpha, _ = net.attention(torch.randn(3, 7, 96))
    assert alpha.shape == (3, 4, 7, 7)
    assert torch.allclose(alpha.sum(-1), torch.ones(3, 4, 7))

    series = np.random.default_rng(0).standard_normal((400, 7))
    cfg = GatConfig(hidden_dim=8, heads=4, epochs=1, batch_size=64)
    res = GatInfluenceScorer(cfg, torch.device("cpu")).score(series, 96, [6], seed=0)
    assert res.scores.shape == (7,) and abs(res.scores.sum() - 1.0) < 1e-5   # một kênh đích: tổng attention = 1
    assert len(res.history) == 1


def _snap(epoch, mae_c, mae_pa, mse):
    return StateSnapshot(epoch, {"mae_c_norm": mae_c, "mae_pa_norm": mae_pa}, mse, {}, None)


def test_state_selectors():
    es = EarlyStoppingSelector(patience=2)
    stops = [es.observe(_snap(e, 0, 0, mse)) for e, mse in enumerate([0.5, 0.4, 0.45, 0.46], start=1)]
    assert stops == [False, False, False, True] and es.select()[0].epoch == 2

    cc = CleanConstrainedAttackSelector(tolerance=0.05)
    for e, (c, pa) in enumerate([(0.40, 0.9), (0.41, 0.5), (0.45, 0.3)], start=1):
        assert cc.observe(_snap(e, c, pa, 0)) is False

    chosen, info = cc.select()
    assert chosen.epoch == 2 and info["feasible_epochs"] == [1, 2]     # 0.45 > 1.05 * 0.40

In [ ]:
%%writefile tests/test_training.py
"""Huấn luyện thật trên ETTh1 (CPU, mô hình nhỏ, 2 epoch). Chỉ chạy khi BADTIME_SLOW=1.

CPU để so sánh chính xác giữa hai nhánh (cuDNN có thể không tất định). Không dùng để so số với bài báo.
"""

import os
import shutil
import tempfile

import pytest

torch = pytest.importorskip("torch")

from badtime_repro.evaluation.controls import ControlExperiment  # noqa: E402
from badtime_repro.evaluation.recompute import MetricRecomputer  # noqa: E402
from badtime_repro.experiment import Experiment  # noqa: E402
from badtime_repro.runs.store import RunFiles  # noqa: E402
from helpers import HAS_ETTH1, ROOT, default_config  # noqa: E402

SLOW = os.environ.get("BADTIME_SLOW") == "1" and HAS_ETTH1

SMALL = [
    "train.epochs=2", "train.device=cpu", "attack.gat.epochs=1",
    "timesnet.d_model=8", "timesnet.d_ff=8", "timesnet.e_layers=1", "timesnet.top_k=3",
]


def _project_copy():
    tmp = tempfile.mkdtemp()
    shutil.copytree(os.path.join(ROOT, "dataset"), os.path.join(tmp, "dataset"))
    return tmp


@pytest.mark.skipif(not SLOW, reason="đặt BADTIME_SLOW=1 để chạy huấn luyện thật")
def test_pass_through_equals_clean_and_artifacts():
    """BadTime với 0% mẫu bị chọn phải cho cùng lịch sử huấn luyện với nhánh sạch cùng loss."""
    tmp = _project_copy()
    try:
        cfg = default_config(*SMALL, "train.clean_loss=mse_mae", "train.early_stopping_patience=99",
                             "attack.poison_ratio=0.0")
        exp = Experiment(tmp)
        clean = exp.train(cfg, "clean", 7)
        bad = exp.train(cfg, "badtime", 7)

        for a, b in zip(clean.history(), bad.history()):
            assert a["train"]["total"] == pytest.approx(b["train"]["total"], rel=1e-6)
            assert a["val"]["mae_c_norm"] == pytest.approx(b["val"]["mae_c_norm"], rel=1e-6)

        for run in (clean, bad):
            assert all(v["ok"] for v in MetricRecomputer().run(run.path).values())

        pair = bad.state_pair()
        assert pair.tau is not None and pair.plan is not None
        assert bad.metrics()["selection"]["chosen_epoch"] == pair.epoch
        assert bad.status()["state"] == "completed"
    finally:
        shutil.rmtree(tmp)


@pytest.mark.skipif(not SLOW, reason="đặt BADTIME_SLOW=1 để chạy huấn luyện thật")
def test_badtime_short_run_and_controls():
    tmp = _project_copy()
    try:
        cfg = default_config(*SMALL)
        exp = Experiment(tmp)
        clean = exp.train(cfg, "clean", 3)
        bad = exp.train(cfg, "badtime", 3)

        hist = bad.history()
        assert "trigger_update" in hist[1] and "trigger_update" not in hist[0]
        assert all(v["ok"] for v in MetricRecomputer().run(bad.path).values())
        assert bad.metrics()["input_change_test"]["changed_cells_per_window_mean"] <= 29
        assert bad.has(RunFiles.PLAN)

        ctl = ControlExperiment(tmp).run(clean.path, bad.path)
        assert set(ctl["test"]) == {
            "clean_model|clean_input", "clean_model|trigger_input",
            "attacked_model|clean_input", "attacked_model|trigger_input",
        }
        # Cùng mô hình sạch, đầu vào sạch -> khớp chỉ số đã lưu của nhánh sạch.
        saved = clean.metrics()["test"]["mae_c_norm"]
        assert ctl["test"]["clean_model|clean_input"]["mae_c_norm"] == pytest.approx(saved, rel=1e-9)
        assert bad.controls_files()
    finally:
        shutil.rmtree(tmp)

In [ ]:
%%writefile configs/etth1_timesnet_96.json
{
  "dataset": "ETTh1",
  "model": "TimesNet",
  "method": "clean",
  "seq_len": 96,
  "pred_len": 96,
  "runs_root": "runs",
  "data": {
    "split_protocol": "ratio_6_2_2",
    "allow_history_from_previous_split": true
  },
  "timesnet": {
    "label_len": 48,
    "d_model": 16,
    "d_ff": 32,
    "e_layers": 2,
    "top_k": 5,
    "num_kernels": 6,
    "dropout": 0.1,
    "embed": "timeF"
  },
  "train": {
    "epochs": 10,
    "batch_size": 32,
    "learning_rate": 0.0001,
    "lradj": "type1",
    "clean_loss": "mse",
    "early_stopping_patience": 3,
    "eval_batch_size": 32,
    "device": "auto",
    "deterministic": false
  },
  "attack": {
    "target_channels": [
      "OT"
    ],
    "target_channel_ratio": 0.1,
    "pattern": {
      "shape": "hump_backtime_cone",
      "length": 53,
      "amplitude": 1.0,
      "baseline": 0.0,
      "scale": "normalized"
    },
    "poison_ratio": 0.1,
    "similar_ratio": 0.3,
    "poison_channel_ratio": 0.3,
    "exclude_target_from_poison_channels": false,
    "trigger_total_len": 29,
    "budget_protocol": "keep_total",
    "interp_lambda": 1.0,
    "trigger_update_every": 2,
    "lag": {
      "min_corr_len": 8
    },
    "gat": {
      "hidden_dim": 64,
      "heads": 4,
      "epochs": 20,
      "learning_rate": 0.001,
      "batch_size": 256,
      "self_loops": true,
      "leaky_slope": 0.2
    },
    "model_loss_weights": [
      1.0,
      1.0,
      1.0
    ],
    "trigger_loss_weights": [
      1.0,
      1.0,
      1.0,
      1.0
    ],
    "loss_reduction": "sum",
    "smooth_eps": 0.8,
    "trigger_learning_rate": 0.01,
    "trigger_passes_per_update": 1,
    "trigger_batch_size": 32,
    "state_selection": {
      "clean_tolerance": 0.05
    }
  }
}

In [ ]:
%%writefile dataset/ETTh1/SOURCE.md
# Nguồn ETTh1.csv

- Kho: https://github.com/zhouhaoyi/ETDataset, commit `1d16c8f4f943005d613b5bc962e9eeb06058cf07`, file `ETT-small/ETTh1.csv`
- sha256: `f18de3ad269cef59bb07b5438d79bb3042d3be49bdeecf01c1cd6d29695ee066`
- 17420 hàng, 2016-07-01 00:00 → 2018-06-26 19:00, tần suất 1 giờ
- Cột: date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT
- Kiểm tra (`python -m badtime_repro check`): 0 giá trị thiếu, 0 mốc trùng, 0 bước không đều

File được đặt nguyên trạng, không sửa nội dung. Mỗi lần chạy ghi lại sha256 vào profile.json.

In [ ]:
%%writefile pytest.ini
[pytest]
testpaths = tests

## 4. Dữ liệu ETTh1

Tải đúng commit `1d16c8f4` của zhouhaoyi/ETDataset và kiểm tra sha256. Không tự sửa dữ liệu. Nếu tải không được, ô này tìm `ETTh1.csv` trong `/kaggle/input`; nếu vẫn không có, hãy upload file vào `dataset/ETTh1/`.

In [ ]:
import glob, hashlib, shutil, urllib.request

ETTH1_URL = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/1d16c8f4f943005d613b5bc962e9eeb06058cf07/ETT-small/ETTh1.csv"
ETTH1_SHA256 = "f18de3ad269cef59bb07b5438d79bb3042d3be49bdeecf01c1cd6d29695ee066"
dst = os.path.join(PROJECT, "dataset", "ETTh1", "ETTh1.csv")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not (os.path.isfile(dst) and sha256(dst) == ETTH1_SHA256):
    try:
        urllib.request.urlretrieve(ETTH1_URL, dst)
        print("Đã tải từ GitHub.")
    except Exception as exc:
        print("Không tải được:", exc)
        for cand in glob.glob("/kaggle/input/**/ETTh1.csv", recursive=True):
            if sha256(cand) == ETTH1_SHA256:
                shutil.copy(cand, dst)
                print("Đã chép từ", cand)
                break

assert os.path.isfile(dst), "Thiếu dataset/ETTh1/ETTh1.csv"
got = sha256(dst)
assert got == ETTH1_SHA256, f"sha256 không khớp: {got}"
print("ETTh1.csv OK", got)

In [ ]:
!python -m badtime_repro check

Kiểm tra nhanh thông báo khi chọn tổ hợp ngoài PoC (mã thoát 3 = chưa hỗ trợ, 2 = không hợp lệ). Không có tổ hợp nào được tự chuyển sang ETTh1 + TimesNet.

In [ ]:
!python -m badtime_repro check --set dataset=ETTH2; echo "exit=$?"
!python -m badtime_repro check --set model=Autoformer; echo "exit=$?"
!python -m badtime_repro check --set pred_len=720; echo "exit=$?"
!python -m badtime_repro check --method backtime; echo "exit=$?"
!python -m badtime_repro check --set dataset=Foo; echo "exit=$?"

## 5. Kiểm tra tính đúng

Nếu ô nào ở đây **fail**, dừng lại và gửi log; đừng diễn giải số ở các mục sau.
- Nhanh: các test numpy và torch không huấn luyện.
- Chậm (`RUN_SLOW_TESTS`): huấn luyện thật 2 epoch trên CPU với mô hình nhỏ. Kiểm tra (a) nhánh sạch và BadTime đã tắt (0% mẫu) cho cùng lịch sử huấn luyện; (b) tính lại chỉ số từ dự báo đã lưu khớp `metrics.json`; (c) trigger được cập nhật đúng lịch; (d) đối chứng 4 điều kiện chạy được.

In [ ]:
!python -m pytest -q tests -p no:cacheprovider

In [ ]:
if RUN_SLOW_TESTS:
    !BADTIME_SLOW=1 python -m pytest -q tests/test_torch_units.py tests/test_training.py -p no:cacheprovider -x
else:
    print("Bỏ qua kiểm tra chậm.")

## 6. Huấn luyện

Mỗi lần chạy có thư mục riêng `runs/ETTh1_TimesNet_96_96/<method>/seed<seed>_<thời điểm>/` và không ghi đè lần khác. Log in MAE val theo epoch; BadTime in thêm MAE_pa / MAE_pn.

In [ ]:
!python -u -m badtime_repro train --method clean --seeds {SEEDS_ARG} {SET_ARGS}

In [ ]:
!python -u -m badtime_repro train --method badtime --seeds {SEEDS_ARG} {SET_ARGS}

## 7. Đối chứng 4 điều kiện và báo cáo

Ghép từng seed: mô hình sạch seed *s* với trigger đã khóa của BadTime seed *s*. Benign MAE_pa / MAE_pn trong báo cáo lấy từ ô `clean_model|trigger_input`.

In [ ]:
import sys
sys.path.insert(0, PROJECT)

from badtime_repro.config import load_config
from badtime_repro.runs.store import RunIndex

CFG = load_config(os.path.join(PROJECT, "configs", "etth1_timesnet_96.json"), EXTRA_SET)
SLOT_DIR = os.path.join(PROJECT, CFG.runs_root, CFG.slot)
INDEX = RunIndex(SLOT_DIR)

for seed in SEEDS:
    clean_run, attack_run = INDEX.latest("clean", seed), INDEX.latest("badtime", seed)
    if clean_run is None or attack_run is None:
        print(f"seed {seed}: thiếu lần chạy hoàn thành (clean={clean_run is not None}, badtime={attack_run is not None})")
        continue

    clean_path, attack_path = clean_run.path, attack_run.path
    print(f"=== seed {seed}")
    !python -u -m badtime_repro controls --clean-run "{clean_path}" --attack-run "{attack_path}"

In [ ]:
from IPython.display import Markdown, display

!python -m badtime_repro report {SET_ARGS} > /dev/null
display(Markdown(open(os.path.join(SLOT_DIR, "report.md"), encoding="utf-8").read()))

## 8. Xem nhanh một lần chạy BadTime

Kênh bị sửa và độ trễ mà GAT/Thuật toán 3 chọn, mức sửa đầu vào, và một cửa sổ test trên kênh đích: nhãn gốc, dự báo từ đầu vào sạch, dự báo từ đầu vào có trigger, cùng mục tiêu.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

run = INDEX.latest("badtime", SEEDS[0])
plan, met, preds = run.plan(), run.metrics(), run.predictions("test")
print("lần chạy:", run.path)
print("kênh đích:", [plan.channels[i] for i in plan.target_idx], "| kênh bị sửa:", [plan.channels[i] for i in plan.poison_idx])
print("đoạn:", plan.seg_lens, "| độ trễ:", plan.lags, "| vị trí trong đầu vào [s, e):",
      [(o, o + l) for o, l in zip(plan.offsets, plan.seg_lens)])
print("ảnh hưởng GAT:", dict(zip(plan.channels, np.round(plan.trace["influence"], 4))))
print("số cửa sổ bị chọn:", len(plan.poison_starts), "| ô sửa mỗi cửa sổ:", plan.trigger_len)
print("chọn epoch:", met["selection"])
print("mức sửa đầu vào (test):", met["input_change_test"])

t = plan.target_idx[0]
for k in (0, len(preds["y"]) // 2):
    plt.figure(figsize=(9, 3))
    plt.plot(preds["y"][k, :, t], label="nhãn gốc")
    plt.plot(preds["pred_clean"][k, :, t], label="dự báo, đầu vào sạch")
    plt.plot(preds["pred_trig"][k, :, t], label="dự báo, có trigger")
    plt.plot(range(plan.t_pin), plan.pattern[:, 0], "k--", label="mục tiêu")
    plt.title(f"test window {k} (s={int(preds['starts'][k])}), kênh {plan.channels[t]}, thang chuẩn hóa")
    plt.legend(fontsize=8)
    plt.show()

## 9. (Tùy chọn) Độ nhạy với giả định chưa xác định

Mỗi biến thể dùng `runs_root` riêng nên không trộn với lần chạy mặc định, và mỗi lần chỉ đổi một giả định: A06 (dạng và biên độ mục tiêu), A08 (cách chia ngân sách), A13 (phép tổng hợp loss). Chỉ train lại BadTime, vì các giả định này không ảnh hưởng nhánh sạch (clean dùng loss MSE). Chi phí: số seed × số biến thể lần train BadTime. Báo cáo của mỗi biến thể vì vậy không có dòng Benign.

In [ ]:
VARIANTS = {
    "amp2":        ["attack.pattern.amplitude=2.0"],
    "hump_sine":   ["attack.pattern.shape=hump_sine"],
    "equal_seg":   ["attack.budget_protocol=equal_segments"],
    "loss_mean":   ["attack.loss_reduction=mean"],
}

if RUN_SENSITIVITY:
    for name, sets in VARIANTS.items():
        args = " ".join(f"--set {s}" for s in sets + [f"runs_root=runs_sens/{name}"])
        print(f"##### {name}")
        !python -u -m badtime_repro train --method badtime --seeds {SEEDS_ARG} {args}
        !python -m badtime_repro report {args}
else:
    print("RUN_SENSITIVITY = False")

## 10. Lưu kết quả

In [ ]:
import shutil

archive = shutil.make_archive(os.path.join(os.path.dirname(PROJECT), "badtime_runs"), "zip", PROJECT,
                              base_dir=".")
print("Đã nén:", archive, f"({os.path.getsize(archive) / 2**20:.1f} MB)")

if PLATFORM == "colab":
    from google.colab import files
    files.download(archive)
    # Hoặc lưu vào Drive:
    # from google.colab import drive; drive.mount("/content/drive"); shutil.copy(archive, "/content/drive/MyDrive/")
elif PLATFORM == "kaggle":
    print("Kaggle: file trong /kaggle/working được giữ ở tab Output sau khi Save Version.")

## Phụ lục: sổ giả định đang áp dụng

In [ ]:
from badtime_repro.assumptions import ASSUMPTIONS
for a in ASSUMPTIONS:
    print(f"{a['id']} [{a['status']}] {a['topic']}: {a['choice']}")
    if a.get("note"):
        print("      ghi chú:", a["note"])